In [ ]:
# Hero figure. Public scores are the board's (ours and other authors'). Forecasts and nested CV of the candidate steps come from
# our offline model of the public board (fitted on scored files) and our out-of-fold members; they are reported as measured.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
PATH = [('0.95442\nRealMLP step', 0.95442, None, 'o'), ('step B\n(v1)', 0.95446, 1.92, 'u'), ('1.5 x B', 0.95447, None, 'o'),
        ('step C\n(v2)', 0.95450, 0.93, 'u'), ('1.5 x C', 0.95452, None, 'o'), ('+ TabM step\n(v19)', 0.95479, None, 'o'),
        ('step D\n(v3)', 0.95482, 2.76, 'u'), ('2 x D +\nTabM (v20)', 0.95484, None, 'o'), ('step E\n(v4)', 0.95487, 2.27, 'u'), ('step F\n(v5)', 0.95489, 2.40, 'u'), ('6 x F\n(v22)', 0.95496, None, 'o'), ('step G\n(v6)', 0.95498, 0.71, 'u'), ('G ext. +\nzone 1', 0.95503, None, 'o'), ('step H\n+ zone 2', 0.95504, None, 'u'), ('step I\n(v8)', 0.95504, None, 'u'), ('step J\n(v9)', 0.95508, 1.10, 'u'), ('J x 3', 0.95512, None, 'o'), ('step K\n(v10)', 0.95519, 2.94, 'u'), ('K x 1.5', 0.95520, None, 'o'), ('step L\n(v12)', 0.95527, 3.36, 'u'), ('step M\n(v14)', 0.95531, 3.92, 'u'), ('step N\n(v16)', 0.95533, 2.16, 'u'), ('N + 1.5 r', 0.95535, None, 'o'), ('step O\n(v18)', 0.95536, 0.82, 'u'), ('O fc.', 0.95537, None, 'o'), ('step P\n(v20)', 0.95539, 1.14, 'u'), ('P fc.', 0.95539, None, 'o'), ('step Q\n(v22)', 0.95540, 0.33, 'u'), ('O + orth.\n(E1)', 0.95544, None, 'o'), ('step R\n(v24)', 0.95546, 2.44, 'u'), ('R + fusion\nresidual', 0.95547, None, 'o'), ('step S\n(v26)', 0.95548, 2.57, 'u'), ('board move\nof 0.95547', 0.95551, None, 'o'), ('step T\n(v27)', 0.95553, 3.96, 'u'), ('T + median\nresidual', 0.95554, None, 'o'), ('step U\n(v29)', 0.95555, 3.76, 'u'), ('board move\nof step U', 0.95557, None, 'o'), ('step V\n(v31)', 0.95558, 0.77, 'u'), ('two\nresiduals', 0.95559, None, 'o'), ('step W\n(v33)', 0.95559, 0.0, 'u'), ('step X\n(v35)', 0.95560, 0.28, 'u'), ('step Y\n(v37)', 0.95561, 0.57, 'u'), ('step Z\n(v39)', 0.95562, 2.18, 'u'), ('dennis6886\nclip', 0.95562, None, 'o'), ('Malyshev\nv42', 0.95565, None, 'o'), ('Malyshev\nv43', 0.95567, None, 'o'), ('step AA\n(v41)', 0.95567, 1.05, 'u'), ('Areeb Ali\nclip', 0.95568, None, 'o'), ('Sriram S\northog.', 0.95568, None, 'o'), ('step AB\n(v43)', 0.95569, 0.58, 'u')]
XO = 10                                                    # points added after step W (label rules count from step W)
TABM = [(0.03, 0.95450), (0.05, 0.95455), (0.07, 0.95459), (0.10, 0.95463), (0.15, 0.95465), (0.20, 0.95463)]   # busyaprime, on the 0.95442 file
GRID1 = [[-0.06, 12.68, 5.0], [-3.05, 21.47, 5.0], [-8.97, 26.67, 5.0], [0.05, 12.78, 5.0], [-2.6, 22.2, 5.0], [-7.94, 28.28, 5.0], [0.0, 14.46, 5.0], [-2.7, 25.52, 5.0], [-8.12, 33.28, 5.0], [0.09, 14.13, 5.0], [-2.31, 25.18, 5.0], [-7.22, 33.21, 5.0], [0.05, 15.76, 5.0], [-1.96, 29.13, 5.0], [-6.02, 40.07, 5.0], [0.09, 15.05, 5.0], [-1.7, 27.97, 5.0], [-5.4, 38.64, 5.0], [0.94, 6.34, 5.0], [0.41, 11.01, 5.0], [-1.6, 13.71, 4.0], [0.95, 6.64, 5.0], [0.49, 11.71, 5.0], [-1.38, 14.92, 5.0], [0.92, 7.27, 5.0], [0.45, 12.96, 5.0], [-1.43, 16.79, 5.0], [0.93, 7.34, 5.0], [0.53, 13.21, 5.0], [-1.21, 17.38, 5.0], [0.79, 8.91, 5.0], [0.48, 16.63, 5.0], [-0.93, 23.09, 5.0], [0.79, 8.65, 5.0], [0.55, 16.22, 5.0], [-0.74, 22.58, 5.0], [1.31, 2.99, 4.0], [1.63, 4.81, 4.0], [0.95, 5.33, 4.0], [1.17, 4.21, 4.0], [1.51, 7.38, 4.0], [1.03, 9.67, 4.0], [1.16, 4.21, 4.0], [1.52, 7.4, 4.0], [1.07, 9.77, 4.0], [1.38, 1.17, 4.0], [1.93, 1.22, 4.0], [1.67, 0.13, 4.0], [1.36, 1.27, 4.0], [1.92, 1.49, 4.0], [1.67, 0.54, 4.0]]                                          # round 1, from the 0.95442 file: (forecast, nested CV, folds up)
GRID2 = [[1.62, -1.39, 2.0], [2.0, -4.58, 0.0], [1.13, -9.44, 0.0], [1.44, -0.41, 2.0], [1.77, -2.32, 1.0], [0.99, -5.73, 1.0], [1.43, -0.2, 3.0], [1.88, -1.8, 2.0], [1.33, -4.7, 1.0], [1.28, 0.5, 4.0], [1.67, -0.19, 2.0], [1.17, -1.99, 1.0], [0.69, 2.54, 5.0], [0.93, 4.49, 5.0], [0.71, 5.89, 5.0], [0.63, 2.72, 5.0], [0.84, 4.9, 5.0], [0.62, 6.62, 5.0], [2.46, -6.87, 0.0], [3.62, -15.6, 0.0], [3.46, -26.2, 0.0], [2.27, -5.88, 0.0], [3.37, -13.38, 0.0], [3.29, -22.46, 0.0], [2.34, -6.29, 0.0], [3.54, -14.21, 0.0], [3.62, -23.65, 0.0], [2.16, -5.4, 0.0], [3.3, -12.22, 0.0], [3.43, -20.35, 0.0], [1.65, -4.0, 0.0], [2.77, -8.68, 0.0], [3.36, -14.08, 0.0], [1.55, -3.51, 0.0], [2.62, -7.6, 0.0], [3.19, -12.38, 0.0], [2.87, -10.38, 0.0], [4.18, -23.04, 0.0], [3.92, -38.2, 0.0], [2.72, -9.54, 0.0], [3.99, -21.12, 0.0], [3.82, -34.95, 0.0], [2.8, -10.09, 0.0], [4.14, -22.29, 0.0], [4.04, -36.91, 0.0], [2.65, -9.28, 0.0], [3.96, -20.55, 0.0], [3.93, -33.84, 0.0], [2.31, -8.44, 0.0], [3.71, -18.26, 0.0], [4.21, -29.49, 0.0], [2.21, -7.9, 0.0], [3.57, -17.07, 0.0], [4.09, -27.51, 0.0], [3.07, -12.33, 0.0], [4.36, -27.53, 0.0], [3.88, -45.65, 0.0], [2.96, -11.71, 0.0], [4.24, -26.06, 0.0], [3.84, -43.33, 0.0], [3.03, -12.21, 0.0], [4.35, -27.16, 0.0], [3.95, -44.99, 0.0], [2.92, -11.59, 0.0], [4.23, -25.74, 0.0], [3.91, -42.69, 0.0], [2.73, -11.23, 0.0], [4.15, -24.56, 0.0], [4.27, -40.13, 0.0], [2.65, -10.74, 0.0], [4.05, -23.5, 0.0], [4.21, -38.42, 0.0]]                                         # round 2, from the 0.95447 file
GRID3 = [[3.25, 5.16, 4.0], [2.9, 5.68, 4.0], [-1.07, 1.33, 2.0], [3.05, 6.3, 4.0], [2.81, 8.67, 4.0], [-0.77, 6.58, 4.0], [2.97, 7.17, 5.0], [2.88, 10.73, 4.0], [-0.29, 10.23, 4.0], [2.76, 7.84, 5.0], [2.71, 12.44, 4.0], [-0.19, 13.47, 4.0], [1.92, 10.1, 5.0], [1.72, 17.83, 5.0], [-0.62, 23.06, 5.0], [1.78, 10.16, 5.0], [1.61, 18.07, 5.0], [-0.54, 23.79, 5.0], [5.84, -5.26, 0.0], [8.33, -14.77, 0.0], [7.44, -28.65, 0.0], [5.5, -3.77, 1.0], [7.92, -11.47, 0.0], [7.2, -23.08, 0.0], [5.58, -4.05, 0.0], [8.19, -11.87, 0.0], [7.8, -23.54, 0.0], [5.26, -2.88, 2.0], [7.78, -9.14, 0.0], [7.53, -18.83, 0.0], [4.31, -0.52, 2.0], [6.68, -3.54, 2.0], [7.1, -8.93, 0.0], [4.08, 0.07, 2.0], [6.37, -2.13, 2.0], [6.86, -6.56, 2.0], [7.16, -12.4, 0.0], [10.37, -29.88, 0.0], [9.58, -52.93, 0.0], [6.85, -11.01, 0.0], [10.01, -26.79, 0.0], [9.43, -47.52, 0.0], [6.99, -11.76, 0.0], [10.29, -28.31, 0.0], [9.85, -49.97, 0.0], [6.69, -10.5, 0.0], [9.94, -25.4, 0.0], [9.69, -44.97, 0.0], [5.96, -8.73, 0.0], [9.29, -21.1, 0.0], [9.95, -37.13, 0.0], [5.73, -7.89, 0.0], [9.0, -19.19, 0.0], [9.77, -33.77, 0.0], [7.81, -16.75, 0.0], [11.05, -39.67, 0.0], [9.65, -69.13, 0.0], [7.58, -15.64, 0.0], [10.81, -37.08, 0.0], [9.62, -64.74, 0.0], [7.72, -16.41, 0.0], [11.02, -38.78, 0.0], [9.84, -67.55, 0.0], [7.49, -15.35, 0.0], [10.78, -36.29, 0.0], [9.81, -63.34, 0.0], [7.03, -14.26, 0.0], [10.52, -33.61, 0.0], [10.42, -58.25, 0.0], [6.85, -13.46, 0.0], [10.31, -31.75, 0.0], [10.36, -55.02, 0.0], [8.13, -19.17, 0.0], [11.26, -45.29, 0.0], [9.33, -78.73, 0.0], [7.98, -18.42, 0.0], [11.12, -43.54, 0.0], [9.35, -75.61, 0.0], [8.08, -18.99, 0.0], [11.25, -44.84, 0.0], [9.44, -77.89, 0.0], [7.93, -18.27, 0.0], [11.11, -43.1, 0.0], [9.46, -74.85, 0.0], [7.68, -17.68, 0.0], [11.04, -41.57, 0.0], [10.05, -71.98, 0.0], [7.55, -17.1, 0.0], [10.92, -40.13, 0.0], [10.05, -69.47, 0.0]]                                         # round 3, from the 0.95479 file
OURS = {'step A': (1.52, 7.4, 0.95445), 'step B': (1.92, 1.5, 0.95446), 'step C': (0.93, 4.5, 0.95450), 'step D': (2.76, 7.8, 0.95482),
        'step G (v6)': (0.71, 7.2, 0.95498), 'step I (v8)': (-0.61, 16.5, 0.95504), 'step K (v10)': (2.94, 36.3, 0.95519), 'step L (v12)': (3.36, 0.5, 0.95527), 'step Q (v22)': (0.33, 0.7, 0.95540)}

SURFACE, PAGE = '#fcfcfb', '#f9f9f7'
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'
GRID_C, AXIS, RING = '#e1e0d9', '#c3c2b7', '#e6e5df'
BLUE, ORANGE, GREEN = '#2a78d6', '#eb6834', '#1baf7a'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'text.color': INK, 'axes.edgecolor': AXIS, 'axes.labelcolor': INK2,
                     'axes.facecolor': SURFACE, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.8,
                     'xtick.color': MUTED, 'ytick.color': MUTED, 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'axes.grid': False,
                     'figure.facecolor': PAGE, 'savefig.facecolor': PAGE})

def tile(fig, x, y0, w, h, label, value, note):
    ax = fig.add_axes([x, y0, w, h]); ax.set_axis_off()
    ax.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle='round,pad=0,rounding_size=0.06', transform=ax.transAxes, facecolor=SURFACE, edgecolor=RING, lw=1))
    ax.text(0.06, 0.74, label, transform=ax.transAxes, fontsize=10, color=INK2, va='center')
    ax.text(0.06, 0.42, value, transform=ax.transAxes, fontsize=21, fontweight='bold', color=INK, va='center')
    ax.text(0.06, 0.14, note, transform=ax.transAxes, fontsize=9, color=MUTED, va='center')

def head(ax, title, sub):
    ax.text(0, 1.10, title, transform=ax.transAxes, fontsize=12.5, fontweight='bold', color=INK, va='bottom')
    ax.text(0, 1.035, sub, transform=ax.transAxes, fontsize=9.5, color=INK2, va='bottom')

fig = plt.figure(figsize=(16, 9.6), dpi=110)
fig.text(0.035, 0.952, 'Past the public top, step by step', fontsize=23, fontweight='bold', color=INK, va='center')
fig.text(0.035, 0.905, "A board model and out-of-fold members pick each step. Version 43 adds step AB: a small correction on Sriram S's orthogonal continuation, on 233 scored files.", fontsize=12, color=INK2, va='center')
tw, gap = 0.2175, 0.0125
TILES = [["Public LB", "0.95569", "one point above 0.95568"], ["Correction", "3 parts", "less recent, less 0.95557 move, more members"], ["Scored files in the model", "233", "with Sriram S's and Areeb Ali's"], ["Public files it builds on", "4", "Sriram S, Malyshev Danil, Areeb Ali, dennis6886"]]
for i, t in enumerate(TILES):
    tile(fig, 0.035 + i * (tw + gap), 0.75, tw, 0.11, *t)

# left: the path past the public top
ax = fig.add_axes([0.055, 0.08, 0.50, 0.55])
xs = np.arange(len(PATH)); pub = np.array([1e5 * p - 95400 for _, p, _, _ in PATH])
ax.plot(xs, pub, color=AXIS, lw=1.5, zorder=2)
for i, (_, _, fc, who) in enumerate(PATH):
    if fc is not None:
        f = pub[i - 1] + fc
        ax.plot([xs[i], xs[i]], [f, pub[i]], color=MUTED, lw=1.2, zorder=3)
        ax.plot([xs[i]], [f], marker='o', ms=10, mfc=SURFACE, mec=MUTED, mew=1.6, zorder=4)
for i, (name, p, _, who) in enumerate(PATH):
    last = i == len(PATH) - 1
    if who == 'o' and i >= len(PATH) - XO - 18:                           # recent files of other authors: scores are in the table
        ax.plot([xs[i]], [pub[i]], marker='o', ms=9, color=MUTED, mec=SURFACE, mew=1.6, zorder=5); continue
    ax.plot([xs[i]], [pub[i]], marker='o', ms=13 if last else 9, color=BLUE if who == 'u' else MUTED, mec=INK if last else SURFACE, mew=1.6, zorder=5)
    if XO > 3 and name.split('\n')[0] in ('step X', 'step Y'): continue          # labelled in the tables; keeps the top readable
    off, va, ha = ((6, 10), 'bottom', 'left') if XO > 2 and last else ((-4, 16), 'bottom', 'right') if XO and last else ((6, -8), 'top', 'left') if XO > 1 and i == len(PATH) - 2 else ((-4, 16), 'bottom', 'right') if XO > 2 and i == len(PATH) - 3 else ((4, -10), 'top', 'right') if i == len(PATH) - XO - 27 else ((-4, -10), 'top', 'left') if i == len(PATH) - XO - 26 else ((-6, -6), 'top', 'right') if i == len(PATH) - XO - 23 else ((6, -8), 'top', 'left') if i in (len(PATH) - XO - 19, len(PATH) - XO - 17, len(PATH) - XO - 15, len(PATH) - XO - 13, len(PATH) - XO - 11, len(PATH) - XO - 5, len(PATH) - XO - 3) else ((-10, 6), 'bottom', 'right')
    ax.annotate(f'{p:.5f}', (xs[i], pub[i]), xytext=off, textcoords='offset points', fontsize=9 if not last else 9.5, color=INK, ha=ha, va=va,
                fontweight='bold' if last else 'normal')
ins = ax.inset_axes([0.50, 0.17, 0.27, 0.27])
ins.plot([0] + [w for w, _ in TABM], [42] + [1e5 * s_ - 95400 for _, s_ in TABM], color=ORANGE, marker='o', ms=4, lw=1.6)
ins.set_title("busyaprime's TabM step on 0.95442", fontsize=8.5, color=INK2, loc='left'); ins.tick_params(labelsize=7.5)
ins.set_xlabel('step size', fontsize=8); ins.set_yticks([42, 50, 58, 66]); ins.set_yticklabels(['0.95442', '0.95450', '0.95458', '0.95466'])
ax.set_xticks(xs); ax.set_xticklabels([n for n, _, _, _ in PATH]); ax.tick_params(axis='x', labelsize=5.4, colors=INK2, length=0)
ax.set_xlim(-0.5, xs[-1] + 0.5); ax.set_ylim(39, max(161, pub.max() + 5))
ax.set_yticks(range(40, 165, 10)); ax.set_yticklabels([f'0.954{v:02d}' if v < 100 else f'0.955{v - 100:02d}' for v in range(40, 165, 10)])
ax.set_ylabel('public LB'); ax.grid(True, axis='y', color=GRID_C, lw=0.8); ax.set_axisbelow(True)
ax.legend(handles=[plt.Line2D([], [], color=BLUE, marker='o', lw=0, ms=9, label='our steps'),
                   plt.Line2D([], [], color=MUTED, marker='o', lw=0, ms=8, label="other authors' files"),
                   plt.Line2D([], [], color=MUTED, marker='o', mfc=SURFACE, mew=1.6, lw=0, ms=9, label='forecast made before submitting')],
          loc='upper left', frameon=False, fontsize=9.5)
head(ax, 'The top, file by file, since 0.95442', 'our steps (blue) alternate with other authors\' moves; each of ours forecast first')

# right: forecast and nested CV of the candidate steps, two rounds
ax = fig.add_axes([0.63, 0.08, 0.345, 0.55])
G1, G2, G3 = np.array(GRID1), np.array(GRID2), np.array(GRID3)
ax.axhline(0, color=AXIS, lw=1); ax.axvline(0, color=AXIS, lw=1)
ax.scatter(G1[:, 0], G1[:, 1], s=36, color=MUTED, alpha=0.65, edgecolor=SURFACE, linewidth=0.8, zorder=3)
ax.scatter(G2[:, 0], G2[:, 1], s=36, color=AXIS, alpha=0.8, edgecolor=SURFACE, linewidth=0.8, zorder=3, marker='s')
ax.scatter(G3[:, 0], G3[:, 1], s=40, color=ORANGE, alpha=0.35, edgecolor=SURFACE, linewidth=0.8, zorder=3, marker='^')
for name, (fc, cv, lb) in OURS.items():
    this = name.startswith('step Q')
    ax.plot([fc], [cv], marker='o', ms=13 if this else 10, color=BLUE, mec=INK if this else SURFACE, mew=1.6, zorder=6)
    if this:
        ax.annotate(f'{name}: {lb:.5f}', (fc, cv), xytext=(14, -22), textcoords='offset points', fontsize=9.5, ha='left',
                    color=INK, fontweight='bold', bbox=dict(boxstyle='round,pad=0.2', fc=SURFACE, ec='none', alpha=0.85))
x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
ax.fill_between([0, x1], 0, y1, color=GREEN, alpha=0.06, lw=0, zorder=1); ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
ax.text(x1 - 0.1, y1 - 1, 'both up', fontsize=10, color=GREEN, ha='right', va='top', fontweight='bold')
ax.set_xlabel('forecast of the public change, points'); ax.set_ylabel('nested CV change, points (weights fitted on four folds)')
ax.grid(True, color=GRID_C, lw=0.8); ax.set_axisbelow(True)
ax.legend(handles=[plt.Line2D([], [], color=MUTED, marker='o', lw=0, ms=7, label='candidate steps from the 0.95442 file'),
                   plt.Line2D([], [], color=AXIS, marker='s', lw=0, ms=7, label='candidate steps from the 0.95447 file'),
                   plt.Line2D([], [], color=ORANGE, alpha=0.5, marker='^', lw=0, ms=7, label='candidate steps from the 0.95479 file'),
                   plt.Line2D([], [], color=BLUE, marker='o', lw=0, ms=9, label='our steps with a nested CV reading (A to D, G, I, K, L, Q)')], loc='lower left', frameon=False, fontsize=9.5)
head(ax, 'Steps that raise the forecast and CV together', 'each dot: one setting of the step search, scored by the board model and nested CV')
plt.show()

# S6E11: past the public top, step by step

**Public LB 0.95569** (version 43), one point above the best public file before it (0.95568; 1 point = 0.00001).

**New in version 43: a correction on Sriram S's orthogonal continuation.** Sriram S carried step AA's move (from v43 to step AA) one more
step, six ways (all 0.95568); the orthogonal one, with its overlap with an earlier update projected out, is the one Kaggle keeps as
Sriram S's best. Areeb Ali kept 70% of step AA's move under a 95th-percentile cap (0.95568). With these files the board model holds 233
scored files. Around them it reads the recent residual and Malyshev Danil's 0.95557 move as slightly overshot and the member step as
slightly short. Three files with these corrections, each forecast first (points above Sriram S's logit continuation):

| file | forecast | public LB |
|---|---|---|
| logit continuation - 0.6 x recent residual - 0.3 x 0.95557 move + 0.2 x member step | +0.88 | 0.95568 |
| logit continuation - 0.4 x recent residual - 0.15 x 0.95557 move + 0.1 x member step | +0.62 | 0.95568 |
| **step AB** = orthogonal continuation - 0.4 x recent residual - 0.15 x 0.95557 move + 0.15 x member step | **+0.58** | **0.95569** |

* **The gain came on one base only.** Two sizes of the correction tie the logit continuation (0.95568); a similar one on the orthogonal
  continuation scores 0.95569. One rounding step apart, so this is a single comparison, not a rule.
* **The forecasts did not order the three**: the largest forecast tied, the smallest gained. At this height one point is one rounding step.

**Step AB** = Sriram S's orthogonal continuation (pinned copy) - 0.4 x the recent residual - 0.15 x Malyshev Danil's 0.95557 move + 0.15 x
the member step, the four zero-fraud zones re-applied: **0.95569**.

Step AA is kept as `submission_step_aa.csv`, step Z as `submission_step_z.csv`.

## Version 41

**Public LB 0.95567** (version 41), level with the best public file before it (0.95567; 1 point = 0.00001).

**Malyshev Danil's newest move, clipped, with the earlier residual.** Malyshev Danil moved step Z with his board
model (v42, 0.95565) and carried the same move once more (v43, **0.95567**). Areeb Ali scored two further sizes of it, a quarter
step and a half step clipped at the 99th percentile (both 0.95567), and dennis6886 moved step Z back towards step Y by 0.26 with the
same clipping (0.95562). With these files the board model holds 225 scored files. It reads the clipped move as still rising
(0.75 x: +0.58) and the earlier residual as adding on top (1.5 x: about +0.5).

| file | public LB |
|---|---|
| step Z (version 39) | 0.95562 |
| step Z moved back towards step Y by 0.26, clipped (dennis6886) | 0.95562 |
| v42: step Z moved by Malyshev Danil's board model | 0.95565 |
| v43: v42 + the same move again (Malyshev Danil) | 0.95567 |
| v43 + 0.25 x the move; v43 + 0.5 x the move clipped (Areeb Ali) | 0.95567; 0.95567 |
| step AA (version 41): v43 + 0.75 x the move clipped + 1.5 x earlier residual | **0.95567** |

* **Clipping keeps the bulk of a move.** Cutting the largest 1% of row moves at the 99th percentile lets the move go further: the
  board model gives 0.75 x the clipped move +0.58 against +0.53 unclipped.
* **The earlier residual still adds.** Malyshev Danil's move already carries some of it; 1.5 x more on top lifts the forecast to
  +1.05 (inside 0.98).

**Step AA** = Malyshev Danil's 0.95567 file (pinned copy) + 0.75 x his move from v42 to v43, clipped at its 99th percentile outside
the zones, + 1.5 x the earlier residual, the four zero-fraud zones re-applied: **0.95567**. On the public leaderboard of
24 November it was listed first among the 0.95567 entries.

Step Z is kept as `submission_step_z.csv`, step Y as `submission_step_y.csv`.

## Version 39

**Public LB 0.95562** (version 39), one point above step Y (0.95561), the best public file before it; 1 point = 0.00001.

**The earlier residual, carried much further.** With step Y in it, the board model (214 scored files) finds the
axis term spent (taking back more of step U's axis no longer adds) and the earlier residual still rising almost in a straight line,
about half a point per unit; a little more of the recent residual adds a tenth. Three sizes from step Y, each forecast first:

| file | board-model forecast | public LB |
|---|---|---|
| step Y + 1.5 x earlier residual + 0.3 x recent residual | +1.23 | 0.95562 |
| step Y + 2.0 x earlier residual + 0.3 x recent residual | +1.56 | 0.95562 |
| step Z = step Y + 3.0 x earlier residual + 0.3 x recent residual | **+2.18** | **0.95562** |

* **All three sizes gain the point**: the rise holds from 1.5 to 3.0 more of the earlier residual.
* **Step Z takes the largest.** The local model, refitted with all three, places it highest inside the rounding step.

**Step Z** = Areeb Ali's 0.95559 file + 5.2 x the earlier residual (5.65 x in total) - 0.8 x step U's axis + 0.3 x the recent
residual (0.68 x in total), the four zero-fraud zones re-applied: **0.95562**.

Step Y is kept as `submission_step_y.csv`, step X as `submission_step_x.csv`.

## Version 37

**Public LB 0.95561** (version 37), one point above step X (0.95560), the best public file before it; 1 point = 0.00001.

**The move from step W to step X, carried further.** With step X scored, the board model (210 scored files)
reads the move from step W to step X (more of the earlier residual, more of step U's axis taken back) as still rising, and the local
model agrees. Four probes from step X, each forecast first:

| file | board-model forecast | public LB |
|---|---|---|
| step X + 0.5 x earlier residual - 0.3 x step U's axis | +0.40 | 0.95560 |
| **step Y** = step X + 0.8 x earlier residual - 0.5 x step U's axis | **+0.57** | **0.95561** |
| the first row + 0.15 x member step | +0.27 | 0.95560 |
| the first row - 0.1 x member step | +0.46 | 0.95560 |

* **The longer step scores.** The two sizes of the same move split at the rounding edge: half a step again ties step X, 0.8 of the
  earlier residual with 0.5 of the axis gains the point.
* **The member step moves less than a point either way**, so step Y leaves it where step V put it.

**Step Y** = Areeb Ali's 0.95559 file + 2.2 x the earlier residual (2.65 x in total) - 0.8 x step U's axis, the four zero-fraud zones
re-applied: **0.95561**.

Step X is kept as `submission_step_x.csv`, step W as `submission_step_w.csv`.

## Version 35

**Public LB 0.95560** (version 35), one point above the best public file (0.95559; 1 point = 0.00001).

**The earlier residual, carried further again.** Four files now score 0.95559: Areeb Ali's (0.45 x the
earlier residual on top of step V), step W (0.9 x), and two blends of the two by Mirza Ishran
([Blending the Public Leaders](https://www.kaggle.com/code/mzishran/credit-card-fraud-blending-the-public-leaders)) and Rishabh Bafna
([Gram-Schmidt Optimization](https://www.kaggle.com/code/rishabhbafnaiiitd/credit-card-fraud-gram-schmidt-optimization)). On that
day's Code list step W, with the most earlier residual, stood first of the four and the 0.45 x file last. I read that order as a hint that the
residual was still rising. It was only a hint: a later byte copy of a tied file is listed apart from it, so list order does not measure the score.

* **The board model now sees the earlier residual.** With the two blends and the 1.35 x probe it holds 209 scored files and reads the
  residual as rising up to 1.85 x in total (+0.16 above step W). Taking back part of step U's axis (the move from step T to step U)
  adds a little more: +0.06 at 0.2 to 0.3 of it.
* **A second, local model agrees.** Fitted only on the files near the top (steps S to W, the probes and, as a hint only, the order of the ties), it
  points the same two ways.

**Step X** = Areeb Ali's 0.95559 file + 1.4 x the earlier residual (1.85 x in total) - 0.3 x step U's axis, the four zero-fraud zones
re-applied. Board-model forecast +0.28; public LB **0.95560**.

| step | starts from | public LB |
|---|---|---|
| step W (version 33) | 0.95559 | 0.95559 |
| two blends of step W and the 0.95559 file (Mirza Ishran, Rishabh Bafna) | 0.95559 | 0.95559 |
| step X (version 35) | 0.95559 | **0.95560** |

Step W is kept as `submission_step_w.csv`, step V as `submission_step_v.csv`.

## Version 33

**Public LB 0.95559** (version 33), level with the best public file (0.95559; 1 point = 0.00001).

**The earlier residual, carried further.** Areeb Ali
([Which Signals Rank Fraud?](https://www.kaggle.com/code/areebbb/s6e11-which-signals-rank-fraud-lb-0-95559)) took two residuals from
this notebook's version 31 files: the part of Malyshev Danil's 0.95557 move orthogonal to step U's move (*recent*), and the part of
step T's move orthogonal to the three newest moves (*earlier*). Step V + 0.38 x recent + 0.45 x earlier: **0.95559**. He also
published his probes: recent alone at 0.65 and at 1.55 both stay at 0.95558, so the extra point comes with the earlier residual.

* **The earlier residual is new to the board model.** A quarter of it lies outside the span of the 204 scored files (Areeb Ali's
  file and his two published probes included), so the model reads every size of it as flat (within 0.1). The board measures it.
* **It holds over a wide range**: 0.9 x and 1.35 x the earlier residual both score 0.95559. The unused parts of steps T and R on
  top of it lose a point (0.95558).

**Step W** = the 0.95559 file + 0.45 x the earlier residual (0.9 x in total), the four zero-fraud zones re-applied: **0.95559**.
The 0.95559 file is rebuilt here from this notebook's own files and the pinned 0.95557 file.

| step | starts from | public LB |
|---|---|---|
| step V (version 31) | 0.95557 | 0.95558 |
| step V + two residuals (Areeb Ali) | 0.95558 | 0.95559 |
| step W (version 33) | 0.95559 | **0.95559** |

Step V is kept as `submission_step_v.csv`, step U as `submission_step_u.csv`.

## Version 31

**Public LB 0.95558**, one point above the best public file before it (0.95557).

**The newest public move, carried further.** Malyshev Danil
([Fraud Signal Lab](https://www.kaggle.com/code/malyshevdanil/fraud-signal-lab-gpt-2-and-glm-fusion)) moved step U with his board
model (83 scored rankings, 153 members): **0.95557** (pinned copy). With it scored, the board model holds 200 scored files.

* **Sizing the move.** Fitted before the move was scored, the board model put step U 0.4 points below it; the board put it
  2 points below. Calibrated on that pair (step U sits where the board put it), 0.75 more of the move forecasts +0.8 over
  0.95557 and the curve tops out near 1.25 (+0.9). Uncalibrated, the model reads +0.1 at 0.5.
* **The member step adds on top**: +0.1 at a fifth of its size, as in steps S, T and U.

**Step V** = the 0.95557 file + 0.75 x its move from step U + 0.2 x the member step, the four zero-fraud zones re-applied:
forecast **+0.8** over 0.95557 (calibrated), written down first. The board gave **+1: 0.95558**.
Step V repeats step T's size: 0.75 more of the newest public move.

| step | starts from | forecast | public LB |
|---|---|---|---|
| step U (version 29) | 0.95554 | +3.8 | 0.95555 |
| step U moved by a board model (Malyshev Danil) | 0.95555 | | 0.95557 |
| **step V (version 31)** | 0.95557 | **+0.8** | **0.95558** |

Step U is kept as `submission_step_u.csv`, step T as `submission_step_t.csv`.

## Version 29

**Public LB 0.95555**, one point above the best public file before it (0.95554).

**Four scored directions on the newest public file.** Two public files built on step T and its neighbours:

* **dennis6886** ([Fraud signal rank fusion four refinements](https://www.kaggle.com/code/dennis6886/s6e11-fraud-signal-rank-fusion-four-refinements))
  moved step T 0.12 towards the median of steps R, S and the CV-led file (each minus step T, clipped at the 96% quantile of its
  size): **0.95554**. It is rebuilt here from this notebook's own files.
* **Areeb Ali** ([Which Signals Rank Fraud?](https://www.kaggle.com/code/areebbb/s6e11-which-signals-rank-fraud-lb-0-95552)) added
  an orthogonal residual to Malyshev Danil's 0.95551 file: **0.95552** (pinned copy; Asterios Panoras' notebook carries the same bytes).

With both scored, the board model holds 198 scored files. From the 0.95554 file, three directions still rise: Malyshev Danil's
move (+1.6 at 0.6 more), the member step (+1.3 at 0.4 more) and the move from 0.95551 to 0.95552 (+1.3 at twice its size).
Step R's projected direction is flat on its own (0.0 at 0.1) and adds +0.3 in the combination.

**Step U** = the 0.95554 file + 0.6 x Malyshev Danil's move + 0.4 x the member step + 2 x the move to the 0.95552 file + 0.1 x
step R's projected direction, the four zero-fraud zones re-applied: forecast **+3.8** over 0.95554, written down first. The board
gave **+1: 0.95555**.

| step | starts from | forecast | public LB |
|---|---|---|---|
| step T (version 27) | 0.95551 | +4.0 | 0.95553 |
| step T + 0.12 x median residual (dennis6886) | 0.95553 | | 0.95554 |
| **step U (version 29)** | 0.95554 | **+3.8** | **0.95555** |

Only the member part of step U has out-of-fold predictions (at a fifth of its size it replays +6.6, five of five folds).
Step T is kept as `submission_step_t.csv`, step S as `submission_step_s.csv`.

## Version 27

**Public LB 0.95553**, two points above the best public file before it (0.95551).

**The newest public move, carried further.** Malyshev Danil
([Fraud Signal Lab](https://www.kaggle.com/code/malyshevdanil/fraud-signal-lab-gpt-2-and-glm-fusion)) moved Areeb Ali's 0.95547 file
with his board model: **0.95551** (pinned copy). His move and step S's move, both from the 0.95547 file, have cosine 0.86.
With both scored, the board model holds 195 scored files. Three readings from it:

* **His move has room left**: 0.75 more of it forecasts +2.7 over 0.95551.
* **The member step of step S adds on top**: +0.8 at a fifth of its size (OOF replay +6.6, five of five folds).
* **Step R's projected direction adds a little more**: +0.5 at 0.15.

**Step T** = the 0.95551 file + 0.75 x its move from the 0.95547 file + 0.2 x the member step + 0.15 x step R's projected
direction, the four zero-fraud zones re-applied: forecast **+4.0** over 0.95551, written down first. The board gave
**+2: 0.95553**.

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step S (version 26) | 0.95547 | +2.6 | member part: OOF replay +6.6, 5/5 | 0.95548 |
| the 0.95547 file moved by a board model (Malyshev Danil) | 0.95547 | | | 0.95551 |
| **step T (version 27)** | 0.95551 | **+4.0** | **member part: OOF replay +6.6, 5/5** | **0.95553** |

Only the member part of step T has out-of-fold predictions. Step S is kept as `submission_step_s.csv` and step R as
`submission_step_r.csv`.

## Version 26

**Public LB 0.95548**, one point above the best public file before it (0.95547).

**Two scored directions and a CV-led member step.** Areeb Ali
([Which Signals Rank Fraud?](https://www.kaggle.com/code/areebbb/s6e11-which-signals-rank-fraud-lb-0-95552)) took Malyshev Danil's
newest fusion file (0.95541; pinned copy), removed its projection on our moves O to P, P to Q and Q to R, and added 0.35 of the
rest to step R: **0.95547**. With his file the board model holds 192 scored files. Three readings from it:

* **The fusion residual** still rises past 0.35 (forecast +0.2 at 0.7), but slowly.
* **Step R's own direction** has room left: 0.2 more of it (0.6 in all) forecasts +1.0.
* **Member steps are now both up.** Once the orthogonal line is in the model, a member step that raises CV also raises the
  forecast: the step with the 152 members (mu 16) at a fifth of its size gains **+6.6 on the replayed out-of-fold rows, five of
  five folds**, and forecasts about +1. Before the orthogonal files were scored the same direction forecast +0.2.

**Step S** = step R + 0.7 x the fusion residual + 0.2 x step R's projected direction + 0.2 x that member step, the four zero-fraud
zones re-applied (Areeb Ali's file leaves 356 zone rows above the lowest other row): forecast **+2.6** over 0.95547, written
down first. The board gave **+1: 0.95548**.

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step R (version 24) | 0.95540 | +2.4 | | 0.95546 |
| step R + 0.35 x fusion residual (Areeb Ali) | 0.95546 | | | 0.95547 |
| **step S (version 26)** | 0.95546 | **+2.6** | **member part: OOF replay +6.6, 5/5** | **0.95548** |

The fusion residual and PM8r8b have no out-of-fold predictions, so only the member part of step S has an OOF reading.
Step R is kept as `submission_step_r.csv` and step Q (nested +0.7) as `submission_step_q.csv`.

## Version 24

**Public LB 0.95546**, two points above the best public file before it (0.95544).

**A new orthogonal direction, sized at its measured peak.** Md. Hamid Hosen
([E1 orthogonal step](https://www.kaggle.com/code/hosen42/s6e11-e1-orthogonal-step)) took his own board file PM8r8b (0.95510,
a different basin), removed its projection on our steps J to O and added 0.30 of the rest to step O: **0.95544**
(0.45 of it scored the same). With his three scored files the board model holds 189 equations. Two readings from it:

* **Which part to add.** Malyshev Danil's 0.95540 file (step Q moved towards 39 scored files) already points partly along
  PM8r8b - step O (cosine 0.36), and the whole direction partly undoes steps P and Q. Step R therefore projects out
  **eight** scored moves: J to K to L to M to N to O, O to P, P to Q, and Q to the 0.95540 file. 78% of PM8r8b - step O is kept.
* **How much.** The two scored sizes (0.30 and 0.45, the same score) put the peak near 0.375 of the original direction;
  0.4 of the projected direction has about the same length.

**Step R** = the 0.95540 file + 0.4 x that projected direction, the four zero-fraud zones re-applied: forecast **+2.4** over
0.95544, written down first. The board gave **+2: 0.95546**.

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step Q (version 22) | 0.95539 | +0.3 | nested +0.7, 4/5 | 0.95540 |
| step Q moved towards 39 scored files (Malyshev Danil) | 0.95540 | | | 0.95540 |
| step O + 0.30 x orthogonal PM8r8b (Md. Hamid Hosen) | 0.95536 | | | 0.95544 |
| **step R (version 24)** | 0.95540 | **+2.4** | PM8r8b has no OOF rows | **0.95546** |

PM8r8b is a public board file without out-of-fold predictions, so step R has no OOF reading.

## Version 22

**Public LB 0.95540**, one point above the best public file before it (0.95539).

**The forecast and CV up together.** Malyshev Danil moved step P towards 37 scored public files with his
board model and also scored **0.95539**, slightly above step P (same five digits). With his file as one more equation the
board model holds 184 scored files (a left-out file near the top: 0.41 points). Around that file the member surface is flat:
no member step forecasts more than +0.4, while steps that raise nested CV cost almost nothing on the forecast.

**Step Q** is the member step from the 0.95539 file (152 members) with the highest forecast: **+0.3**, written down first, with
nested CV **+0.7 (four of five folds up)** and +3.0 on the replayed out-of-fold rows against step P. The board gave
**+1: 0.95540**.

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step P (version 20) | 0.95537 | +1.1 | replay -14.6 against step O | 0.95539 |
| step P moved towards 37 scored files (Malyshev Danil) | 0.95539 | | | 0.95539 |
| **step Q (version 22)** | 0.95539 | **+0.3** | **nested +0.7, 4/5** | **0.95540** |

Versions 21, 23 and 25 re-ran versions 20, 22 and 24 unchanged, so that the Code list picks up their scores.

## Version 20

**Public LB 0.95539**, two points above the best public file before it (0.95537).

**New in version 20.** Malyshev Danil built his own board model from 36 scored public files and moved step O towards them
(35 small moves, total size 2.8 in logit ranks): **0.95537**. His earlier notebook versions add seven more scored files
(0.95500 to 0.95534). The board model here now holds 182 scored files and forecasts a left-out file near the top to 0.43 points.

**Step P** starts from the 0.95537 file: half a step more along step O's own direction and a member step (152 members) at 1.25
times the size the search proposes. Forecast **+1.1**, written down first; the board gave **+2: 0.95539**.
On the replayed out-of-fold rows it gives back 14.6 points against step O; the file of this notebook that CV supports is
still step I (`submission_step_i.csv`).

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step O (version 18) | 0.95535 | +0.8 | replay -17 against the 0.95535 file | 0.95536 |
| step O moved towards 35 scored files (Malyshev Danil) | 0.95536 | +0.8 (his model) | | 0.95537 |
| **step P (version 20)** | 0.95537 | **+1.1** | replay -14.6 against step O | **0.95539** |

Version 19 re-ran version 18 unchanged, so that the Code list picks up its score.

## Version 18

**Public LB 0.95536**, one point above the best public file before it (0.95535).

**New in version 18.** Overnight four authors took step N further. Areeb Ali split step N's move `d = L(N) - L(M)` against
Malyshev Danil's 0.95527 file F and kept the part perpendicular to it, `r = d - beta (L(F) - L(M))`; here beta is only 0.011,
so r is nearly step N's own direction. N + 0.5 r scored **0.95534** (N + 0.25 d and 0.85 N + 0.15 F: 0.95533). Malyshev Danil
took N + r (0.95534), Rishabh Bafna N + 1.5 r (**0.95535**), and sriram00777 scored ten files around them (0.95531 to 0.95534);
asteriospanoras ships N + r. Seven of these are new equations: the board model holds 173 scored files and forecasts a
left-out file near the top to 0.43 points. Along step N's direction it puts the top near 2.5 to 3 times the step.

**Step O** is a member step from the 0.95535 file (152 members). Forecast **+0.8**, written down first; the board gave
**+1: 0.95536**. On the replayed out-of-fold rows it gives back 17 points against the 0.95535 file
(nested -20, 0 of 5 folds up); the file of this notebook that CV supports is still step I (`submission_step_i.csv`).

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step N (version 16) | 0.95531 | +2.2 | replay -15.2 against M | 0.95533 |
| N + 0.5 r (Areeb Ali) | 0.95533 | | | 0.95534 |
| N + r (Malyshev Danil) / N + 1.5 r (Rishabh Bafna) | 0.95533 | | replay -25 against N (1.5 r) | 0.95534 / 0.95535 |
| **step O (version 18)** | 0.95535 | **+0.8** | replay -17 against the 0.95535 file | **0.95536** |

Version 17 re-ran version 16 unchanged, so that the Code list picks up its score.

## Version 16

**Public LB 0.95533**, two points above the best public file before it (0.95531).

**New in version 16.** asteriospanoras now ships step M as it is (0.95531) and scored step L at 1.25, 1.5 and 2 times its size
(0.95527, 0.95528, 0.95528); Malyshev Danil's file moved away from Mayank Raj's mix stands at 0.95527 on the leaderboard. Five more
equations: the board model holds 167 scored files and forecasts a left-out file near the top to 0.45 points. Along step M's own
direction it sees the top at step M itself.

**Step N** is a member step from step M (152 members), at half the size the search proposes, where the forecast is the most
certain (95% of the move inside the scored directions). Forecast **+2.2**, written down first; the board gave
**+2: 0.95533**. On the replayed out-of-fold rows it gives back 15.2 points against step M (nested -17.7,
0 of 5 folds up); the file of this notebook that CV supports is still step I (`submission_step_i.csv`).

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step M (version 14) | 0.95527 | +3.9 | replay -10.8 against L | 0.95531 |
| step M as it is (asteriospanoras) | | | | 0.95531 |
| **step N (version 16)** | 0.95531 | **+2.2** | replay -15.2 against M (nested -17.7) | **0.95533** |

Version 15 re-ran version 14 unchanged, so that the Code list picks up its score.

## Version 14

**Public LB 0.95531**, four points above the best public file before it (0.95527).

**New in version 14.** Mayank Raj mixed step L with his own independently trained model (98 / 2 in logit ranks): **0.95526**.
Malyshev Danil moved step L half as far the other way, away from that model: also 0.95527 on the board. The mix is one more
equation (162 scored files; a left-out file near the top is forecast to 0.5 points).

**Step M** starts from step L: a quarter more of step L's own direction (from the 0.95520 file) and a member step (152 members),
taken 1.5 times, the size at the top of the forecast along this direction (1.75 and 2 times forecast lower). Forecast **+3.9**,
written down first; the board gave **+4: 0.95531**. On the replayed out-of-fold rows it gives back 10.8 points
against step L (one fold of five up) and stays 7 points above step I.

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step L (version 12) | 0.95520 | +3.4 | replay +10.1, 5/5 | 0.95527 |
| 98 / 2 mix with an own model (Mayank Raj) | 0.95527 | | | 0.95526 |
| **step M (version 14)** | 0.95527 | **+3.9** | replay -10.8 against L, +7 against I | **0.95531** |

Version 13 re-ran version 12 unchanged, so that the Code list picks up its score.

## Version 12

**Public LB 0.95527**, seven points above the best public file before it (0.95520).

**New in version 12.** Malyshev Danil took step K one and a half times (`K + 0.5 (K - M)`, M = his 0.95512 file) and scored
**0.95520**, one point above step K. That score is one more equation (160 scored files; a left-out file near the top is
forecast to 0.6 points). Along the K direction the board model now puts the top at that file: going further forecasts lower.

**Step L** is a member step from the 0.95520 file (152 members), chosen with CV in the objective and sized at the top of its
own direction: half a step more forecasts lower (+2.6 instead of +3.4), so the gain does not come from simply going further.
Forecast **+3.4**, written down first; the board gave **+7: 0.95527**. On the replayed
out-of-fold rows step L is **+10.1 against the 0.95520 file (5 of 5 folds up)** and +18 against step I; with the weights refitted on
four folds and scored on the fifth it is +0.5 (3 of 5).

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step K (version 10) | 0.95512 | +2.9 | nested +36.3, 5/5 | 0.95519 |
| K x 1.5 (Malyshev Danil) | 0.95519 | | replay +17.6 against K | 0.95520 |
| **step L (version 12)** | 0.95520 | **+3.4** | **replay +10.1, 5/5 (nested +0.5)** | **0.95527** |

Version 11 re-ran version 10 unchanged, so that the Code list picks up its score.

## Version 10

**Public LB 0.95519**, seven points above the best public file before it (0.95512).

**New in version 10: the forecast and CV up together again.** Malyshev Danil took step J two more times along its own
direction, with the part along step I removed, and scored **0.95512**. sriram00777 scored ten files around step J (blends
with the orthogonal and Gram-Schmidt files, step G with the four zones: 0.95500 to 0.95507). These seven scores are new
equations: the board model now holds 158 scored files and forecasts a left-out file near the top to 0.7 points.

From the 0.95512 file the picture is different from the last two rounds. That file sits 52 points below step I on the
replayed out-of-fold rows, and the board model sees that its last stretch along step J went past the top of the board. So
moving back towards the members now raises both. **Step K** (152 members): forecast **+2.9**, written down first; nested CV
**+36.3, up in all five folds** (+42.5 in-sample against the 0.95512 file). The board gave **+7: 0.95519**.

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step J (version 9) | 0.95504 | +1.1 | nested -23.6, 0/5 | 0.95508 |
| J x 3, orthogonal (Malyshev Danil) | 0.95508 | | replay -31 against J | 0.95512 |
| **step K (version 10)** | 0.95512 | **+2.9** | **nested +36.3, 5/5** | **0.95519** |

## Version 9

**Public LB 0.95508**, four points above the best public file before it (0.95504).

**New in version 9: 152 members.** Two new public sets of out-of-fold models on the shared folds join the members:

* **Weni Mohamed's [Diverse OOF Pack](https://www.kaggle.com/datasets/wenimohamed/s6e11-diverse-bpe-realmlp-tabm-oof-pack)** (CC0):
  seven models on BPE-digit features, among them LightGBM (OOF 0.95093), CatBoost (0.9507) and a GLM (0.95027). Their fold
  scores match the published table exactly, so they use the same five folds.
* **yekenot's [TabM (PyTabKit)](https://www.kaggle.com/code/yekenot/ps-s6-e11-tabm-pytabkit)** (OOF 0.94900).

On the labelled rows the BPE models are strong: from step I, a 0.3 step towards the BPE LightGBM alone gains about 55 points
(nested over folds, 5/5). The board model, now fitted on 151 scored files, forecasts that same move at -40: inside the scored
directions, moves towards single models have scored lower at the top. With all 152 members, the step search finds the move the
forecast asks for. **Step J**: forecast **+1.1**, written down first; the board gave **+4: 0.95508**. On the labelled rows it
gives back 23.6 points (nested, 0 of 5 folds up), so step I (`submission_step_i.csv`, nested +16.5) stays the file of this
notebook that CV supports.

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step I (version 8) | 0.95504 | -0.6 | nested +16.5, 5/5 | 0.95504 |
| **step J (version 9)** | 0.95504 | **+1.1** | nested -23.6, 0/5 | **0.95508** |

## Version 8

**Public LB 0.95504**, on a file with **16.5 points more nested CV** than version 7.

**New in version 8.** asteriospanoras now ships our step H as it is (0.95504). Three things were tried, two kept:

* **Two more zero-fraud zones (kept).** Weni Mohamed found that with all eight risk flags clear, the rows with `velocity_score <= 4.1`
  and `merchant_risk_score <= 33.8` (zone 3), or `velocity_score <= 6.0` and `merchant_risk_score <= 25` (zone 4), hold no fraud:
  39,149 training rows in zones 1 to 4, none a fraud. Checked here: zone 3 holds no fraud on a held-out fold even with its edge refitted
  on the other four (+1.4 nested); zone 4's edge is fragile (refitted, it lets two frauds in), so it stays at 6.0, where it holds.
  Sending zones 3 and 4 to the bottom as well scored 0.95504.
* **A wider search for zones (not kept).** Scanning every pair of columns, flags and categories for corner cells with no fraud finds
  21 such cells in the training rows, but when the whole search is redone on four folds and applied to the fifth, the cells it picks
  hold 62 frauds and cost 313 points. Most zero-fraud cells are luck; only zones 1 to 4 are used.
* **A step that CV leads, with 52 more members (kept).** busyaprime's 52 public models (CC0, the shared folds) join the members (144).
  From the zones file, the step search finds a move with **nested CV +16.5, up in all five folds**, forecast -0.6 inside the scored
  directions. The board gave **0.95504** again. Same public score, a file the labelled rows like much better.

| step | starts from | what it adds | CV | public LB |
|---|---|---|---|---|
| step H (version 7) | 0.95503 | zone 2 | nested +0.98, 5/5 | 0.95504 |
| step H + zones 3, 4 | 0.95504 | Weni Mohamed's zones | +1.5, 5/5 | 0.95504 |
| **step I (version 8)** | 0.95504 | **a member step, 144 members** | **nested +16.5, 5/5** | **0.95504** |

## Version 7

**Public LB 0.95504**, one point above the best public file before it (0.95503).

**New in version 7: a second zero-fraud zone.** daoviet found that no training row with `merchant_risk_score < 13.1` is a fraud
(0 of 12,736) and sent those test rows to the bottom (0.95498 to 0.95500). Malyshev Danil combined that rule with step G's move,
with its part along the step-F ladder removed and doubled: **0.95503**. Rishabh Bafna scored a similar split at 0.95501.

We looked for more regions like it, on the training rows only:

* **Single columns:** `merchant_risk_score < 13.1` is the only tail of any column with no fraud in more than 500 rows.
* **Shallow decision trees** find leaves with no fraud, but on a held-out fold those leaves contain frauds: -1.6e-3 nested. Not used.
* **Pairs of columns, deciles by deciles:** one cell of 7,436 rows has no fraud, `velocity_score <= 4.1` and
  `merchant_risk_score <= 20.0`. With the thresholds refitted on four folds and scored on the fifth it still holds no fraud and
  gains **+0.98 points of nested CV, up in all five folds** (zone 1 gives +2.1). Widening the box up to the first fraud on the
  fitting folds lets frauds in on the held-out fold (-6 to -8 nested), so we keep the decile box.

**Step H** first takes a member step and two scored directions from the 0.95503 file (scored 0.95503: level), then sends zone 2
to the bottom as well (2,609 more test rows). The board model had never seen such a move and could not forecast it; the CV
reading said about +1. The board gave **+1: 0.95504**.

| step | starts from | what it adds | CV | public LB |
|---|---|---|---|---|
| step G (version 6) | 0.95496 | a member step | nested +7.2, 5/5 | 0.95498 |
| step G + zone 1 (daoviet) | 0.95498 | merchant_risk_score < 13.1 to the bottom | +2.1, 5/5 | 0.95500 |
| G move without the ladder part, x 2, + zone 1 (Malyshev Danil) | 0.95498 | | | 0.95503 |
| member step + two scored directions | 0.95503 | | | 0.95503 |
| **step H (this version)** | 0.95503 | **+ zone 2 to the bottom** | **nested +0.98, 5/5** | **0.95504** |

## Version 6

**Public LB 0.95498**, two points above the best public file before it (0.95496).

**New in version 6.** asteriospanoras scored step F at 1 to 10 times its size and kept 6 times: **0.95496** (0.95491, 0.95493,
0.95495, 0.95496, 0.95495, 0.95492 at 2, 3, 4.5, 6, 8 and 10 times). Seven new equations: 143 scored files. Along every scored
direction the board model now sees a flat top (+0.3 at most). What it does see is a move that CV asks for and the board has not
tested yet. **Step G**: forecast +0.7 inside the scored directions, +2.4 with the part outside them read from CV; nested CV
**+7.2, up in all five folds**; the board gave **+2: 0.95498**. DavutKursun's new XGBoost-on-GLM bag (OOF 0.95069) joins the
members with a small weight.

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step F (version 5) | 0.95487 | +2.4 | nested -11.7, 0/5 | 0.95489 |
| 6 x step F (asteriospanoras v22) | 0.95487 | | | 0.95496 |
| **step G (this version)** | 0.95496 | **+0.7 (+2.4)** | **nested +7.2, 5/5** | **0.95498** |

## Version 5

**Public LB 0.95489**, two points above step E (0.95487), which asteriospanoras ships as it is.

**New in version 5.** asteriospanoras scored 2 x step E at 0.95486 and kept step E itself. With that equation the board model
(134 scored files) finds, from step E, no move that raises both the forecast and CV: the moves CV asks for forecast -4 or less.
**Step F** is the move the forecast asks for: forecast **+2.4**, written down first; the board gave **+2: 0.95489**. On
out-of-fold rows it gives back 11.7 points (nested, 0 of 5 folds up), so it is a board step, and step E
(`submission_step_e.csv`) stays the file of this notebook that CV supports. A second file built from scored directions only
(forecast +2.1) also scored 0.95489.

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step E (version 4) | 0.95484 | +2.3 | out of fold +6.7, 4/5 | 0.95487 |
| 2 x step E (asteriospanoras) | 0.95484 | | | 0.95486 |
| **step F (this version)** | 0.95487 | **+2.4** | nested -11.7, 0/5 | **0.95489** |
| scored directions only | 0.95487 | +2.1 | out of fold -13.7, 0/5 | 0.95489 |

## Version 4

**Public LB 0.95487**, three points above the best public file before it (0.95484).

**New in version 4.** asteriospanoras took 2 x our step D from the 0.95479 file (0.95483; 1.5 x also 0.95483, 3 x 0.95481) and
added 0.3 more of the TabM step: **0.95484**. With those four scores the board model holds 131 files. From the 0.95484 file,
**step E** adds the difference between our two scored steps from v19 (0.95484 and 0.95482) and two member steps of the
refitted search (nested CV +0.9 and +5.8 on their own). Forecast **+2.3**, written down first; the board gave **+3: 0.95487**.
On the replayed out-of-fold rows step E is **+6.7, up in four of five folds**. A variant with one member step at 1.5 x
(forecast +2.8, out-of-fold +0.9) also scored 0.95487; we keep step E.

| step | starts from | forecast | CV | public LB |
|---|---|---|---|---|
| step D (version 3) | 0.95479 | +2.8 | nested +7.8, 5/5 | 0.95482 |
| 2 x step D + 0.3 TabM step (asteriospanoras v20) | 0.95479 | | | 0.95484 |
| **step E (this version)** | 0.95484 | **+2.3** | **out of fold +6.7, 4/5** | **0.95487** |
| a variant of step E | 0.95484 | +2.8 | out of fold +0.9, 3/5 | 0.95487 |

## Version 3

**Public LB 0.95482**, three points above the best public file before it (0.95479).

**New in version 3.** busyaprime trained a TabM model on the shared folds (CC0 dataset) and stepped from the 0.95442 file towards it:
six step sizes on the board, 0.95450 to 0.95465. asteriospanoras took 1.5 x our step C (0.95452, also at 2 x) and added the TabM
step on top: **0.95479** (0.95478 at 0.8 of it). Those ten scores are new equations (125 scored files in the board model), and
TabM joins the members with its real out-of-fold predictions. From the 0.95479 file, **step D**: forecast +2.8, nested CV
**+7.8, up in all five folds**; the board gave **+3: 0.95482**. A larger step of the same search (forecast +4.1, nested CV +0.1,
two folds up) scored 0.95484; we keep step D, the one CV supports.

| step | starts from | forecast | nested CV | folds up | public LB |
|---|---|---|---|---|---|
| step C (version 2) | 0.95447 | +0.9 | +4.5 | 5/5 | 0.95450 |
| 1.5 x step C, + TabM step (asteriospanoras v19) | | | | | 0.95479 |
| **step D (version 3)** | 0.95479 | **+2.8** | **+7.8** | **5/5** | **0.95482** |
| a larger step, not kept | 0.95479 | +4.1 | +0.1 | 2/5 | 0.95484 |

## Version 2

**Public LB 0.95450**, three points above the best public file before it (0.95447).

**New in version 2.** asteriospanoras took 1.5 x our step B and scored 0.95447. With that score as one more equation, the same
search from the 0.95447 file found **step C**: a move of the same 90 members, forecast +0.9 (+1.5 counting the part outside
the scored directions, read from CV), nested CV **+4.5, up in all five folds**. The board gave **+3: 0.95450**.

| step | starts from | forecast | nested CV | folds up | public LB |
|---|---|---|---|---|---|
| step A, CV-led (version 1) | 0.95442 | +1.5 | +7.4 | 4/5 | 0.95445 |
| step B (version 1) | 0.95442 | +1.9 | +1.5 | 4/5 | 0.95446 |
| 1.5 x step B (asteriospanoras) | 0.95442 | | | | 0.95447 |
| **step C (this version)** | 0.95447 | **+0.9** | **+4.5** | **5/5** | **0.95450** |

## Version 1

**Public LB 0.95446**, four points above the best public file on 17 Nov (0.95442).

**Where 0.95442 came from.** Vladimir Demidov (yekenot) released a new RealMLP run (PyTabKit v15). sriram00777 moved two top
files by `w D`, `D = L(RealMLP v15) - L(RealMLP v28)` on logit ranks, and averaged them; asteriospanoras took `w = 0.4` on
BhuvanNR's 0.95427 file and our 0.95432 file and scored 0.95442. busyaprime scored the same recipe at four step sizes
(0.95439, 0.95442, 0.95441, 0.95440). Those scores are five new equations for our model of the public board.

**What this notebook adds.** Both RealMLP runs publish out-of-fold predictions, so they can join our members as real models,
not only as a difference of two files. From the 0.95442 file we ask one question: which move towards 90 members (the 87 of the
LM Ladder notebook, alexchilton's shallow-tree stack and the two RealMLP runs) raises `CV change + mu x forecast`? Two answers
were scored, each forecast written down first:

| step | forecast | nested CV (four folds fit, fifth scores) | folds up | public LB |
|---|---|---|---|---|
| step A, CV-led | +1.5 | +7.4 | 4/5 | 0.95445 (+3) |
| **step B (this notebook)** | **+1.9** | **+1.5** | **4/5** | **0.95446 (+4)** |

Both raise the forecast and nested CV together. Since version 43, `submission.csv` is step AB (0.95569), step AA is `submission_step_aa.csv` (0.95567), step Z is `submission_step_z.csv` (0.95562), step Y is `submission_step_y.csv` (0.95561), step X is `submission_step_x.csv` (0.95560), step W is `submission_step_w.csv` (0.95559), step V is `submission_step_v.csv` (0.95558), step U is `submission_step_u.csv` (0.95555), step T is `submission_step_t.csv` (0.95553), step S is `submission_step_s.csv` (0.95548), step R is
`submission_step_r.csv` (0.95546), step Q is
`submission_step_q.csv` (0.95540), step P is
`submission_step_p.csv` (0.95539), step O is
`submission_step_o.csv` (0.95536), step N is
`submission_step_n.csv` (0.95533), step M is
`submission_step_m.csv` (0.95531), step L is
`submission_step_l.csv` (0.95527), step K is
`submission_step_k.csv` (0.95519), step J is
`submission_step_j.csv` (0.95508), step I is
`submission_step_i.csv` (0.95504), step H is `submission_step_h.csv` (0.95504) and step G `submission_step_g.csv` (0.95498); step F is
`submission_step_f.csv` (0.95489) and step E `submission_step_e.csv` (0.95487); step D is
`submission_step_d.csv` (0.95482), step C `submission_step_c.csv` (0.95450); step B is
`submission_step_b.csv` (0.95446) and step A `submission_cv_led.csv` (0.95445). The notebook also rebuilds the 0.95442 file
(`submission_0p95442.csv`) as a check of the inputs.

**How to read it.** The board has 60,000 public rows; a step of this size moves their AUC by about one point by chance, and the
size of the step was chosen with public scores in the model. The nested CV column is the part measured on 700,000 labelled rows.

In [ ]:
import os, json, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import rankdata
from scipy.special import logit
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')
INPUT = Path(os.environ.get('KAGGLE_INPUT', '/kaggle/input')); WORK = Path(os.environ.get('KAGGLE_WORKING', '/kaggle/working'))

def sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def find(name, digest=None, inside=None, required=True):   # a file anywhere under the inputs, checked by SHA-256
    hits = [p for p in sorted(INPUT.rglob(name)) if inside is None or inside in str(p)]
    if digest is not None: hits = [p for p in hits if sha256(p) == digest]
    assert hits or not required, (name, digest, inside)
    return hits[0] if hits else None
def U(v): return (rankdata(v) - 0.5) / len(v)               # rank fraction
def L(v): return logit(U(v))                                # logit rank: AUC sees order only
def sub(name, digest): return pd.read_csv(find(name, digest))['is_fraud'].to_numpy(np.float64)

PIN = {   # pinned public files (asteriospanoras/s6e11-public-board-files, Apache 2.0), each checked by SHA-256
    'base_ours':  ('goodpjw2008_v9_submission.csv', '4d88b6f6dc7612df8d5ff9d77799748815b79f9762103341c0be48aa3dd45c1d'),  # our 0.95432 file
    'base_bhuv':  ('bhuvannr_step10_submission.csv', 'ce020d84c6279f5946f94983db6fe68b96a0dd96b5809d0a96ecd263fe586703'), # BhuvanNR 0.95427
    'rmlp_v15':   ('yekenot_realmlp_pytabkit_v15_submission.csv', 'b1462d0c7f64a48a4739acc4e68673d63e4648660744acb734b6770bc5207055'),
    'rmlp_v28':   ('yekenot_realmlp_pytorch_v28_submission.csv', 'c8ff11aeedb8e7a6268a1617ae14133f2dc7fa347a581f3cceed0ac6d44bfc98'),
    'stack':      ('alexchilton_v23_submission.csv', '36686835b6ca3bd805ef578fc50cd62b3cbbe4d2a70275e50d6c38c072d4ad77'),
}
T = {k: sub(*v) for k, v in PIN.items()}
sample = pd.read_csv(find('sample_submission.csv', inside='playground-series-s6e11'))

# 1. The 0.95442 file: each base moved by 0.4 D, as rank fractions, averaged (asteriospanoras, after sriram00777)
D = L(T['rmlp_v15']) - L(T['rmlp_v28'])
TOP = np.mean([U(L(T[b]) + 0.4 * D) for b in ('base_ours', 'base_bhuv')], axis=0)
sample[['id']].assign(is_fraud=TOP).to_csv(WORK / 'submission_0p95442.csv', index=False)
print('the 0.95442 file rebuilt; rank correlation of the two bases', round(float(np.corrcoef(L(T['base_ours']), L(T['base_bhuv']))[0, 1]), 6))

In [ ]:
# 2. Members on the test side: 16 public streams (Aman Atar's bundle), our wide LM stack, 70 models of our OOF bank,
#    alexchilton's stack and the two RealMLP runs. Same five folds everywhere (StratifiedKFold, shuffle, seed 42).
SHA = {'bundle': ('rank1_q17_streams.npz', '0fe4ab1263a673c8065d7145590b92da440263161bf860ce45f2976b5c4fa56f'),
       'wide_oof': ('wide_lm_stack_nested.npy', '69375871698f1adf2b4c666fc7f5180edf00ac6836668353b8e7e4ca43eb2722'),
       'wide_te': ('wide_lm_stack_test.npy', 'b37471e6c34a50715284e807d68e15e5f3ad033e72400904f27876b24041f8cc'),
       'bank': 'ddd0b8e8112d49846e415a4ddbead968bce6d50fb8f351785f21a8053f420538'}
Z = np.load(find(*SHA['bundle']), allow_pickle=False); y, FOLD = Z['y_train'].astype(int), Z['folds']
BANK = find('manifest.csv', inside='oof-bank').parent
bank = sorted(p for p in BANK.glob('*.npy') if p.name.startswith(('oof_', 'test_')))
assert hashlib.sha256(''.join(sha256(p) for p in bank).encode()).hexdigest() == SHA['bank'], 'the OOF bank changed'
XT = {'x:Sold': T['stack'], 'x:rmlpnew': T['rmlp_v15'], 'x:rmlpold': T['rmlp_v28']}
def member(k, part):
    if k.startswith('s:'): return L(Z[f'{part}_{k[2:]}'])
    if k.startswith('b:'): return L(np.load(BANK / f'{part}_{k[2:]}.npy'))
    if k == 'wide': return L(np.load(find(*SHA['wide_oof' if part == 'oof' else 'wide_te'])))
    return L(XT[k]) if part == 'test' else XO[k]

# 3. The step: weights chosen offline (forecast + CV, see the text), applied to the 0.95442 file in logit ranks
STEP_B = json.loads(""" {"s:megayak_5f_lm": 0.03923601219405702, "s:masaya_gpt2": 0.02100615554449594, "s:lm7_5fold": 0.012814801097451938, "s:gpt2_seed2026": -0.002062819413137699, "s:tabpfn_slimypunk_tp1": 0.004381824667652334, "s:sadam_tabpfn_100k_tp3": 0.034383747113219454, "s:yekenot_realmlp": -0.006297044499610565, "s:alex_glm_gen": -0.0152049122133897, "s:daoviet_gpdvae": 0.0025995792146871746, "s:gbn_bag5_w512": -0.051100071817507545, "s:sameer_glm10f": 0.02113020673141546, "s:sadam_v6_gb_xgb": 0.043932224509936314, "s:sadam_v6_gb_lgb": 0.00700780261139355, "s:sadam_v4_cat": 0.005891239156082329, "s:gb_bag5": -0.004884950152637307, "s:megayak_shallow_tree": -0.013858627186124219, "wide": -0.006059022387688783, "b:ebm_all_bag": 0.0175994958535856, "b:ebm_dig": -0.029417269555371307, "b:ebm_dig_bag": 0.01704782130332946, "b:ebm_dig_r7": -0.02323942639365016, "b:ebm_diglead_bag": -0.010788476099717375, "b:ebm_diglead_i25": -0.021674334055692378, "b:ebm_diglead_r11": -0.01600704988074788, "b:ebm_diglead_r13": -0.004225508929921286, "b:ebm_i25_bag": 0.017829013480691748, "b:ebm_i25_bag4": -0.010173138321692268, "b:ebm_lead4_bag": 0.04915568516364219, "b:gb_bag10": -0.0019789043553305793, "b:gb_bag5": -0.011000181265195206, "b:gb_l3": 0.00589437329353527, "b:gb_ohk": 0.022742905587172322, "b:gbc_bag3": 0.022615397994662154, "b:gbe_bag3": 0.003070382812559204, "b:gbn_bag5_w128": 0.029489284669171403, "b:gbn_bag5_w512": -0.055608575684664444, "b:gbr_bag3": -0.0112414293282544, "b:gbx_bag3": -0.024796162279452104, "b:gbx_bag5": -0.0018472080524488351, "b:glm_dig": -0.025163168193270042, "b:glm_dl": -0.008605395331418535, "b:glm_dl_bag": 0.024504246257353403, "b:glm_dl_s50": -0.010243780459640143, "b:glm_dl_s50_bag": 0.0437135632214679, "b:glm_ohk": -0.005198261421035855, "b:glm_ohk2": 0.0019720672388578524, "b:glm_ohk2_bag": 0.0360590972880872, "b:glm_ohk_c01": 0.0012108841214886338, "b:glm_ohkl": -0.0343164009583192, "b:glm_ohkl_bag": -0.03687038328774335, "b:glm_sb": -0.014052866504609768, "b:glm_te": 0.003795988147568311, "b:lgb_dig_l4": 0.011381697154334453, "b:lgb_dig_l7": 0.01146095997313025, "b:lgb_dig_s100": -0.020194377081782202, "b:lgb_dig_s100_bag": 0.014952304260612387, "b:lgb_dig_s100_bag5": 0.007897071846248136, "b:lgb_dig_s100_l3": -0.007826383444887754, "b:lgb_dig_s100_l3_bag": -0.031179089615732892, "b:lgb_diglead_bag": 0.03275490932840199, "b:lgb_diglead_bag5": -0.02067642237566645, "b:lgb_diglead_l2": -0.0002723579385114147, "b:lgb_diglead_l3": 0.031037961702126506, "b:lgb_diglead_l3_bag": -0.011813787974016167, "b:lgb_diglead_l4": -0.02229143449162717, "b:lgb_dk_l4": -0.01967969298700047, "b:lgb_raw_l31": -0.013339626333845596, "b:lgb_raw_l4": -0.002747132416601006, "b:lgb_raw_l4_cnt": 0.014214075706204532, "b:lgb_rawdig_l4": 0.002950864503497185, "b:lgb_te_l4": -0.012207552621807597, "b:lgb_te_l4_s100": 0.009589574841534639, "b:lgb_te_l4_s300": 0.007092880951011072, "b:lgb_te_l4_s5": 0.006558554226467163, "b:realmlp_allcat": 0.004051312881484887, "b:realmlp_cnt": -0.003124950845110124, "b:realmlp_dig": -0.001060171328120443, "b:realmlp_dig_s100": 0.0007606707860173356, "b:realmlp_dig_s2": -0.016841257776431685, "b:realmlp_rawdig": -0.0015593430466307322, "b:realmlp_te": 0.009655260861925433, "b:realmlp_te_cnt": -0.012165843739760908, "b:realmlp_te_s2": -0.020130673962685773, "b:realmlp_v1": -0.014714304478827774, "b:xgb_dig_d2": 0.001749237318780515, "b:xgb_raw_d3": 0.0024131111515847334, "b:xgb_te_d2": 0.018360214041675458, "x:Sold": -0.023729730453156204, "x:rmlpnew": 0.03251394349967517, "x:rmlpold": -0.006250340062569244} """)
STEP_A = json.loads(""" {"s:megayak_5f_lm": 0.02300161815139416, "s:masaya_gpt2": 0.017594604702160507, "s:lm7_5fold": 0.012169377472655831, "s:gpt2_seed2026": 0.00016135871614354183, "s:tabpfn_slimypunk_tp1": 0.0032607499025798722, "s:sadam_tabpfn_100k_tp3": 0.026902942716805062, "s:yekenot_realmlp": -0.0038316948675359667, "s:alex_glm_gen": -0.008171548034763955, "s:daoviet_gpdvae": -0.0008344006599590653, "s:gbn_bag5_w512": -0.05042554345797759, "s:sameer_glm10f": 0.028683000975576488, "s:sadam_v6_gb_xgb": 0.02231268454295146, "s:sadam_v6_gb_lgb": 0.016511441636264806, "s:sadam_v4_cat": 0.013015936074227701, "s:gb_bag5": -0.008944796717033434, "s:megayak_shallow_tree": 0.006655667592064714, "wide": -0.0005830501031048433, "b:ebm_all_bag": -0.007777462308928624, "b:ebm_dig": -0.01688011425354607, "b:ebm_dig_bag": -0.002515562738624244, "b:ebm_dig_r7": -0.002815094386824033, "b:ebm_diglead_bag": -0.014188370390945344, "b:ebm_diglead_i25": -0.023218009206276816, "b:ebm_diglead_r11": -0.008097853184243487, "b:ebm_diglead_r13": 0.015842112793455234, "b:ebm_i25_bag": 0.023157585022008636, "b:ebm_i25_bag4": 0.006359926944907146, "b:ebm_lead4_bag": 0.019865435090385426, "b:gb_bag10": -0.003113230488781459, "b:gb_bag5": -0.008012846707130806, "b:gb_l3": -0.0014607095694840662, "b:gb_ohk": 0.015130681073832735, "b:gbc_bag3": 0.01662647037147316, "b:gbe_bag3": 0.0001823656484826016, "b:gbn_bag5_w128": 0.02617936479949314, "b:gbn_bag5_w512": -0.047983143438677836, "b:gbr_bag3": -0.01480324438758971, "b:gbx_bag3": -0.03156043896427411, "b:gbx_bag5": 0.0036906612274057804, "b:glm_dig": -0.024174353047913863, "b:glm_dl": -0.002133787873320501, "b:glm_dl_bag": 0.019309946389979838, "b:glm_dl_s50": 0.003125522315555946, "b:glm_dl_s50_bag": 0.032132598480513, "b:glm_ohk": -0.0044650448997546185, "b:glm_ohk2": 0.01044097696486954, "b:glm_ohk2_bag": 0.020858540891043022, "b:glm_ohk_c01": 0.003516382568864321, "b:glm_ohkl": -0.030876525649648354, "b:glm_ohkl_bag": -0.03475832412265395, "b:glm_sb": -0.013233979670992664, "b:glm_te": 0.003757360469286918, "b:lgb_dig_l4": 0.020893545379598387, "b:lgb_dig_l7": 0.01297887030558544, "b:lgb_dig_s100": -0.02816067571907555, "b:lgb_dig_s100_bag": 0.022351487311963864, "b:lgb_dig_s100_bag5": 0.004838955945915298, "b:lgb_dig_s100_l3": 0.005037379397998553, "b:lgb_dig_s100_l3_bag": -0.038723087644798884, "b:lgb_diglead_bag": 0.037441625123208574, "b:lgb_diglead_bag5": -0.03748745324861766, "b:lgb_diglead_l2": -0.004576021041184137, "b:lgb_diglead_l3": 0.043920594411370036, "b:lgb_diglead_l3_bag": -0.020126413422062985, "b:lgb_diglead_l4": -0.02220821999802639, "b:lgb_dk_l4": -0.01845034854549232, "b:lgb_raw_l31": -0.01045517158533521, "b:lgb_raw_l4": -0.004610817478633037, "b:lgb_raw_l4_cnt": 0.010645641536779873, "b:lgb_rawdig_l4": 0.005255555539638223, "b:lgb_te_l4": -0.010812544677324207, "b:lgb_te_l4_s100": 0.009192630584386813, "b:lgb_te_l4_s300": 0.011992021594852026, "b:lgb_te_l4_s5": 0.006096446883282697, "b:realmlp_allcat": 0.008341241148910087, "b:realmlp_cnt": -0.00035682866881736554, "b:realmlp_dig": -0.0034212641816312576, "b:realmlp_dig_s100": 0.004707347610120412, "b:realmlp_dig_s2": -0.01723242076951222, "b:realmlp_rawdig": -0.007431998335290918, "b:realmlp_te": 0.009236416318157477, "b:realmlp_te_cnt": -0.010729396608351092, "b:realmlp_te_s2": -0.02197410094473936, "b:realmlp_v1": -0.011890280955399474, "b:xgb_dig_d2": -0.005464786134016091, "b:xgb_raw_d3": 0.005996571313545906, "b:xgb_te_d2": 0.012020652051781697, "x:Sold": 0.004150928861472411, "x:rmlpnew": 0.022406677824291527, "x:rmlpold": -0.003804057319103267} """)
def apply(w, base, part):
    return (1 - sum(w.values())) * base + sum(v * member(k, part) for k, v in w.items())
FILE_B, FILE_A = apply(STEP_B, L(TOP), 'test'), apply(STEP_A, L(TOP), 'test')
# Version 2. asteriospanoras took 1.5 x step B (public 0.95447); step C is one more member step from that file.
STEP_C = json.loads(""" {"s:megayak_5f_lm": 0.0015969765350754804, "s:masaya_gpt2": 0.04813873504693399, "s:lm7_5fold": -0.004213269697272437, "s:gpt2_seed2026": -0.0020661370630784153, "s:tabpfn_slimypunk_tp1": -0.010659360195695609, "s:sadam_tabpfn_100k_tp3": 0.018511993780150275, "s:yekenot_realmlp": 0.009473227106992794, "s:alex_glm_gen": -0.017751614358658293, "s:daoviet_gpdvae": 0.025513788851085344, "s:gbn_bag5_w512": -0.03334450940878887, "s:sameer_glm10f": 0.012587606072198157, "s:sadam_v6_gb_xgb": 0.01905690557176969, "s:sadam_v6_gb_lgb": 0.027950791185854626, "s:sadam_v4_cat": 0.011189286810292793, "s:gb_bag5": 0.011015008029140074, "s:megayak_shallow_tree": -0.002495466062376726, "wide": -0.0010855191375293565, "b:ebm_all_bag": 0.07888781068423502, "b:ebm_dig": -0.0776801307876967, "b:ebm_dig_bag": 0.06141133659874996, "b:ebm_dig_r7": -0.0551470772527649, "b:ebm_diglead_bag": -0.0812309825144001, "b:ebm_diglead_i25": 0.020594864206033035, "b:ebm_diglead_r11": -0.0034081913982483597, "b:ebm_diglead_r13": 0.004302309280241314, "b:ebm_i25_bag": -0.022434604088320344, "b:ebm_i25_bag4": -0.023634372798384488, "b:ebm_lead4_bag": 0.09011580950216454, "b:gb_bag10": -0.030349205601600038, "b:gb_bag5": 0.05386557035194959, "b:gb_l3": -0.017036583777529226, "b:gb_ohk": 0.020749228823477094, "b:gbc_bag3": 0.011291019166751976, "b:gbe_bag3": 0.008692534887854849, "b:gbn_bag5_w128": 0.03292349759593321, "b:gbn_bag5_w512": -0.011703972542041868, "b:gbr_bag3": -0.004701056649695216, "b:gbx_bag3": 0.003964431498138502, "b:gbx_bag5": -6.619282639878514e-05, "b:glm_dig": -0.02758385215936766, "b:glm_dl": 0.02452850444470143, "b:glm_dl_bag": 0.046066191190517226, "b:glm_dl_s50": -0.05477211572737818, "b:glm_dl_s50_bag": 0.021755576094300146, "b:glm_ohk": 0.04734494269969706, "b:glm_ohk2": 0.025202439856772976, "b:glm_ohk2_bag": 0.007688845847539438, "b:glm_ohk_c01": -0.052588815727706704, "b:glm_ohkl": -0.08767476259077175, "b:glm_ohkl_bag": -0.009954181798737095, "b:glm_sb": -0.003806638097839537, "b:glm_te": 0.009662950416179344, "b:lgb_dig_l4": 0.012690048409737146, "b:lgb_dig_l7": 0.033372098369409964, "b:lgb_dig_s100": -0.026119271186222353, "b:lgb_dig_s100_bag": -0.00407273671702019, "b:lgb_dig_s100_bag5": 0.0185892840554012, "b:lgb_dig_s100_l3": -0.0014819864067953331, "b:lgb_dig_s100_l3_bag": -0.02175800923665176, "b:lgb_diglead_bag": 0.013759800602467561, "b:lgb_diglead_bag5": -0.017671934695402906, "b:lgb_diglead_l2": -0.02645493827019353, "b:lgb_diglead_l3": 0.0456187626820622, "b:lgb_diglead_l3_bag": -0.010334188527207598, "b:lgb_diglead_l4": -0.011940440126024052, "b:lgb_dk_l4": -0.01657868955183302, "b:lgb_raw_l31": -0.009126480119738687, "b:lgb_raw_l4": -0.024739214321489546, "b:lgb_raw_l4_cnt": 0.002491643454694705, "b:lgb_rawdig_l4": 0.01290103241473295, "b:lgb_te_l4": -0.0013720976051019452, "b:lgb_te_l4_s100": 0.016766543515043305, "b:lgb_te_l4_s300": 0.003923275877128726, "b:lgb_te_l4_s5": 0.014813336154743315, "b:realmlp_allcat": -0.0025376016815567173, "b:realmlp_cnt": -0.006494245438072598, "b:realmlp_dig": -0.00514383725665221, "b:realmlp_dig_s100": -0.00038002053031605226, "b:realmlp_dig_s2": -0.013737074154299273, "b:realmlp_rawdig": -0.0073387177652187024, "b:realmlp_te": 0.01022871981587584, "b:realmlp_te_cnt": -0.00839945802865434, "b:realmlp_te_s2": -0.026257334457562714, "b:realmlp_v1": 0.006080916419554444, "b:xgb_dig_d2": -0.00341550313300887, "b:xgb_raw_d3": 0.0026766118524038135, "b:xgb_te_d2": -0.0041949549622297575, "x:Sold": -0.06259930562808587, "x:rmlpnew": 0.01991336973026524, "x:rmlpold": 0.009391588255731546} """)
B15 = U(L(TOP) + 1.5 * (L(U(FILE_B)) - L(TOP)))
FILE_C = apply(STEP_C, L(B15), 'test')
# Version 3. asteriospanoras v18 = 1.5 x step C; v19 = v18 + busyaprime's TabM step (0.15 (L(TabM) - L(RealMLP v28)) on the 0.95442 recipe),
# public 0.95479. Step D is one more member step from v19, now with TabM as a member (busyaprime's CC0 dataset, checked by SHA-256).
XT['x:tabm'] = np.load(find('tabm_test.npy', 'bb8be80481656526e179e4132cc1e1a5086ffb685c88c39cbad905b10b23a94b')).astype(np.float64)
TD = L(XT['x:tabm']) - L(T['rmlp_v28'])
S_T = np.mean([U(L(T[b]) + 0.4 * D + 0.15 * TD) for b in ('base_ours', 'base_bhuv')], axis=0)     # busyaprime's 0.95465 file
V18 = L(B15) + 1.5 * (L(U(FILE_C)) - L(B15))
V19 = U(L(U(V18)) + (L(S_T) - L(TOP)))
STEP_D = json.loads(""" {"s:megayak_5f_lm": -0.008318439414231204, "s:masaya_gpt2": 0.010557158479085227, "s:lm7_5fold": 0.0072294462771758305, "s:gpt2_seed2026": 0.017074221008245124, "s:tabpfn_slimypunk_tp1": -0.0037061675647217227, "s:sadam_tabpfn_100k_tp3": 0.009283648245916164, "s:yekenot_realmlp": 0.03300394404978027, "s:alex_glm_gen": 0.016485004361934005, "s:daoviet_gpdvae": 0.015108650351033733, "s:gbn_bag5_w512": 0.001518022723009372, "s:sameer_glm10f": -0.017112432600840985, "s:sadam_v6_gb_xgb": 0.002219537941039714, "s:sadam_v6_gb_lgb": 0.0030874713097465505, "s:sadam_v4_cat": 0.010029588870555354, "s:gb_bag5": -0.005789417033409965, "s:megayak_shallow_tree": 0.002680390484940892, "wide": 0.01504585640239224, "b:ebm_all_bag": 0.0021528730669589415, "b:ebm_dig": -0.01915853098794716, "b:ebm_dig_bag": 0.00430637580492611, "b:ebm_dig_r7": -0.0008978688215933083, "b:ebm_diglead_bag": -0.005941279254741969, "b:ebm_diglead_i25": 0.017255564814842233, "b:ebm_diglead_r11": -0.0021589122851456647, "b:ebm_diglead_r13": -0.00018253440287730493, "b:ebm_i25_bag": 0.015491581018016965, "b:ebm_i25_bag4": -0.017361887704097967, "b:ebm_lead4_bag": 0.009805922684060494, "b:gb_bag10": -0.016842117471380428, "b:gb_bag5": 0.02344669176477947, "b:gb_l3": -0.016810663461622743, "b:gb_ohk": -0.008849725696778813, "b:gbc_bag3": 0.020428215251939805, "b:gbe_bag3": -0.015436059083202131, "b:gbn_bag5_w128": -0.0037244549167189415, "b:gbn_bag5_w512": 0.002048753112809033, "b:gbr_bag3": -0.022409611126469973, "b:gbx_bag3": 0.004687698432236257, "b:gbx_bag5": 0.011038450108262867, "b:glm_dig": -0.03193299789997096, "b:glm_dl": 0.010741165108002916, "b:glm_dl_bag": 0.02871308289201037, "b:glm_dl_s50": -0.009139205299555014, "b:glm_dl_s50_bag": 0.0014070854799798109, "b:glm_ohk": 0.03424838013672356, "b:glm_ohk2": 0.0049095310138493635, "b:glm_ohk2_bag": 0.005435380147099089, "b:glm_ohk_c01": -0.002712305221936206, "b:glm_ohkl": -0.03813127276721911, "b:glm_ohkl_bag": -0.021840486456477192, "b:glm_sb": 0.0025620711771632537, "b:glm_te": 0.015488717359099962, "b:lgb_dig_l4": -0.0065939493926591205, "b:lgb_dig_l7": 0.034815588795066965, "b:lgb_dig_s100": -0.01905271330558543, "b:lgb_dig_s100_bag": 1.1475946270773852e-05, "b:lgb_dig_s100_bag5": 0.00036536255448656966, "b:lgb_dig_s100_l3": -0.00025407095678604385, "b:lgb_dig_s100_l3_bag": -0.028299208999170156, "b:lgb_diglead_bag": 0.0057193703080510834, "b:lgb_diglead_bag5": -0.002862134471948647, "b:lgb_diglead_l2": -0.011185257128310096, "b:lgb_diglead_l3": 0.04080027005736215, "b:lgb_diglead_l3_bag": -0.0027336413892547567, "b:lgb_diglead_l4": -0.007318231515448024, "b:lgb_dk_l4": -0.01458725280736681, "b:lgb_raw_l31": -0.00580224729852739, "b:lgb_raw_l4": -0.01959039326840886, "b:lgb_raw_l4_cnt": 0.01292588460372307, "b:lgb_rawdig_l4": 0.006193051419682669, "b:lgb_te_l4": 0.011971827288423048, "b:lgb_te_l4_s100": 0.013012757027971788, "b:lgb_te_l4_s300": 0.0036335990193795895, "b:lgb_te_l4_s5": 0.015120114099887077, "b:realmlp_allcat": -0.004736309278097022, "b:realmlp_cnt": -0.013493391793193555, "b:realmlp_dig": 0.0004857538795053019, "b:realmlp_dig_s100": -0.0019156985999186009, "b:realmlp_dig_s2": -0.015050964012466368, "b:realmlp_rawdig": -0.003964016342661259, "b:realmlp_te": 0.0046995303188978555, "b:realmlp_te_cnt": -0.00902116356744173, "b:realmlp_te_s2": -0.013402782067674406, "b:realmlp_v1": 0.017250976826344768, "b:xgb_dig_d2": -0.011583002851158045, "b:xgb_raw_d3": -0.0010394161963591005, "b:xgb_te_d2": -0.01725817614226139, "x:Sold": -0.020122078117712824, "x:rmlpnew": -0.03192334245700829, "x:rmlpold": 0.03297344764479364, "x:tabm": 0.0043281882687563385} """)
FILE_D = apply(STEP_D, L(V19), 'test')
# Version 4. asteriospanoras v20 = v19 + 2 x step D, then 0.3 more of the TabM step (public 0.95484). Step E starts there and adds
# (a) the difference between our two scored steps from v19 (the larger one scored 0.95484, step D 0.95482) and (b) two member steps
# found from v20 with the board model refitted on 131 scored files.
STEP_Q2, STEP_E1, STEP_E2 = (json.loads(j) for j in (""" {"s:megayak_5f_lm": 0.010798582055495996, "s:masaya_gpt2": 0.015712804588647006, "s:lm7_5fold": 0.009661702833031341, "s:gpt2_seed2026": 0.009462179097841441, "s:tabpfn_slimypunk_tp1": -0.004054403029112391, "s:sadam_tabpfn_100k_tp3": 0.015558319002645385, "s:yekenot_realmlp": 0.027674103363290938, "s:alex_glm_gen": 0.002596650138210803, "s:daoviet_gpdvae": 0.024352779782911775, "s:gbn_bag5_w512": -0.011264248038329854, "s:sameer_glm10f": -0.016385982635198776, "s:sadam_v6_gb_xgb": 0.02940068479715008, "s:sadam_v6_gb_lgb": -0.0046460415226564755, "s:sadam_v4_cat": 0.0014770335621043693, "s:gb_bag5": -0.007547964814889444, "s:megayak_shallow_tree": -0.02261125606504392, "wide": 0.009066566169187301, "b:ebm_all_bag": 0.02601479653122719, "b:ebm_dig": -0.03060171904096148, "b:ebm_dig_bag": 0.021726609933980668, "b:ebm_dig_r7": -0.021020444475597817, "b:ebm_diglead_bag": -0.006629919360317226, "b:ebm_diglead_i25": 0.019896925608859906, "b:ebm_diglead_r11": -0.0043078203510177725, "b:ebm_diglead_r13": -0.010523024164166488, "b:ebm_i25_bag": 0.01719276112430102, "b:ebm_i25_bag4": -0.03915745949687777, "b:ebm_lead4_bag": 0.024259253183325814, "b:gb_bag10": -0.022299028761103085, "b:gb_bag5": 0.032656513841688974, "b:gb_l3": -0.008954351606364562, "b:gb_ohk": 0.008324931346756633, "b:gbc_bag3": 0.01920760021392945, "b:gbe_bag3": -0.003549949465070813, "b:gbn_bag5_w128": 0.010969331906988691, "b:gbn_bag5_w512": -0.011861296090145716, "b:gbr_bag3": -0.009833985872571052, "b:gbx_bag3": 0.009891799778454688, "b:gbx_bag5": 0.0024855080516809594, "b:glm_dig": -0.023123085704719628, "b:glm_dl": 0.004812068885070522, "b:glm_dl_bag": 0.040357644727938444, "b:glm_dl_s50": -0.025963887886641168, "b:glm_dl_s50_bag": 0.006513927135241722, "b:glm_ohk": 0.03331941875139507, "b:glm_ohk2": -0.0023094120501510502, "b:glm_ohk2_bag": 0.024004864129266047, "b:glm_ohk_c01": -0.019076674350825096, "b:glm_ohkl": -0.051867199048049276, "b:glm_ohkl_bag": -0.015101497721325574, "b:glm_sb": 0.00039884255361251407, "b:glm_te": 0.011910135127230072, "b:lgb_dig_l4": -0.010422902702414836, "b:lgb_dig_l7": 0.02531632431033889, "b:lgb_dig_s100": -0.00900588516724239, "b:lgb_dig_s100_bag": -0.0054635519624886665, "b:lgb_dig_s100_bag5": 0.006340527222377902, "b:lgb_dig_s100_l3": -0.011494711491100314, "b:lgb_dig_s100_l3_bag": -0.015419329440979441, "b:lgb_diglead_bag": 0.0035849009891217236, "b:lgb_diglead_bag5": 0.009836350924136133, "b:lgb_diglead_l2": -0.007543991018811618, "b:lgb_diglead_l3": 0.021397737630243323, "b:lgb_diglead_l3_bag": 0.0021485270617623733, "b:lgb_diglead_l4": -0.008409726955462623, "b:lgb_dk_l4": -0.012449430343107195, "b:lgb_raw_l31": -0.007549945096302063, "b:lgb_raw_l4": -0.016812210323569436, "b:lgb_raw_l4_cnt": 0.01213586880928215, "b:lgb_rawdig_l4": 0.0030015456459856916, "b:lgb_te_l4": 0.004358053201799656, "b:lgb_te_l4_s100": 0.010831124332891446, "b:lgb_te_l4_s300": -0.0026332088065958784, "b:lgb_te_l4_s5": 0.016700929400466337, "b:realmlp_allcat": -0.00945722712213189, "b:realmlp_cnt": -0.011884818927937064, "b:realmlp_dig": 0.00179252618694719, "b:realmlp_dig_s100": -0.004853453112851313, "b:realmlp_dig_s2": -0.011611058093449669, "b:realmlp_rawdig": 0.0024878783981297312, "b:realmlp_te": 0.006305367778024918, "b:realmlp_te_cnt": -0.007108898401432902, "b:realmlp_te_s2": -0.01175321459005821, "b:realmlp_v1": 0.009076373537657195, "b:xgb_dig_d2": 0.0018870670356917875, "b:xgb_raw_d3": -0.002139862828236917, "b:xgb_te_d2": -0.007461208225671063, "x:Sold": -0.050918432964865, "x:rmlpnew": -0.0326588658097637, "x:rmlpold": 0.027643273393329195, "x:tabm": 0.00939480433354178} """, """ {"s:megayak_5f_lm": -0.0010980607367030533, "s:masaya_gpt2": 0.006196846976496547, "s:lm7_5fold": -0.00273744686556568, "s:gpt2_seed2026": 0.0006497706282696029, "s:tabpfn_slimypunk_tp1": 0.0008584039422340716, "s:sadam_tabpfn_100k_tp3": 0.00509033418957048, "s:yekenot_realmlp": 0.01737139515209897, "s:alex_glm_gen": 0.012753270690534882, "s:daoviet_gpdvae": -0.006500273993201099, "s:gbn_bag5_w512": 0.013839055607265881, "s:sameer_glm10f": -0.014534764510613064, "s:sadam_v6_gb_xgb": 0.006121127869943948, "s:sadam_v6_gb_lgb": -0.00960460464232, "s:sadam_v4_cat": -0.0006069325173816634, "s:gb_bag5": -0.01656511280227288, "s:megayak_shallow_tree": 0.011983957232716336, "wide": 0.001682798089962671, "b:ebm_all_bag": -0.002476082289744149, "b:ebm_dig": -0.006589687863530619, "b:ebm_dig_bag": 0.016544485724956468, "b:ebm_dig_r7": -0.010511658117479806, "b:ebm_diglead_bag": -0.053573910513694215, "b:ebm_diglead_i25": 0.008035628576238709, "b:ebm_diglead_r11": 0.027325436766670957, "b:ebm_diglead_r13": 0.021184674653711686, "b:ebm_i25_bag": 0.01733410105596513, "b:ebm_i25_bag4": -0.028451224007169762, "b:ebm_lead4_bag": 0.012916353346627859, "b:gb_bag10": -0.04198118198817049, "b:gb_bag5": 0.04725211498915036, "b:gb_l3": -0.005848995871540602, "b:gb_ohk": -0.007456125589568184, "b:gbc_bag3": 0.013060454899911785, "b:gbe_bag3": -0.008285386742632886, "b:gbn_bag5_w128": -0.013579703464925501, "b:gbn_bag5_w512": 0.009385515076274114, "b:gbr_bag3": -0.011938597338610241, "b:gbx_bag3": 0.003033609287424278, "b:gbx_bag5": 0.014267814439452645, "b:glm_dig": -0.011559442115371103, "b:glm_dl": -0.003089844578459844, "b:glm_dl_bag": 0.0233638917526985, "b:glm_dl_s50": -0.008058371875039307, "b:glm_dl_s50_bag": -0.010577054561815855, "b:glm_ohk": 0.017998455942952206, "b:glm_ohk2": 0.04632996036770939, "b:glm_ohk2_bag": -0.03741974753415575, "b:glm_ohk_c01": 0.017102759403650313, "b:glm_ohkl": -0.07140798494616842, "b:glm_ohkl_bag": 0.03450866800017309, "b:glm_sb": 0.0024039874990496504, "b:glm_te": 0.007167941358205961, "b:lgb_dig_l4": -0.012980410524949405, "b:lgb_dig_l7": 0.009913272918380463, "b:lgb_dig_s100": -0.010622843021607861, "b:lgb_dig_s100_bag": 0.0029575056393297394, "b:lgb_dig_s100_bag5": 0.0035359188745603303, "b:lgb_dig_s100_l3": 0.006255031981950446, "b:lgb_dig_s100_l3_bag": -0.016561434994502897, "b:lgb_diglead_bag": 0.019710132599256767, "b:lgb_diglead_bag5": 0.0017745931008707153, "b:lgb_diglead_l2": 0.005472488972754202, "b:lgb_diglead_l3": 0.014696874200636314, "b:lgb_diglead_l3_bag": -0.012470822935805854, "b:lgb_diglead_l4": -0.009056872133861834, "b:lgb_dk_l4": -0.0026454843762240858, "b:lgb_raw_l31": -0.002040197029777148, "b:lgb_raw_l4": -0.004960200359650335, "b:lgb_raw_l4_cnt": 0.006877774081690668, "b:lgb_rawdig_l4": -0.0017026147651026304, "b:lgb_te_l4": 0.01064990590014719, "b:lgb_te_l4_s100": 0.0018834882888162725, "b:lgb_te_l4_s300": -0.001222729161362491, "b:lgb_te_l4_s5": 0.0037326257216239098, "b:realmlp_allcat": -0.007401232995705278, "b:realmlp_cnt": -0.006808626140454465, "b:realmlp_dig": 0.004641332600268511, "b:realmlp_dig_s100": -0.0032040567532335535, "b:realmlp_dig_s2": -0.005283257866386181, "b:realmlp_rawdig": 0.0001327717158548359, "b:realmlp_te": -0.0009935928452946884, "b:realmlp_te_cnt": -0.002972778271386939, "b:realmlp_te_s2": 4.4939216579431375e-05, "b:realmlp_v1": 0.010290591402980812, "b:xgb_dig_d2": -0.004358565929035219, "b:xgb_raw_d3": -0.0013459555147725814, "b:xgb_te_d2": -0.009428106695326045, "x:Sold": -0.028799276442871353, "x:rmlpnew": 0.010552199154102524, "x:rmlpold": 0.017332126627026236, "x:tabm": -0.026073083469687657} """, """ {"s:megayak_5f_lm": -0.0012654075042716959, "s:masaya_gpt2": -0.00016011678166759276, "s:lm7_5fold": -0.0004464441913952112, "s:gpt2_seed2026": 0.0037911799017018057, "s:tabpfn_slimypunk_tp1": 0.0003086915028791502, "s:sadam_tabpfn_100k_tp3": 0.0029884754783351696, "s:yekenot_realmlp": 0.015316738256216352, "s:alex_glm_gen": 0.007710316407717952, "s:daoviet_gpdvae": -0.0009050700426451363, "s:gbn_bag5_w512": 0.007228414656351711, "s:sameer_glm10f": -0.012742833666406305, "s:sadam_v6_gb_xgb": 0.0016493737701403061, "s:sadam_v6_gb_lgb": -0.004685985077350389, "s:sadam_v4_cat": -2.8775939075211718e-05, "s:gb_bag5": -0.003232345908733814, "s:megayak_shallow_tree": 0.0027827409612823204, "wide": 0.002753618814321776, "b:ebm_all_bag": 0.0004405715581113449, "b:ebm_dig": -0.0015245766350363557, "b:ebm_dig_bag": 0.002984540333204209, "b:ebm_dig_r7": -0.0019973198175799326, "b:ebm_diglead_bag": -0.006675341443681401, "b:ebm_diglead_i25": 0.005799625254365736, "b:ebm_diglead_r11": 0.0038777453316753913, "b:ebm_diglead_r13": 6.657329351420458e-06, "b:ebm_i25_bag": 0.004872126058894574, "b:ebm_i25_bag4": -0.010258468895932784, "b:ebm_lead4_bag": 0.003013461249354216, "b:gb_bag10": -0.010140539355522215, "b:gb_bag5": 0.011019406843959665, "b:gb_l3": -0.004726168829258784, "b:gb_ohk": -0.005084512003661356, "b:gbc_bag3": 0.0071859524244288365, "b:gbe_bag3": -0.006245434721944435, "b:gbn_bag5_w128": -0.007090893194814097, "b:gbn_bag5_w512": 0.005781590958494558, "b:gbr_bag3": -0.006754622854045403, "b:gbx_bag3": 0.004633515247379264, "b:gbx_bag5": 0.004326925225787669, "b:glm_dig": -0.008221212083663934, "b:glm_dl": 0.0024605243122930573, "b:glm_dl_bag": 0.0066795192844208914, "b:glm_dl_s50": -0.0028890746437946566, "b:glm_dl_s50_bag": -0.0024357986714310846, "b:glm_ohk": 0.008812467021149838, "b:glm_ohk2": 0.004678794011530721, "b:glm_ohk2_bag": -0.0026992547062744707, "b:glm_ohk_c01": 0.007444519585382685, "b:glm_ohkl": -0.013515790990333024, "b:glm_ohkl_bag": 0.00035689699955640037, "b:glm_sb": 0.002126175345814667, "b:glm_te": 0.00479607344904796, "b:lgb_dig_l4": -0.007102486739927118, "b:lgb_dig_l7": 0.007687292098073806, "b:lgb_dig_s100": -0.0033338691599416595, "b:lgb_dig_s100_bag": -0.0007575772912077608, "b:lgb_dig_s100_bag5": -0.0003879129050526038, "b:lgb_dig_s100_l3": -0.0006586175427901817, "b:lgb_dig_s100_l3_bag": -0.006266067423523663, "b:lgb_diglead_bag": 0.002177436193467822, "b:lgb_diglead_bag5": 0.004644467041814723, "b:lgb_diglead_l2": 0.00188438170333418, "b:lgb_diglead_l3": 0.00635771477180152, "b:lgb_diglead_l3_bag": -0.0006354210854198356, "b:lgb_diglead_l4": -0.0014137049297009293, "b:lgb_dk_l4": -0.002673746267558817, "b:lgb_raw_l31": -0.0016376063888300818, "b:lgb_raw_l4": -0.004119928420948031, "b:lgb_raw_l4_cnt": 0.004328110134123989, "b:lgb_rawdig_l4": -0.0004200415495963953, "b:lgb_te_l4": 0.00539139996306673, "b:lgb_te_l4_s100": 0.0017267708022321393, "b:lgb_te_l4_s300": -0.0005844387175309505, "b:lgb_te_l4_s5": 0.003349584303377302, "b:realmlp_allcat": -0.0035097805363213143, "b:realmlp_cnt": -0.004893092210203982, "b:realmlp_dig": 0.0024549670352322175, "b:realmlp_dig_s100": -0.0016976946018117261, "b:realmlp_dig_s2": -0.003937854482267998, "b:realmlp_rawdig": 0.001154436526449603, "b:realmlp_te": -0.000363689966039742, "b:realmlp_te_cnt": -0.0022197522957224476, "b:realmlp_te_s2": -0.0004874883327363335, "b:realmlp_v1": 0.006294807621539771, "b:xgb_dig_d2": -0.0028497609323904645, "b:xgb_raw_d3": -0.001470046420181113, "b:xgb_te_d2": -0.006127200481021189, "x:Sold": -0.011561230734943417, "x:rmlpnew": 0.004366165000145025, "x:rmlpold": 0.015307824113975441, "x:tabm": -0.022794271352182434} """))
Q20 = U(L(V19) + 2.0 * (L(U(FILE_D)) - L(V19)))
V20 = U(L(Q20) + 0.3 * (L(S_T) - L(TOP)))
FILE_Q2 = apply(STEP_Q2, L(V19), 'test')
E1, E2 = apply(STEP_E1, L(V20), 'test'), apply(STEP_E2, L(V20), 'test')
FILE_E = L(V20) + (L(U(FILE_Q2)) - L(U(FILE_D))) + (L(U(E1)) - L(V20)) + (L(U(E2)) - L(V20))
# Version 5. Step F: one more member step from step E (board model refitted with 2 x step E, 0.95486, as one more equation).
STEP_F = json.loads(""" {"s:megayak_5f_lm": 0.00445171718787565, "s:masaya_gpt2": 0.004215097998781201, "s:lm7_5fold": -0.0015221414824222916, "s:gpt2_seed2026": -0.0013401813415732378, "s:tabpfn_slimypunk_tp1": 0.0015228162125725262, "s:sadam_tabpfn_100k_tp3": 0.010284167843431112, "s:yekenot_realmlp": 0.0017478745408252537, "s:alex_glm_gen": 0.009198860583435303, "s:daoviet_gpdvae": -0.0015628521085009179, "s:gbn_bag5_w512": 0.009427910065872515, "s:sameer_glm10f": -0.020670556633456766, "s:sadam_v6_gb_xgb": 0.026384537570077554, "s:sadam_v6_gb_lgb": -0.01963460173019338, "s:sadam_v4_cat": -0.00707833968566167, "s:gb_bag5": -0.016728640324417232, "s:megayak_shallow_tree": 0.016976258561921673, "wide": -0.005908026378061112, "b:ebm_all_bag": 0.00598388722574612, "b:ebm_dig": -0.001966349125520575, "b:ebm_dig_bag": 0.011370123047125343, "b:ebm_dig_r7": -0.013226159259954084, "b:ebm_diglead_bag": -0.01409734330226074, "b:ebm_diglead_i25": 0.009184292304971949, "b:ebm_diglead_r11": 0.012937592416250187, "b:ebm_diglead_r13": -0.007052164106921581, "b:ebm_i25_bag": 0.024500938952140908, "b:ebm_i25_bag4": -0.04342095464965218, "b:ebm_lead4_bag": 0.014224192113440249, "b:gb_bag10": -0.03387208388415313, "b:gb_bag5": 0.0335375134302165, "b:gb_l3": -0.005127379801547051, "b:gb_ohk": -0.0014455824688620192, "b:gbc_bag3": 0.017200787496210058, "b:gbe_bag3": -0.0049226328575816035, "b:gbn_bag5_w128": -0.007976353452613417, "b:gbn_bag5_w512": 0.00863684613214282, "b:gbr_bag3": -0.0071625566747177604, "b:gbx_bag3": 0.004704596288888793, "b:gbx_bag5": 0.013562405426896792, "b:glm_dig": -0.007821168530650003, "b:glm_dl": -0.007075732541968996, "b:glm_dl_bag": 0.021725988517889226, "b:glm_dl_s50": -0.021911392174886194, "b:glm_dl_s50_bag": -0.0004884230851130048, "b:glm_ohk": 0.014752163326995794, "b:glm_ohk2": 0.003985221589409975, "b:glm_ohk2_bag": 0.007023166248800645, "b:glm_ohk_c01": 0.014772690414628377, "b:glm_ohkl": -0.02933393065675954, "b:glm_ohkl_bag": -0.006067398715523779, "b:glm_sb": 0.0017657067941318797, "b:glm_te": 0.0074236246534626765, "b:lgb_dig_l4": -0.020657022017052693, "b:lgb_dig_l7": 0.005935034404263975, "b:lgb_dig_s100": -0.0044519406305130365, "b:lgb_dig_s100_bag": 0.001806320834917038, "b:lgb_dig_s100_bag5": 0.002749968715709341, "b:lgb_dig_s100_l3": -0.0022108785485970804, "b:lgb_dig_s100_l3_bag": -0.01193266736233792, "b:lgb_diglead_bag": 0.017277381236838158, "b:lgb_diglead_bag5": 0.016112384494083833, "b:lgb_diglead_l2": 0.008535264040710212, "b:lgb_diglead_l3": -0.00014693139707008308, "b:lgb_diglead_l3_bag": -0.004838695000161752, "b:lgb_diglead_l4": -0.00955181152496147, "b:lgb_dk_l4": -0.0014295012964984676, "b:lgb_raw_l31": -0.004014306976858737, "b:lgb_raw_l4": -0.002793105576879393, "b:lgb_raw_l4_cnt": 0.006862656736420142, "b:lgb_rawdig_l4": -0.006280040645933097, "b:lgb_te_l4": 0.006147596225521526, "b:lgb_te_l4_s100": 0.0018653553179635509, "b:lgb_te_l4_s300": -0.00691101859481069, "b:lgb_te_l4_s5": 0.0062999767726967795, "b:realmlp_allcat": -0.014137186978391163, "b:realmlp_cnt": -0.00863396333853951, "b:realmlp_dig": 0.007826189680837621, "b:realmlp_dig_s100": -0.006489514735931891, "b:realmlp_dig_s2": -0.0045278145973742575, "b:realmlp_rawdig": 0.005694709125560906, "b:realmlp_te": -0.0010123653315433966, "b:realmlp_te_cnt": -0.003053015942784096, "b:realmlp_te_s2": 0.003512954847111517, "b:realmlp_v1": 0.008827843870562228, "b:xgb_dig_d2": 0.003349827459468495, "b:xgb_raw_d3": -0.004671888554087571, "b:xgb_te_d2": -0.004241930604716719, "x:Sold": -0.042533713965545734, "x:rmlpnew": 0.01780057772940731, "x:rmlpold": 0.0017608628907425572, "x:tabm": -0.000130851552279187} """)
FILE_F = apply(STEP_F, L(FILE_E), 'test')
# Version 6. asteriospanoras v22 = step E + 6 x step F (public 0.95496). Step G is a member step from there that CV leads; DavutKursun's
# XGBoost-on-GLM bag (pinned copy, checked by SHA-256) joins the members.
XT['x:davut'] = pd.read_csv(find('davutkursun_xgb_glm_bag_test.csv', '503594986324e32b55a56926c82e3b5c7a0571da636b73a811b14bb0f18eccfd'))['pred'].to_numpy(np.float64)
V22 = U(L(FILE_E) + 6.0 * (L(U(FILE_F)) - L(FILE_E)))
STEP_G = json.loads(""" {"s:megayak_5f_lm": -0.0026816792019076978, "s:masaya_gpt2": 0.015350219345559062, "s:lm7_5fold": 0.0029505903472576735, "s:gpt2_seed2026": 0.009308819216476395, "s:tabpfn_slimypunk_tp1": -0.004784382587952412, "s:sadam_tabpfn_100k_tp3": -0.001036904206227624, "s:yekenot_realmlp": -0.01892496056022718, "s:alex_glm_gen": -0.010430844532670304, "s:daoviet_gpdvae": -0.0020336110709855176, "s:gbn_bag5_w512": -0.03725494118874913, "s:sameer_glm10f": 0.09075685629853932, "s:sadam_v6_gb_xgb": 0.04095636104219841, "s:sadam_v6_gb_lgb": 0.009557158196046155, "s:sadam_v4_cat": 0.00774128501920661, "s:gb_bag5": -0.005582339534736459, "s:megayak_shallow_tree": 0.039284452244410746, "wide": 0.0034035098848522113, "b:ebm_all_bag": -0.0035197910398252073, "b:ebm_dig": -0.012243487948262773, "b:ebm_dig_bag": 0.005552109563840294, "b:ebm_dig_r7": -0.008595691583737413, "b:ebm_diglead_bag": -0.01643512312217733, "b:ebm_diglead_i25": -0.01598625056380461, "b:ebm_diglead_r11": 0.0034924039980883314, "b:ebm_diglead_r13": 0.013031896916426486, "b:ebm_i25_bag": 0.02455269327437453, "b:ebm_i25_bag4": 0.01578277674178556, "b:ebm_lead4_bag": 0.009332209113723074, "b:gb_bag10": 0.008586741701260144, "b:gb_bag5": -0.042180902644230474, "b:gb_l3": 0.011126235703428267, "b:gb_ohk": 0.023635953281009665, "b:gbc_bag3": -0.011420612602011226, "b:gbe_bag3": 0.02244869976127124, "b:gbn_bag5_w128": 0.055271866975012816, "b:gbn_bag5_w512": -0.043941992696542455, "b:gbr_bag3": 0.01568445064709026, "b:gbx_bag3": -0.033836731458937584, "b:gbx_bag5": 0.015530334935661652, "b:glm_dig": 0.029118927954681843, "b:glm_dl": -0.021627657675852053, "b:glm_dl_bag": 0.0022595390728901016, "b:glm_dl_s50": -0.009776709955713732, "b:glm_dl_s50_bag": -0.0019622839157094154, "b:glm_ohk": 0.008627337446425756, "b:glm_ohk2": 0.01727067445478582, "b:glm_ohk2_bag": -0.011055463282502002, "b:glm_ohk_c01": -0.04800395030770713, "b:glm_ohkl": -0.01407469646811973, "b:glm_ohkl_bag": -0.0346054123129083, "b:glm_sb": -0.012852438743751925, "b:glm_te": -0.009130181452884158, "b:lgb_dig_l4": 0.03224839777546554, "b:lgb_dig_l7": -0.012742301395895517, "b:lgb_dig_s100": -0.020195411111792762, "b:lgb_dig_s100_bag": 0.03697584475619115, "b:lgb_dig_s100_bag5": 0.0028140897711496425, "b:lgb_dig_s100_l3": 0.033594925914310333, "b:lgb_dig_s100_l3_bag": -0.03761707680445874, "b:lgb_diglead_bag": 0.05554411197976351, "b:lgb_diglead_bag5": -0.03190541395831809, "b:lgb_diglead_l2": -0.03257982136832729, "b:lgb_diglead_l3": 0.006449194944948749, "b:lgb_diglead_l3_bag": -0.02413605912768249, "b:lgb_diglead_l4": -0.0265537032739698, "b:lgb_dk_l4": 0.010393331358078958, "b:lgb_raw_l31": 0.00902627334589993, "b:lgb_raw_l4": 0.003554244358039517, "b:lgb_raw_l4_cnt": -0.005212563349837763, "b:lgb_rawdig_l4": -0.005055859116477932, "b:lgb_te_l4": -0.025790998434787847, "b:lgb_te_l4_s100": 0.003611839068302751, "b:lgb_te_l4_s300": 0.001558947520479608, "b:lgb_te_l4_s5": -0.0055937710914450425, "b:realmlp_allcat": -0.0050353419748035465, "b:realmlp_cnt": 0.01410834493983348, "b:realmlp_dig": -0.007813244946259186, "b:realmlp_dig_s100": 0.0015790644927951188, "b:realmlp_dig_s2": 0.01162281020839712, "b:realmlp_rawdig": -0.016040730955964852, "b:realmlp_te": 0.00665256880228094, "b:realmlp_te_cnt": 0.006069295224232036, "b:realmlp_te_s2": -0.007962179072742949, "b:realmlp_v1": -0.008877551410738744, "b:xgb_dig_d2": -0.001516866456801537, "b:xgb_raw_d3": 0.0010717814633836188, "b:xgb_te_d2": 0.029507850646777585, "x:Sold": -0.013318295432944676, "x:rmlpnew": 0.014455980798875091, "x:rmlpold": -0.018906359084839328, "x:tabm": 0.005119558582869847, "x:davut": -0.004922768438782678} """)
FILE_G = apply(STEP_G, L(V22), 'test')
# Version 7. Malyshev Danil: step G's move with its part along the step-F ladder removed, doubled, and daoviet's zero-fraud zone
# (merchant_risk_score < 13.1) sent to the bottom (public 0.95503). Step H: a member step and two scored directions from there,
# then a second zero-fraud zone found on the training rows (velocity_score <= 4.1 and merchant_risk_score <= 20).
tst = pd.read_csv(find('test.csv', inside='playground-series-s6e11'), usecols=['id', 'velocity_score', 'merchant_risk_score'])
assert (tst['id'].to_numpy() == sample['id'].to_numpy()).all()
ZONE1 = tst['merchant_risk_score'].to_numpy() < 13.1
ZONE2 = (tst['velocity_score'].to_numpy() <= 4.1) & (tst['merchant_risk_score'].to_numpy() <= 20.0)
vF = L(FILE_F) - L(FILE_E); dG = L(FILE_G) - L(V22); prior = L(V22) - L(FILE_E)
dGp = dG - float(dG @ prior / (prior @ prior)) * prior          # step G's move without its part along the ladder so far
MAL = U(U(L(FILE_G) + 2.0 * dGp) - 2 * ZONE1)                   # Malyshev Danil's 0.95503 file
STEP_M1 = json.loads(""" {"s:megayak_5f_lm": -0.010290242805871605, "s:masaya_gpt2": -0.03454454463002969, "s:lm7_5fold": 0.008665107077276517, "s:gpt2_seed2026": 0.026337314591893753, "s:tabpfn_slimypunk_tp1": 7.321646107133348e-05, "s:sadam_tabpfn_100k_tp3": -0.0012944267946482157, "s:yekenot_realmlp": 0.022338959429791697, "s:alex_glm_gen": 0.0422211685773225, "s:daoviet_gpdvae": 0.02138028534961218, "s:gbn_bag5_w512": -0.011742539450128662, "s:sameer_glm10f": -0.004682163172133442, "s:sadam_v6_gb_xgb": 0.0034196611967843658, "s:sadam_v6_gb_lgb": -0.018854281291572638, "s:sadam_v4_cat": 0.004562832864285517, "s:gb_bag5": -0.011006883614766677, "s:megayak_shallow_tree": -0.04263153538824559, "wide": 0.02362581362592894, "b:ebm_all_bag": 0.01121537219028919, "b:ebm_dig": -0.011771981920844402, "b:ebm_dig_bag": 0.018805464509752876, "b:ebm_dig_r7": -0.008113539291042329, "b:ebm_diglead_bag": -0.01669666154711641, "b:ebm_diglead_i25": -0.000463764681407025, "b:ebm_diglead_r11": -0.0036310441089590544, "b:ebm_diglead_r13": -0.0028024240564077673, "b:ebm_i25_bag": 0.014520082056729151, "b:ebm_i25_bag4": -0.0035560440471021676, "b:ebm_lead4_bag": 0.00830640405460379, "b:gb_bag10": 0.02015288667156836, "b:gb_bag5": -0.037423607165620196, "b:gb_l3": 0.00479119127749848, "b:gb_ohk": 0.0010807690614202368, "b:gbc_bag3": -0.011610644414731185, "b:gbe_bag3": -0.01298494898152769, "b:gbn_bag5_w128": 0.032015360697505074, "b:gbn_bag5_w512": -0.029959663096958522, "b:gbr_bag3": -0.001073611486075377, "b:gbx_bag3": 0.003373253991777224, "b:gbx_bag5": 0.006354586862423415, "b:glm_dig": 0.012584240765373782, "b:glm_dl": 0.02783096219867463, "b:glm_dl_bag": 0.0007858767787299495, "b:glm_dl_s50": 0.0009229792827720055, "b:glm_dl_s50_bag": -0.012791328308114093, "b:glm_ohk": 0.045000636691395424, "b:glm_ohk2": 0.003890401044575262, "b:glm_ohk2_bag": -0.005190823979125411, "b:glm_ohk_c01": -0.013854221810161755, "b:glm_ohkl": -0.020864762687885917, "b:glm_ohkl_bag": -0.023338161529752235, "b:glm_sb": 0.005177149078744059, "b:glm_te": -0.0026818657529189657, "b:lgb_dig_l4": 0.012373928242090769, "b:lgb_dig_l7": 0.006204601155646754, "b:lgb_dig_s100": 0.0029246682370578396, "b:lgb_dig_s100_bag": 0.02442556427047694, "b:lgb_dig_s100_bag5": -0.019590674640457856, "b:lgb_dig_s100_l3": 0.017902512855416757, "b:lgb_dig_s100_l3_bag": -0.029589437910984204, "b:lgb_diglead_bag": 0.022083949769678226, "b:lgb_diglead_bag5": -0.005158995898314353, "b:lgb_diglead_l2": -0.014591027467055375, "b:lgb_diglead_l3": 0.0005886136519608704, "b:lgb_diglead_l3_bag": -0.0053000416364090905, "b:lgb_diglead_l4": -0.00618152326675072, "b:lgb_dk_l4": 0.008444440996710873, "b:lgb_raw_l31": 0.010778429731210107, "b:lgb_raw_l4": -0.015421330602945415, "b:lgb_raw_l4_cnt": 0.012834081784130655, "b:lgb_rawdig_l4": -0.009929307982479683, "b:lgb_te_l4": -0.02112483649104772, "b:lgb_te_l4_s100": -0.0064436608636077065, "b:lgb_te_l4_s300": 0.006348428005295876, "b:lgb_te_l4_s5": 0.004891169964924961, "b:realmlp_allcat": 0.0015347414107635818, "b:realmlp_cnt": 0.00085450811212591, "b:realmlp_dig": -0.003277331867265406, "b:realmlp_dig_s100": 0.0048475626555564005, "b:realmlp_dig_s2": -0.0035948778791228637, "b:realmlp_rawdig": -0.002788705703036062, "b:realmlp_te": -0.0019860979333547733, "b:realmlp_te_cnt": 0.007496797919760606, "b:realmlp_te_s2": -0.005136616325852825, "b:realmlp_v1": 0.008373665718533105, "b:xgb_dig_d2": -0.0035841594831028016, "b:xgb_raw_d3": -0.011506915248449223, "b:xgb_te_d2": 0.00272001631486194, "x:Sold": 0.026446438584973805, "x:rmlpnew": -0.04158408319762861, "x:rmlpold": 0.022355758204241256, "x:tabm": -0.004624388826561514, "x:davut": -0.004257714807758579} """)
M1 = apply(STEP_M1, L(MAL), 'test')
K2 = U(U(L(U(M1)) + vF + 0.5 * dGp) - 2 * ZONE1)                # scored 0.95503: level with the start
FILE_H = U(U(K2) - 2 * (ZONE1 | ZONE2))                          # + the second zero-fraud zone: 0.95504
# Version 8. Two more zero-fraud zones (Weni Mohamed's Z3, Z4: all eight risk flags clear and low velocity / merchant risk), then a
# member step that CV leads, with the 52 public models of busyaprime as members too. Public 0.95504 again; nested CV +16.5.
tsf = pd.read_csv(find('test.csv', inside='playground-series-s6e11'))
CLEAN = ((tsf['cvv_retry_count'] == 0) & (tsf['prior_disputes'] == 0) & ~tsf['is_new_merchant'] & ~tsf['is_foreign_transaction'] & ~tsf['used_vpn']
         & ~tsf['ip_country_mismatch'] & ~tsf['billing_shipping_mismatch'] & ~tsf['is_ai_generated_scam_attempt']).to_numpy()
ZONE34 = CLEAN & (((tst['velocity_score'].to_numpy() <= 4.1) & (tst['merchant_risk_score'].to_numpy() <= 33.8))
                  | ((tst['velocity_score'].to_numpy() <= 6.0) & (tst['merchant_risk_score'].to_numpy() <= 25.0)))
ZALL = ZONE1 | ZONE2 | ZONE34
K4 = U(U(FILE_H) - 2 * ZALL)                                     # step H with zones 3 and 4: scored 0.95504
B33 = find('manifest.csv', inside='33-own-members').parent
_m33 = pd.read_csv(B33 / 'manifest.csv'); _t33 = np.load(B33 / 'test.npy')
for j, nm in enumerate(_m33['name']):
    XT['x:b33_' + nm] = _t33[:, j].astype(np.float64)
STEP_I = json.loads(""" {"s:megayak_5f_lm": -0.004807895596552694, "s:masaya_gpt2": -0.02945073859916209, "s:lm7_5fold": 0.0011613779130793221, "s:gpt2_seed2026": -0.0004090551267847625, "s:tabpfn_slimypunk_tp1": -0.001875811224315769, "s:sadam_tabpfn_100k_tp3": -0.012474605916790692, "s:yekenot_realmlp": 0.007271293635984941, "s:alex_glm_gen": 0.014893976838001503, "s:daoviet_gpdvae": 0.0054615355984332335, "s:gbn_bag5_w512": 0.03385794333146074, "s:sameer_glm10f": -0.025439530280895382, "s:sadam_v6_gb_xgb": -0.07607192734936012, "s:sadam_v6_gb_lgb": 0.010268638877461551, "s:sadam_v4_cat": 0.005512357814393832, "s:gb_bag5": 0.01920599622889976, "s:megayak_shallow_tree": -0.0794392519435395, "wide": 0.01613797740356984, "b:ebm_all_bag": 0.039153495676125354, "b:ebm_dig": -0.012230559119926465, "b:ebm_dig_bag": 0.00876869995223001, "b:ebm_dig_r7": 0.010424326752263449, "b:ebm_diglead_bag": -0.06918995373172095, "b:ebm_diglead_i25": -0.006309528422987788, "b:ebm_diglead_r11": 0.016996666588267564, "b:ebm_diglead_r13": 0.04977758049334826, "b:ebm_i25_bag": -0.048995631805726435, "b:ebm_i25_bag4": 0.054979471064625245, "b:ebm_lead4_bag": -0.04656459170195421, "b:gb_bag10": 0.10121223081150181, "b:gb_bag5": -0.08967258868697474, "b:gb_l3": 0.0008394493492262195, "b:gb_ohk": 0.0033179230956621817, "b:gbc_bag3": -0.03048039674461548, "b:gbe_bag3": -0.008761890212061588, "b:gbn_bag5_w128": -0.007352206323739452, "b:gbn_bag5_w512": -0.027484971969241434, "b:gbr_bag3": 0.0023823973563672847, "b:gbx_bag3": 0.013266748813925821, "b:gbx_bag5": -0.029441752763418514, "b:glm_dig": -0.0023976796462025327, "b:glm_dl": 0.01728131922043544, "b:glm_dl_bag": 0.0060691423171110965, "b:glm_dl_s50": 0.037833505672689956, "b:glm_dl_s50_bag": -0.02026047203784219, "b:glm_ohk": 0.016017779396815812, "b:glm_ohk2": -0.01589251074091729, "b:glm_ohk2_bag": -0.01752775451105125, "b:glm_ohk_c01": 0.0030197896691269545, "b:glm_ohkl": 0.02296219568960315, "b:glm_ohkl_bag": 0.09555497470782637, "b:glm_sb": 0.00776264614612791, "b:glm_te": -0.01357689032842001, "b:lgb_dig_l4": 0.0292870076733453, "b:lgb_dig_l7": -0.009812896885229544, "b:lgb_dig_s100": 0.016200382911725986, "b:lgb_dig_s100_bag": 0.016056905438593182, "b:lgb_dig_s100_bag5": -0.021482714698097205, "b:lgb_dig_s100_l3": -0.005987802610640927, "b:lgb_dig_s100_l3_bag": 0.015648115397851523, "b:lgb_diglead_bag": -0.03996538292951223, "b:lgb_diglead_bag5": -0.004869099474768318, "b:lgb_diglead_l2": 0.016843173881339386, "b:lgb_diglead_l3": -0.016379972065824682, "b:lgb_diglead_l3_bag": 0.00250850001422007, "b:lgb_diglead_l4": 0.028233589629612255, "b:lgb_dk_l4": 0.00182782231993264, "b:lgb_raw_l31": 0.004124925778255589, "b:lgb_raw_l4": 0.0010997322362818298, "b:lgb_raw_l4_cnt": 0.002770725167864803, "b:lgb_rawdig_l4": 0.017631992870858492, "b:lgb_te_l4": -0.010176129585795814, "b:lgb_te_l4_s100": -0.017128015039735305, "b:lgb_te_l4_s300": 0.02258540847429139, "b:lgb_te_l4_s5": -0.008690356737580708, "b:realmlp_allcat": 0.030293913799576036, "b:realmlp_cnt": 0.007629151894902083, "b:realmlp_dig": -0.006999014725225534, "b:realmlp_dig_s100": 0.015751846953701038, "b:realmlp_dig_s2": 0.0016392385166132764, "b:realmlp_rawdig": 0.0050356821515937485, "b:realmlp_te": -0.00036946974038828967, "b:realmlp_te_cnt": 0.0048508962435399525, "b:realmlp_te_s2": -0.0022414288615539906, "b:realmlp_v1": -0.01728647979990216, "b:xgb_dig_d2": -0.01621261183381677, "b:xgb_raw_d3": 0.003949673405867162, "b:xgb_te_d2": -0.009834258119857754, "x:Sold": 0.10912136438410029, "x:rmlpnew": -0.03755393125909279, "x:rmlpold": 0.00668955615948027, "x:tabm": -0.005317809511122866, "x:davut": -0.015097022942234393, "x:b33_lgb_g2_plaininit_l4": 0.02465681346838285, "x:b33_lgb_g2_plaininit_l3": 0.018582399024407982, "x:b33_lgb_g2_glminit_l4": 0.019930833154531365, "x:b33_lgb_g2_glminit_l5": 0.04442535775882814, "x:b33_lgb_g2_glminit_s50": -0.009242563164964229, "x:b33_lgb_g2_glminit": -0.02452017610176892, "x:b33_lgb_g2_glminit_l7": -0.020731461682173144, "x:b33_lgb_g2_glminit_l4_hier": -0.04170011957120832, "x:b33_xgb_g2_glminit": 0.06379879893156563, "x:b33_lgb_dl3_g2c3": -0.044577235920561245, "x:b33_lgb_dl3": 0.0032402912065922387, "x:b33_lgb_dl4_g2cc": 0.005023524481005885, "x:b33_lgb_dl4_g2": 0.023030335816423258, "x:b33_glm_plain_g2_s50": -0.04311843252951442, "x:b33_lgb_dl3_cc": 0.04021345801904971, "x:b33_lgb_dl3_hier": 0.03136574523854453, "x:b33_lgb_dl3_trie": 0.01378605297665585, "x:b33_ebm_g2_glminit": -0.06271146361850255, "x:b33_lgb_g2_glminit_l2": 0.014429239173485896, "x:b33_lgb_dl3_s10": -0.002791573921022922, "x:b33_lgb_dl4_hier": 0.03788822793056913, "x:b33_xgb_dl2": -0.00013291511762352588, "x:b33_glm_plain_s50": 0.018124163662438294, "x:b33_lgb_dl4": -0.04378993296011078, "x:b33_lgb_dl4_c3": -0.024904508358790232, "x:b33_lgb_dl3_s100": 0.010235179975044308, "x:b33_lgb_dl4_s100": -0.033258308771183256, "x:b33_lgb_dl4_s10": 0.002943724014716414, "x:b33_glm_ohk_g2_c0003_s50": -0.03141126860137529, "x:b33_glm_ohk_g2_c0003_s100": -0.03121032645122792, "x:b33_glm_ohk_g2_c0005_s50": 0.0392289277059417, "x:b33_glm_ohk_g2_c0005_s100": -0.008549001398305792, "x:b33_glm_ohk_g2_c0002_s50": 0.12193483083142473, "x:b33_glm_ohk_g2_c0003": -0.036875636560936384, "x:b33_glm_ohk_g2_c0005_m2": 0.034481804210262985, "x:b33_glm_ohk_g2_c0005": -0.06910552096025879, "x:b33_lgb_dl2_hier": -0.019502092285362843, "x:b33_glm_ohk_g2x_c0005": -0.0041367511857733075, "x:b33_glm_ohk_c0005": -0.06934695715896293, "x:b33_ebm_dl": 0.0016626503047766864, "x:b33_ebm_dl_g2": 0.01456007576624691, "x:b33_glm_ohk_c0002": 0.033783120975979734, "x:b33_glm_ohk_g2_c001": 0.00878703786983517, "x:b33_glm_ohk_c001": -0.00669431395951818, "x:b33_tabm_dl": -0.003283760107012651, "x:b33_rmlp2_dl_hier": 0.014724879908118148, "x:b33_rmlp2_dl": -0.03211487225901108, "x:b33_glm_dl": 0.003578608090595638, "x:b33_rmlp_dl": 2.1039961494557752e-05, "x:b33_slr_c003": -0.021056460694309803, "x:b33_rmlp_dl_g2": 0.0032626304941642125, "x:b33_lgb_rawdig": -0.012407106552030773} """)
FILE_I = U(U(apply(STEP_I, L(K4), 'test')) - 2 * ZALL)
# Version 9. Two new public sets of models on the shared folds join the members (152): Weni Mohamed's BPE-digit pack (CC0) and
# yekenot's TabM (PyTabKit), pinned copies checked by SHA-256. Step J is the member step the board model forecasts highest (+1.1).
PINS = find('SHA256SUMS', inside='board-model-pinned-inputs').parent
_sums = dict(l.split()[::-1] for l in (PINS / 'SHA256SUMS').read_text().splitlines())
WENI = ['bpe_cat_d3', 'bpe_cat_d5', 'bpe_glm', 'bpe_lgb_l4', 'realmlp_bpe', 'realmlp_bpe_v2', 'tabm_bpe']
for nm in WENI:
    p = find(f'test_weni_{nm}.npy', _sums[f'test_weni_{nm}.npy'], inside='board-model-pinned-inputs')
    XT['x:weni_' + nm] = np.load(p).astype(np.float64)
_yt = pd.read_csv(find('yekenot_tabm_pytabkit_submission.csv', _sums['yekenot_tabm_pytabkit_submission.csv']))
assert (_yt['id'].to_numpy() == sample['id'].to_numpy()).all()
XT['x:yek_tabm'] = _yt['is_fraud'].to_numpy(np.float64)
STEP_J = json.loads(""" {"s:megayak_5f_lm": 0.0201096306303523, "s:masaya_gpt2": 0.014299719143422892, "s:lm7_5fold": 0.0003055002999445913, "s:gpt2_seed2026": -0.0022093669251530466, "s:tabpfn_slimypunk_tp1": -0.001589793787068905, "s:sadam_tabpfn_100k_tp3": 0.01887483875052712, "s:yekenot_realmlp": -0.006767652564417163, "s:alex_glm_gen": -0.017631727887409153, "s:daoviet_gpdvae": 0.0018616751457693316, "s:gbn_bag5_w512": 0.05864110856501563, "s:sameer_glm10f": 0.0011441343491034144, "s:sadam_v6_gb_xgb": 0.05406017287111838, "s:sadam_v6_gb_lgb": -0.016774333638996053, "s:sadam_v4_cat": -0.014541815178067236, "s:gb_bag5": 0.008479473445194974, "s:megayak_shallow_tree": -0.007098849929335375, "wide": -0.011374779555598152, "b:ebm_all_bag": 0.12303892208870107, "b:ebm_dig": -0.04766376973610407, "b:ebm_dig_bag": 0.03919258415159648, "b:ebm_dig_r7": -0.05033699574598067, "b:ebm_diglead_bag": -0.050199346269327386, "b:ebm_diglead_i25": 0.010822215340533537, "b:ebm_diglead_r11": 0.005497148585299377, "b:ebm_diglead_r13": -0.030281460606191354, "b:ebm_i25_bag": -0.005626495872482505, "b:ebm_i25_bag4": -0.030038878107455178, "b:ebm_lead4_bag": 0.037146942128980474, "b:gb_bag10": 0.0568836405610256, "b:gb_bag5": -0.07516723668815102, "b:gb_l3": 0.010368068811959017, "b:gb_ohk": 0.019799558467395454, "b:gbc_bag3": 0.010324716338377678, "b:gbe_bag3": 0.010202351171661926, "b:gbn_bag5_w128": 0.016227260003242372, "b:gbn_bag5_w512": -0.07109876691150208, "b:gbr_bag3": 0.006490911556619242, "b:gbx_bag3": 0.005205857240258507, "b:gbx_bag5": -0.009674049470951733, "b:glm_dig": 0.009739976496850877, "b:glm_dl": -0.007316051652968116, "b:glm_dl_bag": 0.01892001855573115, "b:glm_dl_s50": -0.026496303836689772, "b:glm_dl_s50_bag": -0.009214171181602076, "b:glm_ohk": 0.004149573910396501, "b:glm_ohk2": -0.046764857929028396, "b:glm_ohk2_bag": 0.031228138747828333, "b:glm_ohk_c01": -0.024800780475699967, "b:glm_ohkl": 0.035154399367471674, "b:glm_ohkl_bag": 0.005787254808164149, "b:glm_sb": -0.0013781890193567457, "b:glm_te": -0.0023882709167907207, "b:lgb_dig_l4": -0.011272098757922013, "b:lgb_dig_l7": -0.007533825239804755, "b:lgb_dig_s100": 0.009950548368170867, "b:lgb_dig_s100_bag": 0.007439445777925338, "b:lgb_dig_s100_bag5": -0.011116021813069364, "b:lgb_dig_s100_l3": -0.020034547863172, "b:lgb_dig_s100_l3_bag": -0.0013383493496727287, "b:lgb_diglead_bag": -0.010585626109871321, "b:lgb_diglead_bag5": 0.03682529084900802, "b:lgb_diglead_l2": 0.0021841252743104887, "b:lgb_diglead_l3": -0.02895731769788367, "b:lgb_diglead_l3_bag": 0.00784688820848679, "b:lgb_diglead_l4": -0.00645381336464669, "b:lgb_dk_l4": -0.0025173437842190088, "b:lgb_raw_l31": -0.004254643815663726, "b:lgb_raw_l4": 0.0053380290514255, "b:lgb_raw_l4_cnt": 0.01017035740263193, "b:lgb_rawdig_l4": 0.0013790836824878953, "b:lgb_te_l4": -0.013219309708365564, "b:lgb_te_l4_s100": 0.0061427754533412504, "b:lgb_te_l4_s300": -0.008948920620284911, "b:lgb_te_l4_s5": -0.0009186680496771312, "b:realmlp_allcat": -0.01861644971006353, "b:realmlp_cnt": -0.005975761888455755, "b:realmlp_dig": 0.010726315853396978, "b:realmlp_dig_s100": -0.010755669654954982, "b:realmlp_dig_s2": 0.004337574471792868, "b:realmlp_rawdig": 0.006857845279459059, "b:realmlp_te": 0.006938916233689258, "b:realmlp_te_cnt": -0.002976144401872747, "b:realmlp_te_s2": 0.002192745149518732, "b:realmlp_v1": -0.0036242497025507983, "b:xgb_dig_d2": 0.009028768227719567, "b:xgb_raw_d3": -0.007591997545204241, "b:xgb_te_d2": 0.012975698584594875, "x:Sold": -0.08446726565075384, "x:rmlpnew": 0.020957845118812046, "x:rmlpold": -0.0075122084986469105, "x:tabm": 0.0017563570711936149, "x:davut": -0.010749634999713235, "x:b33_lgb_g2_plaininit_l4": 0.033240365708930616, "x:b33_lgb_g2_plaininit_l3": 0.012980311409859481, "x:b33_lgb_g2_glminit_l4": 0.006407971406030956, "x:b33_lgb_g2_glminit_l5": 0.01650712886970592, "x:b33_lgb_g2_glminit_s50": -0.03402004136885976, "x:b33_lgb_g2_glminit": -0.006487013572077404, "x:b33_lgb_g2_glminit_l7": -0.010086525062084782, "x:b33_lgb_g2_glminit_l4_hier": -0.019974389359132892, "x:b33_xgb_g2_glminit": 0.03161037036362258, "x:b33_lgb_dl3_g2c3": -0.026735337241699433, "x:b33_lgb_dl3": 0.0004331391844970024, "x:b33_lgb_dl4_g2cc": 0.006895492651783617, "x:b33_lgb_dl4_g2": 0.02351903324347602, "x:b33_glm_plain_g2_s50": -0.06177875973120138, "x:b33_lgb_dl3_cc": 0.018794802803340015, "x:b33_lgb_dl3_hier": 0.018524921829463355, "x:b33_lgb_dl3_trie": 0.003926209581706198, "x:b33_ebm_g2_glminit": -0.038653725502064876, "x:b33_lgb_g2_glminit_l2": 0.006803975234394583, "x:b33_lgb_dl3_s10": 0.001612398844642655, "x:b33_lgb_dl4_hier": 0.026832534462043568, "x:b33_xgb_dl2": 0.0031715325181789944, "x:b33_glm_plain_s50": 0.041251265134422274, "x:b33_lgb_dl4": -0.021948582379731005, "x:b33_lgb_dl4_c3": -0.010638380074620547, "x:b33_lgb_dl3_s100": 0.010338779922913578, "x:b33_lgb_dl4_s100": -0.013922580743216202, "x:b33_lgb_dl4_s10": -0.0004307132903247691, "x:b33_glm_ohk_g2_c0003_s50": 0.03035876383305421, "x:b33_glm_ohk_g2_c0003_s100": -0.03427112451239734, "x:b33_glm_ohk_g2_c0005_s50": 0.0021246268816865416, "x:b33_glm_ohk_g2_c0005_s100": -0.0023247823435039935, "x:b33_glm_ohk_g2_c0002_s50": 0.10269354283012103, "x:b33_glm_ohk_g2_c0003": -0.02861446966748872, "x:b33_glm_ohk_g2_c0005_m2": 0.01927643422064663, "x:b33_glm_ohk_g2_c0005": -0.04015367251116394, "x:b33_lgb_dl2_hier": -0.009590062361566863, "x:b33_glm_ohk_g2x_c0005": -0.0015134772440028388, "x:b33_glm_ohk_c0005": -0.06933063694349918, "x:b33_ebm_dl": -0.0013264447919989124, "x:b33_ebm_dl_g2": 0.00881904663017232, "x:b33_glm_ohk_c0002": 0.010225604839508089, "x:b33_glm_ohk_g2_c001": 0.022927095206761467, "x:b33_glm_ohk_c001": -0.001497243466732728, "x:b33_tabm_dl": -0.003064856272236765, "x:b33_rmlp2_dl_hier": -0.0015821406795060388, "x:b33_rmlp2_dl": -0.008350603257294791, "x:b33_glm_dl": -0.0041362091460714805, "x:b33_rmlp_dl": 0.0009747853658195523, "x:b33_slr_c003": -0.008587228415882482, "x:b33_rmlp_dl_g2": -0.0008756482761504102, "x:b33_lgb_rawdig": -0.010537081306488623, "x:yek_tabm": 0.008960558498179734, "x:weni_bpe_cat_d3": -0.0017722659080590595, "x:weni_bpe_cat_d5": -0.005449550505719869, "x:weni_bpe_glm": 0.008758154735413757, "x:weni_bpe_lgb_l4": 0.008649709694946698, "x:weni_realmlp_bpe": 0.008252142158301572, "x:weni_realmlp_bpe_v2": 0.004596000388819993, "x:weni_tabm_bpe": -0.0018885080753701998} """)
FILE_J = U(U(apply(STEP_J, L(FILE_I), 'test')) - 2 * ZALL)
# Version 10. Malyshev Danil continued step J: two more steps along L(J) - L(I), with its part along L(I) - L(H) removed, all four
# zones at the bottom (public 0.95512). Step K is a member step from that file (152 members): forecast +2.9, nested CV +36.3.
def Zr(x): return U(U(x) - 2 * ZALL)
pJ, qI, rH = Zr(FILE_J), Zr(FILE_I), Zr(FILE_H)
dJ, pr = L(pJ) - L(qI), L(qI) - L(rH)
beta = float(dJ @ pr / (pr @ pr))
MAL2 = Zr(L(pJ) + 2.0 * (dJ - beta * pr))                        # Malyshev Danil's 0.95512 file
STEP_K = json.loads(""" {"s:megayak_5f_lm": 0.018083402386502856, "s:masaya_gpt2": 0.06118168316871995, "s:lm7_5fold": -0.0003828223202094869, "s:gpt2_seed2026": -0.012324560795261886, "s:tabpfn_slimypunk_tp1": -0.008944759868360358, "s:sadam_tabpfn_100k_tp3": -0.002896110250787533, "s:yekenot_realmlp": 0.01221176143202764, "s:alex_glm_gen": -0.09512014765191845, "s:daoviet_gpdvae": -0.029383305990936658, "s:gbn_bag5_w512": 0.11962394540659729, "s:sameer_glm10f": 0.049328239785766864, "s:sadam_v6_gb_xgb": -0.033959171220112994, "s:sadam_v6_gb_lgb": 0.06833742486750927, "s:sadam_v4_cat": 0.008091162074934733, "s:gb_bag5": 0.21016842701825, "s:megayak_shallow_tree": -0.03702046328008544, "wide": -0.013701415618940394, "b:ebm_all_bag": 0.03888539969775905, "b:ebm_dig": -0.04394478389172347, "b:ebm_dig_bag": -0.01029949648734194, "b:ebm_dig_r7": 0.03956122255653992, "b:ebm_diglead_bag": -0.08948116932092832, "b:ebm_diglead_i25": -0.00672497682348085, "b:ebm_diglead_r11": 0.05041516951514694, "b:ebm_diglead_r13": 0.049214324039494176, "b:ebm_i25_bag": -0.09655584094901618, "b:ebm_i25_bag4": 0.14474507210737161, "b:ebm_lead4_bag": -0.04767143393086504, "b:gb_bag10": 0.20635768263748228, "b:gb_bag5": -0.33398784584049274, "b:gb_l3": 0.02483487220706831, "b:gb_ohk": 0.01635501663711867, "b:gbc_bag3": -0.02013067703987033, "b:gbe_bag3": 0.02482224440226775, "b:gbn_bag5_w128": 0.018451847238111187, "b:gbn_bag5_w512": -0.1265466136619155, "b:gbr_bag3": 0.0010438545914767394, "b:gbx_bag3": -0.042931122051826705, "b:gbx_bag5": -0.04804777150130837, "b:glm_dig": 0.013422408582441255, "b:glm_dl": 0.008020112516098781, "b:glm_dl_bag": -0.04994173980772334, "b:glm_dl_s50": 0.03724790408213088, "b:glm_dl_s50_bag": 0.06154296301686809, "b:glm_ohk": -0.04521973921020361, "b:glm_ohk2": -0.08620592039900918, "b:glm_ohk2_bag": -0.028084030833836555, "b:glm_ohk_c01": -0.1401736118627964, "b:glm_ohkl": 0.1980250410143015, "b:glm_ohkl_bag": 0.12828675661864397, "b:glm_sb": -0.004069583151896143, "b:glm_te": -0.01524163056597627, "b:lgb_dig_l4": 0.031891155916914195, "b:lgb_dig_l7": -0.005928681654536205, "b:lgb_dig_s100": 0.03344044664454168, "b:lgb_dig_s100_bag": -0.00840586782458287, "b:lgb_dig_s100_bag5": -0.05112225916971257, "b:lgb_dig_s100_l3": -0.06502591167818289, "b:lgb_dig_s100_l3_bag": 0.05565316708279991, "b:lgb_diglead_bag": -0.13438113944657118, "b:lgb_diglead_bag5": -0.00015037054537604704, "b:lgb_diglead_l2": -0.032565640867518425, "b:lgb_diglead_l3": -0.012022408332317167, "b:lgb_diglead_l3_bag": 0.04130398900230118, "b:lgb_diglead_l4": 0.03325951409961539, "b:lgb_dk_l4": -0.029072819370514907, "b:lgb_raw_l31": -0.0049678324971539406, "b:lgb_raw_l4": 0.029094805080832706, "b:lgb_raw_l4_cnt": 0.0037731598720007367, "b:lgb_rawdig_l4": 0.05650356402063033, "b:lgb_te_l4": -0.005572860410494636, "b:lgb_te_l4_s100": 0.027097863439512197, "b:lgb_te_l4_s300": 0.008091526494333806, "b:lgb_te_l4_s5": -0.025263810682610268, "b:realmlp_allcat": 0.016151673138778393, "b:realmlp_cnt": 0.003251847289433773, "b:realmlp_dig": 0.0011689247067531573, "b:realmlp_dig_s100": -0.005725518127390936, "b:realmlp_dig_s2": 0.020094127081733682, "b:realmlp_rawdig": -0.01631285876679574, "b:realmlp_te": 0.02332983614180154, "b:realmlp_te_cnt": -0.009313422482512547, "b:realmlp_te_s2": -0.011824302538884808, "b:realmlp_v1": -0.024023611050430833, "b:xgb_dig_d2": -0.006404188228887919, "b:xgb_raw_d3": 0.01615274096728025, "b:xgb_te_d2": 0.0057481809781372555, "x:Sold": -0.04441166241900984, "x:rmlpnew": -0.06371643584896411, "x:rmlpold": 0.011898164144577965, "x:tabm": -0.012604041636526092, "x:davut": -0.021435846208281463, "x:b33_lgb_g2_plaininit_l4": 0.10584013511903839, "x:b33_lgb_g2_plaininit_l3": 0.03403711252165987, "x:b33_lgb_g2_glminit_l4": 0.007925735228988977, "x:b33_lgb_g2_glminit_l5": 0.044511543183975454, "x:b33_lgb_g2_glminit_s50": -0.10516132178544547, "x:b33_lgb_g2_glminit": -0.024023015018890682, "x:b33_lgb_g2_glminit_l7": -0.020331062243651672, "x:b33_lgb_g2_glminit_l4_hier": -0.04936768817680434, "x:b33_xgb_g2_glminit": 0.09461224320323208, "x:b33_lgb_dl3_g2c3": -0.07309787101645975, "x:b33_lgb_dl3": 0.010029917173982094, "x:b33_lgb_dl4_g2cc": 0.024784869975885842, "x:b33_lgb_dl4_g2": 0.06554658597940143, "x:b33_glm_plain_g2_s50": -0.16767431209657344, "x:b33_lgb_dl3_cc": 0.05100956844372858, "x:b33_lgb_dl3_hier": 0.05230432163166086, "x:b33_lgb_dl3_trie": 0.007727869129616637, "x:b33_ebm_g2_glminit": -0.10252745423804752, "x:b33_lgb_g2_glminit_l2": 0.0055309788029816185, "x:b33_lgb_dl3_s10": 0.008460663519393636, "x:b33_lgb_dl4_hier": 0.07409960548927993, "x:b33_xgb_dl2": 0.011973140332845444, "x:b33_glm_plain_s50": 0.05819558229673518, "x:b33_lgb_dl4": -0.06283251500066425, "x:b33_lgb_dl4_c3": -0.02558871760259049, "x:b33_lgb_dl3_s100": 0.03288383591467086, "x:b33_lgb_dl4_s100": -0.039071334859983584, "x:b33_lgb_dl4_s10": -0.00499487973748171, "x:b33_glm_ohk_g2_c0003_s50": 0.05098669025761595, "x:b33_glm_ohk_g2_c0003_s100": -0.03265124082107479, "x:b33_glm_ohk_g2_c0005_s50": 0.03634590211658706, "x:b33_glm_ohk_g2_c0005_s100": -0.03593077665327265, "x:b33_glm_ohk_g2_c0002_s50": 0.21689509569655058, "x:b33_glm_ohk_g2_c0003": -0.050371192043419805, "x:b33_glm_ohk_g2_c0005_m2": 0.08464250275375348, "x:b33_glm_ohk_g2_c0005": -0.11713103867602774, "x:b33_lgb_dl2_hier": -0.021708321649616863, "x:b33_glm_ohk_g2x_c0005": -0.0015174295172973834, "x:b33_glm_ohk_c0005": -0.10774792846372006, "x:b33_ebm_dl": -0.00222205817553192, "x:b33_ebm_dl_g2": 0.020554059193896402, "x:b33_glm_ohk_c0002": 0.021531655830429694, "x:b33_glm_ohk_g2_c001": 0.052956872245132035, "x:b33_glm_ohk_c001": -0.07844137741641195, "x:b33_tabm_dl": -0.008587040441449308, "x:b33_rmlp2_dl_hier": -0.013849538064177073, "x:b33_rmlp2_dl": -0.01534579478345848, "x:b33_glm_dl": -0.016226908083560112, "x:b33_rmlp_dl": 0.005465589064710509, "x:b33_slr_c003": -0.025287063440033467, "x:b33_rmlp_dl_g2": -0.0048273382471039385, "x:b33_lgb_rawdig": -0.03461787404109455, "x:yek_tabm": 0.040750646075831655, "x:weni_bpe_cat_d3": -0.003552791408223282, "x:weni_bpe_cat_d5": -0.02511357598731016, "x:weni_bpe_glm": 0.02716941738384144, "x:weni_bpe_lgb_l4": 0.03810605960734461, "x:weni_realmlp_bpe": 0.0316350550116824, "x:weni_realmlp_bpe_v2": 0.020388194149291935, "x:weni_tabm_bpe": -0.011362608338963002} """)
FILE_K = Zr(apply(STEP_K, L(MAL2), 'test'))
# Version 12. Malyshev Danil: K x 1.5, i.e. K + 0.5 (K - M) with all four zones at the bottom (public 0.95520). Step L is a member
# step from there (152 members): forecast +3.4, replayed OOF +10.1 against that file.
pK, qM = Zr(FILE_K), Zr(MAL2)
MAL3 = Zr(L(pK) + 0.5 * (L(pK) - L(qM)))                          # Malyshev Danil's 0.95520 file
STEP_L = json.loads(""" {"s:megayak_5f_lm": -0.04173080551013239, "s:masaya_gpt2": 0.030632726007662398, "s:lm7_5fold": -0.008208222861721754, "s:gpt2_seed2026": -0.0013637070103617805, "s:tabpfn_slimypunk_tp1": 0.007216667862337375, "s:sadam_tabpfn_100k_tp3": -0.020413395814966616, "s:yekenot_realmlp": 0.003761745682047529, "s:alex_glm_gen": -0.08745589099608565, "s:daoviet_gpdvae": -0.07066302148230713, "s:gbn_bag5_w512": 0.11981808824293348, "s:sameer_glm10f": -0.012671331411961787, "s:sadam_v6_gb_xgb": -0.07076168693665587, "s:sadam_v6_gb_lgb": 0.06101647205892077, "s:sadam_v4_cat": 0.009346810586209422, "s:gb_bag5": 0.31121014686892157, "s:megayak_shallow_tree": 0.1214204560193142, "wide": 0.007439518893256329, "b:ebm_all_bag": -0.12210114756793809, "b:ebm_dig": 0.05452951267245279, "b:ebm_dig_bag": -0.08333585772701416, "b:ebm_dig_r7": 0.10845629514233379, "b:ebm_diglead_bag": -0.1045802336095453, "b:ebm_diglead_i25": -7.579634688909871e-05, "b:ebm_diglead_r11": 0.1291336394511053, "b:ebm_diglead_r13": 0.0647532373266066, "b:ebm_i25_bag": -0.05489531492538302, "b:ebm_i25_bag4": 0.12804297592946343, "b:ebm_lead4_bag": -0.0740420602486313, "b:gb_bag10": -0.030514143489245027, "b:gb_bag5": -0.25241965586785436, "b:gb_l3": -0.030164391612414618, "b:gb_ohk": -0.04671435685187768, "b:gbc_bag3": 0.009231491519900305, "b:gbe_bag3": 0.009636071421165179, "b:gbn_bag5_w128": -0.04549334356517955, "b:gbn_bag5_w512": 0.007830484791237394, "b:gbr_bag3": -0.04035316961376743, "b:gbx_bag3": -0.052200177443598075, "b:gbx_bag5": 0.006708541478518142, "b:glm_dig": 0.021932540957198356, "b:glm_dl": 0.006467895346866162, "b:glm_dl_bag": -0.0762588794546099, "b:glm_dl_s50": 0.032174911833249165, "b:glm_dl_s50_bag": 0.0029380217188125666, "b:glm_ohk": -0.05307483651848588, "b:glm_ohk2": -0.005769008369429717, "b:glm_ohk2_bag": -0.12880960610505307, "b:glm_ohk_c01": -0.1199951598052977, "b:glm_ohkl": 0.11671269657182683, "b:glm_ohkl_bag": 0.19538330593294806, "b:glm_sb": 0.013422861918114863, "b:glm_te": 0.014566100045944481, "b:lgb_dig_l4": -0.032589570284296814, "b:lgb_dig_l7": 0.016021539088003904, "b:lgb_dig_s100": 0.0393965332897228, "b:lgb_dig_s100_bag": -0.050286241055178954, "b:lgb_dig_s100_bag5": -0.04508378380872484, "b:lgb_dig_s100_l3": -0.060333163536650636, "b:lgb_dig_s100_l3_bag": 0.07679556707926669, "b:lgb_diglead_bag": -0.12204405184259029, "b:lgb_diglead_bag5": -0.03581053060759408, "b:lgb_diglead_l2": -0.04681837450322543, "b:lgb_diglead_l3": 0.018963854294126992, "b:lgb_diglead_l3_bag": 0.04628902328382647, "b:lgb_diglead_l4": 0.03193363357533817, "b:lgb_dk_l4": -0.026229159083112713, "b:lgb_raw_l31": -0.005113737096531116, "b:lgb_raw_l4": 0.04276728016052933, "b:lgb_raw_l4_cnt": -0.021405502930248326, "b:lgb_rawdig_l4": 0.04681017464202399, "b:lgb_te_l4": 0.06746196632681102, "b:lgb_te_l4_s100": 0.027217435889231326, "b:lgb_te_l4_s300": -0.018703006776630247, "b:lgb_te_l4_s5": -0.038736609433583605, "b:realmlp_allcat": -0.014832870660028507, "b:realmlp_cnt": -0.010230533666677404, "b:realmlp_dig": 0.01959726431798909, "b:realmlp_dig_s100": -0.01995702652362139, "b:realmlp_dig_s2": 0.027768340213206504, "b:realmlp_rawdig": -0.024336964550019893, "b:realmlp_te": -0.0038298736395904116, "b:realmlp_te_cnt": -0.010322868712091334, "b:realmlp_te_s2": 0.017037580730046506, "b:realmlp_v1": 0.03409614726497732, "b:xgb_dig_d2": -0.013619407782576068, "b:xgb_raw_d3": 0.018526346733042633, "b:xgb_te_d2": -0.04030804594531376, "x:Sold": -0.04807029970554145, "x:rmlpnew": -0.04264878653923022, "x:rmlpold": 0.0036330357503414914, "x:tabm": -0.007071327949694093, "x:davut": 0.0054220961773129815, "x:b33_lgb_g2_plaininit_l4": 0.0903962255035127, "x:b33_lgb_g2_plaininit_l3": 0.0038113464688459614, "x:b33_lgb_g2_glminit_l4": -0.009167862262894974, "x:b33_lgb_g2_glminit_l5": 0.0033754327643387024, "x:b33_lgb_g2_glminit_s50": -0.10100250318675162, "x:b33_lgb_g2_glminit": -0.034094523508362604, "x:b33_lgb_g2_glminit_l7": 0.016180978259569462, "x:b33_lgb_g2_glminit_l4_hier": 0.008960046931216943, "x:b33_xgb_g2_glminit": 0.044846152493792736, "x:b33_lgb_dl3_g2c3": -0.021579794932420286, "x:b33_lgb_dl3": 0.02696299440428946, "x:b33_lgb_dl4_g2cc": 0.024819952270213223, "x:b33_lgb_dl4_g2": 0.03840942002829257, "x:b33_glm_plain_g2_s50": -0.09191178476855927, "x:b33_lgb_dl3_cc": 0.00624439468881254, "x:b33_lgb_dl3_hier": 0.018055676476404256, "x:b33_lgb_dl3_trie": -0.011678220421014724, "x:b33_ebm_g2_glminit": -0.01007989711015071, "x:b33_lgb_g2_glminit_l2": -0.03164383172864454, "x:b33_lgb_dl3_s10": 0.016372141849300734, "x:b33_lgb_dl4_hier": 0.025131328274975967, "x:b33_xgb_dl2": 0.017257396379944594, "x:b33_glm_plain_s50": 0.03993682843357103, "x:b33_lgb_dl4": -0.020608617741591017, "x:b33_lgb_dl4_c3": 0.0073230848950451444, "x:b33_lgb_dl3_s100": 0.023727967747983195, "x:b33_lgb_dl4_s100": -0.00364166131447562, "x:b33_lgb_dl4_s10": -0.011439651916813912, "x:b33_glm_ohk_g2_c0003_s50": -0.02557185153561022, "x:b33_glm_ohk_g2_c0003_s100": -0.006972253324346642, "x:b33_glm_ohk_g2_c0005_s50": 0.09022801928717622, "x:b33_glm_ohk_g2_c0005_s100": -0.012477265893143356, "x:b33_glm_ohk_g2_c0002_s50": 0.05926930002661718, "x:b33_glm_ohk_g2_c0003": -0.058948162018112954, "x:b33_glm_ohk_g2_c0005_m2": 0.15524479835554886, "x:b33_glm_ohk_g2_c0005": -0.047144405032753994, "x:b33_lgb_dl2_hier": 0.008459795636435341, "x:b33_glm_ohk_g2x_c0005": 0.009497011713278347, "x:b33_glm_ohk_c0005": -0.01730269024138211, "x:b33_ebm_dl": -0.0019835918331806467, "x:b33_ebm_dl_g2": -0.006143501868365741, "x:b33_glm_ohk_c0002": 0.005793602818302212, "x:b33_glm_ohk_g2_c001": -0.011899847912211818, "x:b33_glm_ohk_c001": -0.07688299043485118, "x:b33_tabm_dl": -0.0028391497074409314, "x:b33_rmlp2_dl_hier": -0.035165657595455105, "x:b33_rmlp2_dl": 0.02604520605158797, "x:b33_glm_dl": -0.030516598378983607, "x:b33_rmlp_dl": 0.009111743200653619, "x:b33_slr_c003": -0.0016961002959278207, "x:b33_rmlp_dl_g2": -0.011313353159873085, "x:b33_lgb_rawdig": -0.02956719662373672, "x:yek_tabm": 0.07279998140388713, "x:weni_bpe_cat_d3": 0.006804541810242079, "x:weni_bpe_cat_d5": -0.04738113975600276, "x:weni_bpe_glm": 0.027105340233134698, "x:weni_bpe_lgb_l4": 0.0659969433724099, "x:weni_realmlp_bpe": 0.04274022331741465, "x:weni_realmlp_bpe_v2": 0.04054518769305437, "x:weni_tabm_bpe": -0.028201977345767987} """)
FILE_L = Zr(apply(STEP_L, L(MAL3), 'test'))
# Version 14. Step M: from step L, a quarter more of step L's own direction (from the 0.95520 file) and a member step (152 members),
# both taken 1.5 times (the top of the forecast along this direction).
STEP_S1 = json.loads(""" {"s:megayak_5f_lm": -0.015141398443956235, "s:masaya_gpt2": -0.021529945429482916, "s:lm7_5fold": -0.0055602698530884545, "s:gpt2_seed2026": 0.013821394312516675, "s:tabpfn_slimypunk_tp1": 0.007422719158725368, "s:sadam_tabpfn_100k_tp3": -0.002699365655075782, "s:yekenot_realmlp": -0.003556784483892451, "s:alex_glm_gen": -0.036597271155633146, "s:daoviet_gpdvae": -0.025279179008815933, "s:gbn_bag5_w512": -0.006308409234113186, "s:sameer_glm10f": -0.027175154063579714, "s:sadam_v6_gb_xgb": -0.03985790682683261, "s:sadam_v6_gb_lgb": 0.02186491582302025, "s:sadam_v4_cat": 0.009473953235272772, "s:gb_bag5": 0.12987374119730816, "s:megayak_shallow_tree": 0.07226092862764785, "wide": 0.011996964585757107, "b:ebm_all_bag": -0.05661934178952769, "b:ebm_dig": 0.02356875871040208, "b:ebm_dig_bag": -0.013178688150612778, "b:ebm_dig_r7": 0.021619305819711464, "b:ebm_diglead_bag": -0.05170680176969919, "b:ebm_diglead_i25": 0.0019256343913583716, "b:ebm_diglead_r11": 0.059735912326080984, "b:ebm_diglead_r13": 0.015093602986076097, "b:ebm_i25_bag": 0.004403344081222911, "b:ebm_i25_bag4": 0.03611865850242668, "b:ebm_lead4_bag": -0.02020515535447946, "b:gb_bag10": -0.10359126658014457, "b:gb_bag5": -0.049506769768049345, "b:gb_l3": -0.046700029676696, "b:gb_ohk": -0.013181291393494636, "b:gbc_bag3": 0.01053166997811733, "b:gbe_bag3": 0.012918733674990248, "b:gbn_bag5_w128": -0.0001480048673836442, "b:gbn_bag5_w512": 0.026643103058699075, "b:gbr_bag3": -0.023242137889939778, "b:gbx_bag3": -0.028795078276432153, "b:gbx_bag5": 0.03743369346607733, "b:glm_dig": 0.02182403828695298, "b:glm_dl": 0.05593737832494088, "b:glm_dl_bag": -0.029839956730926757, "b:glm_dl_s50": -0.006560690209647738, "b:glm_dl_s50_bag": -0.02567138870179004, "b:glm_ohk": 0.015567746868632866, "b:glm_ohk2": -0.0027287083328038386, "b:glm_ohk2_bag": -0.018921295926540807, "b:glm_ohk_c01": -0.06412836018865581, "b:glm_ohkl": -0.03765571472859036, "b:glm_ohkl_bag": 0.06868347568099213, "b:glm_sb": 0.012670308735859573, "b:glm_te": 0.014980745935814404, "b:lgb_dig_l4": -0.01318510108961463, "b:lgb_dig_l7": 0.019068846090136586, "b:lgb_dig_s100": 0.007116566229376373, "b:lgb_dig_s100_bag": -0.010873718941142347, "b:lgb_dig_s100_bag5": -0.0035359939482336547, "b:lgb_dig_s100_l3": -0.0014653982947858723, "b:lgb_dig_s100_l3_bag": 0.01441317251373006, "b:lgb_diglead_bag": 0.009204625374757792, "b:lgb_diglead_bag5": -0.07340844830697696, "b:lgb_diglead_l2": -0.04083785091208743, "b:lgb_diglead_l3": 0.03403602289060298, "b:lgb_diglead_l3_bag": 0.002803967296356646, "b:lgb_diglead_l4": -0.001817609417309018, "b:lgb_dk_l4": -0.004695001396799933, "b:lgb_raw_l31": -0.00035060881212215406, "b:lgb_raw_l4": 0.006507954933136115, "b:lgb_raw_l4_cnt": -0.018034533070403504, "b:lgb_rawdig_l4": 0.012391542375025204, "b:lgb_te_l4": 0.042305642425282744, "b:lgb_te_l4_s100": -0.004908981784098958, "b:lgb_te_l4_s300": -0.00865128739939549, "b:lgb_te_l4_s5": -0.020930296813484746, "b:realmlp_allcat": -0.011834017602545158, "b:realmlp_cnt": -0.0021377638825472957, "b:realmlp_dig": 0.00753594117239801, "b:realmlp_dig_s100": -0.0035229189880657667, "b:realmlp_dig_s2": 0.010955599337822053, "b:realmlp_rawdig": -0.013715954313774523, "b:realmlp_te": -0.01823675088516174, "b:realmlp_te_cnt": 0.0008977529900402069, "b:realmlp_te_s2": 0.0061222321775363155, "b:realmlp_v1": 0.03375637303269713, "b:xgb_dig_d2": -0.01332528734164308, "b:xgb_raw_d3": 0.011536245368742468, "b:xgb_te_d2": -0.027556430026994788, "x:Sold": -0.00044153889581631815, "x:rmlpnew": -0.021240263396698864, "x:rmlpold": -0.003356376331449074, "x:tabm": 0.006173372710673793, "x:davut": 0.012849564084711823, "x:b33_lgb_g2_plaininit_l4": 0.02382384198589113, "x:b33_lgb_g2_plaininit_l3": -0.012115136525389623, "x:b33_lgb_g2_glminit_l4": -0.016132335897281233, "x:b33_lgb_g2_glminit_l5": 0.0026359121139384748, "x:b33_lgb_g2_glminit_s50": -0.031584331907150626, "x:b33_lgb_g2_glminit": -0.02967515959172464, "x:b33_lgb_g2_glminit_l7": 0.023877285939067048, "x:b33_lgb_g2_glminit_l4_hier": 0.027773313733966243, "x:b33_xgb_g2_glminit": 0.006460202918125261, "x:b33_lgb_dl3_g2c3": 0.015901025868777185, "x:b33_lgb_dl3": 0.02495642536983631, "x:b33_lgb_dl4_g2cc": 0.010609750908484884, "x:b33_lgb_dl4_g2": -0.0015166916603501447, "x:b33_glm_plain_g2_s50": 0.005226714244247069, "x:b33_lgb_dl3_cc": -0.015462326448125743, "x:b33_lgb_dl3_hier": -0.006616704335144231, "x:b33_lgb_dl3_trie": -0.013404899581219409, "x:b33_ebm_g2_glminit": 0.04782295463984286, "x:b33_lgb_g2_glminit_l2": -0.032817019787363906, "x:b33_lgb_dl3_s10": 0.008192227506931146, "x:b33_lgb_dl4_hier": -0.01623141811963439, "x:b33_xgb_dl2": 0.009141359879987328, "x:b33_glm_plain_s50": -0.023755023891704197, "x:b33_lgb_dl4": 0.004059399577782345, "x:b33_lgb_dl4_c3": 0.017459849998233856, "x:b33_lgb_dl3_s100": 0.0026048062312309904, "x:b33_lgb_dl4_s100": 0.011435896331305786, "x:b33_lgb_dl4_s10": -0.005152269771275208, "x:b33_glm_ohk_g2_c0003_s50": -0.05466118838974102, "x:b33_glm_ohk_g2_c0003_s100": -0.049451223251886514, "x:b33_glm_ohk_g2_c0005_s50": 0.04566897865615226, "x:b33_glm_ohk_g2_c0005_s100": 0.04955055196176928, "x:b33_glm_ohk_g2_c0002_s50": -0.022424951558711336, "x:b33_glm_ohk_g2_c0003": -0.025816277359087, "x:b33_glm_ohk_g2_c0005_m2": 0.11676154570544163, "x:b33_glm_ohk_g2_c0005": -0.018547748639734626, "x:b33_lgb_dl2_hier": 0.01621655076308599, "x:b33_glm_ohk_g2x_c0005": 0.009980532596592534, "x:b33_glm_ohk_c0005": 0.0017862966036325124, "x:b33_ebm_dl": -0.0022750594147963425, "x:b33_ebm_dl_g2": -0.01842037321510717, "x:b33_glm_ohk_c0002": -0.0037120991817951015, "x:b33_glm_ohk_g2_c001": -0.03902681335745781, "x:b33_glm_ohk_c001": 0.007204113753452678, "x:b33_tabm_dl": 0.0022134474772366837, "x:b33_rmlp2_dl_hier": -0.012940317407710157, "x:b33_rmlp2_dl": 0.021093735839016133, "x:b33_glm_dl": -0.02082041196043943, "x:b33_rmlp_dl": 0.005239641886073288, "x:b33_slr_c003": 0.008444182984741895, "x:b33_rmlp_dl_g2": -0.006298713346045701, "x:b33_lgb_rawdig": -0.009811264986178535, "x:yek_tabm": 0.05180694384128274, "x:weni_bpe_cat_d3": 0.0009995617565537154, "x:weni_bpe_cat_d5": -0.029576468553692437, "x:weni_bpe_glm": 0.015818540267569604, "x:weni_bpe_lgb_l4": 0.044247662787100496, "x:weni_realmlp_bpe": 0.019539044107331104, "x:weni_realmlp_bpe_v2": 0.03453399890020754, "x:weni_tabm_bpe": -0.025601206907063766} """)
S1 = apply(STEP_S1, L(FILE_L), 'test')
FILE_M = Zr(L(FILE_L) + 1.5 * (0.25 * (L(FILE_L) - L(MAL3)) + 0.75 * (L(S1) - L(FILE_L))))
# Version 16. Step N: a member step from step M (152 members), board model refitted on 167 scored files.
STEP_N = json.loads(""" {"s:megayak_5f_lm": 0.010521189818831182, "s:masaya_gpt2": -0.006443155898413498, "s:lm7_5fold": -0.0013739784893339758, "s:gpt2_seed2026": 0.0005369997045449071, "s:tabpfn_slimypunk_tp1": -0.00147416953278009, "s:sadam_tabpfn_100k_tp3": 0.010932410433044965, "s:yekenot_realmlp": -0.005232527976279724, "s:alex_glm_gen": 0.0026569217801122934, "s:daoviet_gpdvae": 0.012108292710641904, "s:gbn_bag5_w512": -0.02865245614135055, "s:sameer_glm10f": -0.009207136171976462, "s:sadam_v6_gb_xgb": 0.02857759366946123, "s:sadam_v6_gb_lgb": -0.016297304349502557, "s:sadam_v4_cat": -0.005975855326814823, "s:gb_bag5": 0.013212279609388798, "s:megayak_shallow_tree": -0.0005906063994652295, "wide": -0.005803431697931965, "b:ebm_all_bag": 0.02072571365619569, "b:ebm_dig": -0.004737358923468325, "b:ebm_dig_bag": 0.013211893024442025, "b:ebm_dig_r7": -0.025409603024724, "b:ebm_diglead_bag": -0.04139176712525337, "b:ebm_diglead_i25": -0.0013369443462755014, "b:ebm_diglead_r11": 0.015717414386486748, "b:ebm_diglead_r13": 0.00030189427298231154, "b:ebm_i25_bag": 0.029410103854896508, "b:ebm_i25_bag4": -0.04082712562627962, "b:ebm_lead4_bag": 0.028895703685630304, "b:gb_bag10": -0.06135578704519111, "b:gb_bag5": 0.03110476184800366, "b:gb_l3": -0.007043563857563024, "b:gb_ohk": 0.016106510718494726, "b:gbc_bag3": 0.009445118390473939, "b:gbe_bag3": 0.006565216815254084, "b:gbn_bag5_w128": 0.01806991625972964, "b:gbn_bag5_w512": -0.006019416289862266, "b:gbr_bag3": 0.005734030706753419, "b:gbx_bag3": 0.00039513027210008383, "b:gbx_bag5": 0.01591879863697537, "b:glm_dig": 0.006755857626594087, "b:glm_dl": 0.013153950468062352, "b:glm_dl_bag": 0.027496141162457337, "b:glm_dl_s50": -0.02665575070948645, "b:glm_dl_s50_bag": -0.01705483155197905, "b:glm_ohk": 0.012654173050452449, "b:glm_ohk2": 0.017844970945076203, "b:glm_ohk2_bag": 0.017420836472654588, "b:glm_ohk_c01": 0.006677375020734934, "b:glm_ohkl": -0.06513409396535998, "b:glm_ohkl_bag": 0.0023460791892000853, "b:glm_sb": -0.0010107255619049777, "b:glm_te": 0.0008590513218271586, "b:lgb_dig_l4": -0.004200940785538199, "b:lgb_dig_l7": -0.0007728770123097502, "b:lgb_dig_s100": -0.005763109008736669, "b:lgb_dig_s100_bag": 0.011072832166305812, "b:lgb_dig_s100_bag5": 0.010087740978649864, "b:lgb_dig_s100_l3": 0.008558328039214374, "b:lgb_dig_s100_l3_bag": -0.01895035315724847, "b:lgb_diglead_bag": 0.03825510278969489, "b:lgb_diglead_bag5": -0.005311856777775573, "b:lgb_diglead_l2": 0.0008201712273820268, "b:lgb_diglead_l3": -0.002469307394751763, "b:lgb_diglead_l3_bag": -0.009320543050926272, "b:lgb_diglead_l4": -0.014420197545251307, "b:lgb_dk_l4": 0.004579772730643536, "b:lgb_raw_l31": -0.0021108441635803747, "b:lgb_raw_l4": -0.005049311797705925, "b:lgb_raw_l4_cnt": 0.0024898291839791195, "b:lgb_rawdig_l4": -0.011686045793564623, "b:lgb_te_l4": -0.004013029893013936, "b:lgb_te_l4_s100": -0.007017268737811428, "b:lgb_te_l4_s300": -0.003943064268630365, "b:lgb_te_l4_s5": 0.0049420829678816815, "b:realmlp_allcat": -0.0115994720778325, "b:realmlp_cnt": -0.001432633569998195, "b:realmlp_dig": 0.0025736906827299975, "b:realmlp_dig_s100": -0.0014951598351899723, "b:realmlp_dig_s2": -0.0032167491363314406, "b:realmlp_rawdig": 0.005423836423540621, "b:realmlp_te": -0.002406493265824069, "b:realmlp_te_cnt": 0.001155778486525131, "b:realmlp_te_s2": -0.0004081591010756789, "b:realmlp_v1": 0.001614388676770948, "b:xgb_dig_d2": 0.006478627959725462, "b:xgb_raw_d3": -0.0027106452702697643, "b:xgb_te_d2": 0.006543027334478301, "x:Sold": -0.024001840220190827, "x:rmlpnew": 0.010816193538087942, "x:rmlpold": -0.0052507223491872305, "x:tabm": 0.004950658483046532, "x:davut": 0.003622631668582493, "x:b33_lgb_g2_plaininit_l4": -0.008569155111938764, "x:b33_lgb_g2_plaininit_l3": -0.0027506149065520164, "x:b33_lgb_g2_glminit_l4": -0.0018439271477020812, "x:b33_lgb_g2_glminit_l5": -0.006492570318176961, "x:b33_lgb_g2_glminit_s50": 0.004403058848020574, "x:b33_lgb_g2_glminit": 0.002185423818824443, "x:b33_lgb_g2_glminit_l7": 0.005818921598060031, "x:b33_lgb_g2_glminit_l4_hier": 0.004719411050982345, "x:b33_xgb_g2_glminit": -0.007309770684912769, "x:b33_lgb_dl3_g2c3": 0.01147059405556409, "x:b33_lgb_dl3": 0.0012440902134501725, "x:b33_lgb_dl4_g2cc": -0.0017831649110282251, "x:b33_lgb_dl4_g2": -0.007218221529525728, "x:b33_glm_plain_g2_s50": 0.013324733845847302, "x:b33_lgb_dl3_cc": -0.007371126192870083, "x:b33_lgb_dl3_hier": -0.006214904770343904, "x:b33_lgb_dl3_trie": -0.002578921183542443, "x:b33_ebm_g2_glminit": 0.01934782950020029, "x:b33_lgb_g2_glminit_l2": -0.010487246243744492, "x:b33_lgb_dl3_s10": 0.0014709106604508548, "x:b33_lgb_dl4_hier": -0.009329769897017845, "x:b33_xgb_dl2": 0.001506290608710746, "x:b33_glm_plain_s50": -0.009501530238462222, "x:b33_lgb_dl4": 0.007405585977625172, "x:b33_lgb_dl4_c3": 0.004297067954051827, "x:b33_lgb_dl3_s100": -0.0033534883477068025, "x:b33_lgb_dl4_s100": 0.006051479198673771, "x:b33_lgb_dl4_s10": -0.0005848374817050753, "x:b33_glm_ohk_g2_c0003_s50": -0.0025353858618167016, "x:b33_glm_ohk_g2_c0003_s100": -0.015984264881894916, "x:b33_glm_ohk_g2_c0005_s50": 0.0025323519172317636, "x:b33_glm_ohk_g2_c0005_s100": 0.003171555148574851, "x:b33_glm_ohk_g2_c0002_s50": 0.002913462948303068, "x:b33_glm_ohk_g2_c0003": -0.022811146031764297, "x:b33_glm_ohk_g2_c0005_m2": 0.025312519695627863, "x:b33_glm_ohk_g2_c0005": 0.00015272729394979212, "x:b33_lgb_dl2_hier": 0.0036923632598210186, "x:b33_glm_ohk_g2x_c0005": 0.00179653812529919, "x:b33_glm_ohk_c0005": -0.031083587662903513, "x:b33_ebm_dl": -0.0022238204725617754, "x:b33_ebm_dl_g2": -0.0049734364292514256, "x:b33_glm_ohk_c0002": 0.007548031542184644, "x:b33_glm_ohk_g2_c001": -0.007923832564601434, "x:b33_glm_ohk_c001": 0.02496278703051348, "x:b33_tabm_dl": 0.0017043588316559485, "x:b33_rmlp2_dl_hier": 0.00013673366877395225, "x:b33_rmlp2_dl": 0.00538272344109599, "x:b33_glm_dl": -0.0018256483807489476, "x:b33_rmlp_dl": -0.000327061085386204, "x:b33_slr_c003": 0.003558923586983793, "x:b33_rmlp_dl_g2": -0.00014917944915165545, "x:b33_lgb_rawdig": 0.0030550499368766234, "x:yek_tabm": 0.002188755027930584, "x:weni_bpe_cat_d3": 0.0009785670993113374, "x:weni_bpe_cat_d5": -0.0029989102029196043, "x:weni_bpe_glm": 0.0018168938549424308, "x:weni_bpe_lgb_l4": 0.0020955787072872043, "x:weni_realmlp_bpe": -0.0017310709975595642, "x:weni_realmlp_bpe_v2": 0.003626380294404403, "x:weni_tabm_bpe": -0.0020418315978584073} """)
FILE_N = Zr(apply(STEP_N, L(FILE_M), 'test'))
# Version 18. The 0.95535 file (Rishabh Bafna) on Areeb Ali's residual: r = d - beta (L(F) - L(M)), d = L(N) - L(M), F = Malyshev
# Danil's 0.95527 file (pinned copy); P = N + 0.5 r (0.95534); file = L(P) + 2 (L(P) - L(N)), zones at the bottom. Step O: a member
# step from there (152 members).
FMAL = pd.read_csv(find('malyshevdanil_step_l_moved_0p95527.csv', _sums['malyshevdanil_step_l_moved_0p95527.csv']))['is_fraud'].to_numpy(np.float64)
dN, eF = L(FILE_N) - L(FILE_M), L(FMAL) - L(FILE_M)
rN = dN - float(dN @ eF / (eF @ eF)) * eF
P_AREEB = U(L(FILE_N) + 0.5 * rN)                                 # Areeb Ali's 0.95534 file
RIS = Zr(L(P_AREEB) + 2.0 * (L(P_AREEB) - L(FILE_N)))             # Rishabh Bafna's 0.95535 file
STEP_O = json.loads(""" {"s:megayak_5f_lm": -0.0009918504068583437, "s:masaya_gpt2": 0.02147478847568606, "s:lm7_5fold": -0.0014657035109584687, "s:gpt2_seed2026": -0.007076114317283537, "s:tabpfn_slimypunk_tp1": -0.000959391193931261, "s:sadam_tabpfn_100k_tp3": 0.006931860589310555, "s:yekenot_realmlp": -0.003385259756481499, "s:alex_glm_gen": -0.01377213186963978, "s:daoviet_gpdvae": -0.003504964021392502, "s:gbn_bag5_w512": -0.005862584574117686, "s:sameer_glm10f": -0.003721807972731212, "s:sadam_v6_gb_xgb": 0.029052736837800228, "s:sadam_v6_gb_lgb": -0.003360276766164356, "s:sadam_v4_cat": -0.005753681677798373, "s:gb_bag5": 0.04931972725920131, "s:megayak_shallow_tree": 0.02043724992546142, "wide": -0.0022580475735005574, "b:ebm_all_bag": 0.0027037588368368365, "b:ebm_dig": -0.014631021701483794, "b:ebm_dig_bag": 0.019329677399057298, "b:ebm_dig_r7": -0.014581607305929596, "b:ebm_diglead_bag": -0.05010650392104167, "b:ebm_diglead_i25": -0.0058294202742857106, "b:ebm_diglead_r11": 0.031059411415375743, "b:ebm_diglead_r13": 0.0036456831457350714, "b:ebm_i25_bag": 0.017384076836981475, "b:ebm_i25_bag4": -0.01741028649581878, "b:ebm_lead4_bag": 0.02791858937539589, "b:gb_bag10": -0.03053584061215566, "b:gb_bag5": -0.011995403824742089, "b:gb_l3": 0.0037548981815501196, "b:gb_ohk": 0.004962441892727571, "b:gbc_bag3": 0.007590811102375602, "b:gbe_bag3": 0.005990727102944835, "b:gbn_bag5_w128": 0.016512037296458268, "b:gbn_bag5_w512": -0.01091278002997241, "b:gbr_bag3": -8.8585690706374e-05, "b:gbx_bag3": -0.008522652332096271, "b:gbx_bag5": -0.0006896060878719863, "b:glm_dig": 0.009318197274187144, "b:glm_dl": 0.010830332130292997, "b:glm_dl_bag": 0.021863871282956818, "b:glm_dl_s50": -0.026457934817274698, "b:glm_dl_s50_bag": -0.011080836991882186, "b:glm_ohk": -0.009592052615321442, "b:glm_ohk2": 0.013369306279215403, "b:glm_ohk2_bag": -0.00817673494489482, "b:glm_ohk_c01": -0.023068590166237486, "b:glm_ohkl": -0.04746974940272478, "b:glm_ohkl_bag": 0.030889840283013694, "b:glm_sb": -0.001743132665702919, "b:glm_te": 0.002292801018467519, "b:lgb_dig_l4": -0.011987439227453841, "b:lgb_dig_l7": 0.00021423790069305645, "b:lgb_dig_s100": 0.007844551318615953, "b:lgb_dig_s100_bag": -0.01129086923857201, "b:lgb_dig_s100_bag5": -0.0029734645192744636, "b:lgb_dig_s100_l3": -0.013303414110009749, "b:lgb_dig_s100_l3_bag": 0.003186858497233544, "b:lgb_diglead_bag": -9.627567070493377e-05, "b:lgb_diglead_bag5": -0.0010397886766473532, "b:lgb_diglead_l2": -0.009788215708168293, "b:lgb_diglead_l3": -0.0033506317907932874, "b:lgb_diglead_l3_bag": 0.00861126425053527, "b:lgb_diglead_l4": -0.008733207385621255, "b:lgb_dk_l4": -0.004405781627650546, "b:lgb_raw_l31": -0.0037537620723864192, "b:lgb_raw_l4": 0.005894798367834355, "b:lgb_raw_l4_cnt": 0.001723799547482478, "b:lgb_rawdig_l4": -0.004827667682122572, "b:lgb_te_l4": -0.0014673735258533863, "b:lgb_te_l4_s100": 0.006089014451051141, "b:lgb_te_l4_s300": -0.010145361594766174, "b:lgb_te_l4_s5": 0.002596576296255367, "b:realmlp_allcat": -0.016302445439364637, "b:realmlp_cnt": -0.003935528210041038, "b:realmlp_dig": 0.006244468331086846, "b:realmlp_dig_s100": -0.009399078303512639, "b:realmlp_dig_s2": 0.0006210757588682838, "b:realmlp_rawdig": -0.0007296758069619383, "b:realmlp_te": 0.0037773155093025197, "b:realmlp_te_cnt": -0.002074834102350661, "b:realmlp_te_s2": 0.0023172348043926268, "b:realmlp_v1": 0.003972004699780489, "b:xgb_dig_d2": 0.0162222718663244, "b:xgb_raw_d3": -0.002686553052541288, "b:xgb_te_d2": 0.005801015565507578, "x:Sold": -0.06127165396900509, "x:rmlpnew": 0.007207419981637399, "x:rmlpold": -0.003839600984420287, "x:tabm": 7.99085815527643e-05, "x:davut": 0.007339205559432559, "x:b33_lgb_g2_plaininit_l4": 0.008006493761807192, "x:b33_lgb_g2_plaininit_l3": -0.0027653190442846545, "x:b33_lgb_g2_glminit_l4": -0.00937919612679637, "x:b33_lgb_g2_glminit_l5": -0.021529716275401456, "x:b33_lgb_g2_glminit_s50": -0.020898440177576052, "x:b33_lgb_g2_glminit": 0.006695329103171909, "x:b33_lgb_g2_glminit_l7": 0.008049859536488868, "x:b33_lgb_g2_glminit_l4_hier": 0.014808515714589458, "x:b33_xgb_g2_glminit": -0.016510644449725935, "x:b33_lgb_dl3_g2c3": 0.011401839952562678, "x:b33_lgb_dl3": 0.002465582905770866, "x:b33_lgb_dl4_g2cc": 0.0012092810073581787, "x:b33_lgb_dl4_g2": 0.001441365806509127, "x:b33_glm_plain_g2_s50": -0.013094841130239744, "x:b33_lgb_dl3_cc": -0.014749872868547958, "x:b33_lgb_dl3_hier": -0.010990121304417975, "x:b33_lgb_dl3_trie": -0.0074288577930443495, "x:b33_ebm_g2_glminit": 0.01965925518362415, "x:b33_lgb_g2_glminit_l2": -0.010320401061270275, "x:b33_lgb_dl3_s10": 0.006056701635230777, "x:b33_lgb_dl4_hier": -0.0064958583227475684, "x:b33_xgb_dl2": 0.005599423583765991, "x:b33_glm_plain_s50": 0.012259150495437996, "x:b33_lgb_dl4": 0.01603082870447752, "x:b33_lgb_dl4_c3": 0.011809681370155676, "x:b33_lgb_dl3_s100": 0.0025824911677705484, "x:b33_lgb_dl4_s100": 0.014065314649771302, "x:b33_lgb_dl4_s10": -0.004999777879580155, "x:b33_glm_ohk_g2_c0003_s50": 0.024896382967743675, "x:b33_glm_ohk_g2_c0003_s100": -0.01182877578952544, "x:b33_glm_ohk_g2_c0005_s50": -0.012014498164146748, "x:b33_glm_ohk_g2_c0005_s100": 0.016795547254054775, "x:b33_glm_ohk_g2_c0002_s50": 0.013676023484374775, "x:b33_glm_ohk_g2_c0003": -0.04558815388509186, "x:b33_glm_ohk_g2_c0005_m2": 0.04999154168394958, "x:b33_glm_ohk_g2_c0005": 0.01265059439716096, "x:b33_lgb_dl2_hier": 0.007994985144860698, "x:b33_glm_ohk_g2x_c0005": 0.003493130120652808, "x:b33_glm_ohk_c0005": -0.012529905113605005, "x:b33_ebm_dl": -0.002569178692763733, "x:b33_ebm_dl_g2": -0.00427454256089295, "x:b33_glm_ohk_c0002": -0.003655349243620117, "x:b33_glm_ohk_g2_c001": -0.0118874404526418, "x:b33_glm_ohk_c001": 0.008935299921055087, "x:b33_tabm_dl": 0.0004221120676253505, "x:b33_rmlp2_dl_hier": -0.015436075709296816, "x:b33_rmlp2_dl": 0.02170276019583023, "x:b33_glm_dl": -0.0054235534915086266, "x:b33_rmlp_dl": 0.0011532479189675445, "x:b33_slr_c003": 0.007946834872022654, "x:b33_rmlp_dl_g2": -0.004193962190208483, "x:b33_lgb_rawdig": 0.0003707195735523322, "x:yek_tabm": 0.009210477989446856, "x:weni_bpe_cat_d3": 0.00661430919163227, "x:weni_bpe_cat_d5": -0.014499480248099685, "x:weni_bpe_glm": 0.0062969678233322605, "x:weni_bpe_lgb_l4": 0.012412324689350676, "x:weni_realmlp_bpe": 0.007696578731718148, "x:weni_realmlp_bpe_v2": 0.004023964255108777, "x:weni_tabm_bpe": -0.001959869395201187} """)
FILE_O = Zr(apply(STEP_O, L(RIS), 'test'))
# Version 20. Malyshev Danil's 0.95537 file (step O moved towards 35 scored files; pinned copy). Step P: half a step more along
# step O's own direction (from the 0.95535 file) and a member step (152 members) at 1.25 times.
MALF = pd.read_csv(find('malyshevdanil_forecast_0p95537.csv', _sums['malyshevdanil_forecast_0p95537.csv']))['is_fraud'].to_numpy(np.float64)
STEP_PA = json.loads(""" {"s:megayak_5f_lm": -0.005531363397741428, "s:masaya_gpt2": 0.007700774046400933, "s:lm7_5fold": 0.007647509899670996, "s:gpt2_seed2026": -0.007440389331149139, "s:tabpfn_slimypunk_tp1": 0.00044747067508949906, "s:sadam_tabpfn_100k_tp3": -0.007717912699470571, "s:yekenot_realmlp": 0.0022105011153946974, "s:alex_glm_gen": 0.007476955772344111, "s:daoviet_gpdvae": 0.0077261197021694095, "s:gbn_bag5_w512": -0.030861706841036844, "s:sameer_glm10f": -0.0005106975519914334, "s:sadam_v6_gb_xgb": 0.012707477779787453, "s:sadam_v6_gb_lgb": 0.005885789268523565, "s:sadam_v4_cat": -0.0036999206628665873, "s:gb_bag5": 0.029670210165297348, "s:megayak_shallow_tree": -0.017329035989665873, "wide": -0.004145740312934443, "b:ebm_all_bag": 0.043016941206366274, "b:ebm_dig": -0.035868539334522796, "b:ebm_dig_bag": 0.02375437400214625, "b:ebm_dig_r7": -0.017334287926308093, "b:ebm_diglead_bag": -0.059009457377888576, "b:ebm_diglead_i25": -0.018956436834929, "b:ebm_diglead_r11": 0.013081913140966498, "b:ebm_diglead_r13": 0.004098943021587527, "b:ebm_i25_bag": 0.0012639777938221323, "b:ebm_i25_bag4": 0.004964236278752871, "b:ebm_lead4_bag": 0.02625568396669766, "b:gb_bag10": 0.009265261508302904, "b:gb_bag5": -0.0016256739829125719, "b:gb_l3": 0.030644032692231635, "b:gb_ohk": -0.006432668335429917, "b:gbc_bag3": -0.024237460111406758, "b:gbe_bag3": -0.005620334894495506, "b:gbn_bag5_w128": 0.03292193136776872, "b:gbn_bag5_w512": -0.010217061073422332, "b:gbr_bag3": 0.009096058565264138, "b:gbx_bag3": 0.002490324423663439, "b:gbx_bag5": -0.035565841447851916, "b:glm_dig": 0.01706680877980074, "b:glm_dl": 0.013138983695050246, "b:glm_dl_bag": 0.029431052750822688, "b:glm_dl_s50": -0.0007303700493003315, "b:glm_dl_s50_bag": -0.019200963169800925, "b:glm_ohk": -0.03999584939604021, "b:glm_ohk2": 0.016218391025598803, "b:glm_ohk2_bag": -0.02459530528253442, "b:glm_ohk_c01": -0.01931721668388031, "b:glm_ohkl": -0.08656535726059213, "b:glm_ohkl_bag": 0.03450935252307344, "b:glm_sb": -0.0014030662131532105, "b:glm_te": -0.005178915528382139, "b:lgb_dig_l4": -0.0022895068811897472, "b:lgb_dig_l7": -0.008119199401952652, "b:lgb_dig_s100": 0.03365377849653268, "b:lgb_dig_s100_bag": -0.04342542936016268, "b:lgb_dig_s100_bag5": -0.015282903846304048, "b:lgb_dig_s100_l3": -0.02251428651088315, "b:lgb_dig_s100_l3_bag": 0.033068684496801844, "b:lgb_diglead_bag": -0.03865444772396537, "b:lgb_diglead_bag5": 0.0049388902402674245, "b:lgb_diglead_l2": -0.015738612894019358, "b:lgb_diglead_l3": -0.01686851710219847, "b:lgb_diglead_l3_bag": 0.03281386505342736, "b:lgb_diglead_l4": 0.005285509314528555, "b:lgb_dk_l4": -0.001998559533743483, "b:lgb_raw_l31": 0.004129108033151393, "b:lgb_raw_l4": 0.004708165360044865, "b:lgb_raw_l4_cnt": 0.0001706649657575982, "b:lgb_rawdig_l4": -0.01392096997782938, "b:lgb_te_l4": -0.026791706139516473, "b:lgb_te_l4_s100": 0.007214054836986047, "b:lgb_te_l4_s300": -0.007500324535284161, "b:lgb_te_l4_s5": 0.005134175113076563, "b:realmlp_allcat": -0.003374262904458673, "b:realmlp_cnt": 0.003254666092000185, "b:realmlp_dig": -0.0012896369040942708, "b:realmlp_dig_s100": -0.007048175608410569, "b:realmlp_dig_s2": -0.00018643247519566517, "b:realmlp_rawdig": -0.004103238983633429, "b:realmlp_te": 0.007398317012218358, "b:realmlp_te_cnt": 0.004876105709341487, "b:realmlp_te_s2": 0.00268819419914127, "b:realmlp_v1": -0.0026089674946473958, "b:xgb_dig_d2": 0.036814431939268075, "b:xgb_raw_d3": -0.01199402950744359, "b:xgb_te_d2": 0.015200691916613518, "x:Sold": -0.0019254980570365926, "x:rmlpnew": -0.008038555708590625, "x:rmlpold": 0.0027369200316554048, "x:tabm": -0.0062765284954099014, "x:davut": 0.017144250241036683, "x:b33_lgb_g2_plaininit_l4": -0.010819163371527064, "x:b33_lgb_g2_plaininit_l3": -0.012535343031446397, "x:b33_lgb_g2_glminit_l4": -0.023141234977803116, "x:b33_lgb_g2_glminit_l5": -0.061277588384878054, "x:b33_lgb_g2_glminit_s50": -0.014471847141264631, "x:b33_lgb_g2_glminit": 0.0363003333544036, "x:b33_lgb_g2_glminit_l7": 0.013037858281557665, "x:b33_lgb_g2_glminit_l4_hier": 0.04461775545685638, "x:b33_xgb_g2_glminit": -0.07158649058804586, "x:b33_lgb_dl3_g2c3": 0.030942467282557257, "x:b33_lgb_dl3": -0.005070837056277621, "x:b33_lgb_dl4_g2cc": -0.006261861065658651, "x:b33_lgb_dl4_g2": -0.007137550249632394, "x:b33_glm_plain_g2_s50": 0.004695708687864336, "x:b33_lgb_dl3_cc": -0.042347368328385836, "x:b33_lgb_dl3_hier": -0.03701426470666694, "x:b33_lgb_dl3_trie": -0.014780609549382748, "x:b33_ebm_g2_glminit": 0.048068410606203586, "x:b33_lgb_g2_glminit_l2": -0.004979403331064132, "x:b33_lgb_dl3_s10": 0.008063104337650796, "x:b33_lgb_dl4_hier": -0.02530065105833971, "x:b33_xgb_dl2": 0.00447932215518744, "x:b33_glm_plain_s50": 0.012307188251305713, "x:b33_lgb_dl4": 0.053659229012455276, "x:b33_lgb_dl4_c3": 0.027818418676847773, "x:b33_lgb_dl3_s100": -0.0006490356259837672, "x:b33_lgb_dl4_s100": 0.04030450007488071, "x:b33_lgb_dl4_s10": -0.009051413025577777, "x:b33_glm_ohk_g2_c0003_s50": 0.056756164641241474, "x:b33_glm_ohk_g2_c0003_s100": 0.002306044661814736, "x:b33_glm_ohk_g2_c0005_s50": -0.05021490386767518, "x:b33_glm_ohk_g2_c0005_s100": 0.049085902041579234, "x:b33_glm_ohk_g2_c0002_s50": -0.028337152015611853, "x:b33_glm_ohk_g2_c0003": -0.05774811753720308, "x:b33_glm_ohk_g2_c0005_m2": 0.020978657757278727, "x:b33_glm_ohk_g2_c0005": 0.06913277178086205, "x:b33_lgb_dl2_hier": 0.017888351349466494, "x:b33_glm_ohk_g2x_c0005": 0.004540027136092951, "x:b33_glm_ohk_c0005": 0.014166684839356779, "x:b33_ebm_dl": -0.0008774152191717169, "x:b33_ebm_dl_g2": -0.005857784449468495, "x:b33_glm_ohk_c0002": -0.017683444360106583, "x:b33_glm_ohk_g2_c001": -0.020791966686680905, "x:b33_glm_ohk_c001": 0.050433933985692986, "x:b33_tabm_dl": 0.001259828187652597, "x:b33_rmlp2_dl_hier": -0.03169979688826953, "x:b33_rmlp2_dl": 0.046652700859417695, "x:b33_glm_dl": 0.0017568772142740109, "x:b33_rmlp_dl": 0.0009392792271499163, "x:b33_slr_c003": 0.024664979660470945, "x:b33_rmlp_dl_g2": -0.007193928931759493, "x:b33_lgb_rawdig": 0.012311772133054908, "x:yek_tabm": -0.009356989871494158, "x:weni_bpe_cat_d3": 0.017469736032019945, "x:weni_bpe_cat_d5": -0.016317796146460204, "x:weni_bpe_glm": -0.0008761226230040804, "x:weni_bpe_lgb_l4": 0.002772880602048725, "x:weni_realmlp_bpe": 0.005154999248359224, "x:weni_realmlp_bpe_v2": -0.014194291548742976, "x:weni_tabm_bpe": 0.00928297678649426} """)
SA = apply(STEP_PA, L(MALF), 'test')
FILE_P = Zr(L(MALF) + 0.5 * (L(FILE_O) - L(RIS)) + 1.25 * (L(SA) - L(MALF)))
# Version 22. Malyshev Danil's 0.95539 file (step P moved towards 37 scored files; pinned copy). Step Q: a member step from there.
MALR2 = pd.read_csv(find('malyshevdanil_round2_0p95539.csv', _sums['malyshevdanil_round2_0p95539.csv']))['is_fraud'].to_numpy(np.float64)
STEP_Q = json.loads(""" {"s:megayak_5f_lm": 0.0008004616370247861, "s:masaya_gpt2": -0.033898447297487255, "s:lm7_5fold": 8.973166608538167e-05, "s:gpt2_seed2026": 0.009335974099328351, "s:tabpfn_slimypunk_tp1": 0.002473892866468383, "s:sadam_tabpfn_100k_tp3": -0.0032588684368180915, "s:yekenot_realmlp": -0.0007470288696874058, "s:alex_glm_gen": 0.02488634512405577, "s:daoviet_gpdvae": 0.009909420237504987, "s:gbn_bag5_w512": 0.00887834099152788, "s:sameer_glm10f": -0.00042837874864078, "s:sadam_v6_gb_xgb": -0.0038500006750058335, "s:sadam_v6_gb_lgb": -0.0022566418588662557, "s:sadam_v4_cat": 0.0001581664809123291, "s:gb_bag5": -0.001100231617545331, "s:megayak_shallow_tree": -0.0021753310807968752, "wide": 0.005257719698910621, "b:ebm_all_bag": 0.040165308613645485, "b:ebm_dig": -0.026824402879187668, "b:ebm_dig_bag": 0.026289881209445788, "b:ebm_dig_r7": -0.007746388568999021, "b:ebm_diglead_bag": -0.014786261958147374, "b:ebm_diglead_i25": 0.0030004372017423796, "b:ebm_diglead_r11": -0.006428903495210277, "b:ebm_diglead_r13": -0.009290912432983614, "b:ebm_i25_bag": -0.02182322033287534, "b:ebm_i25_bag4": -0.0012741822362107643, "b:ebm_lead4_bag": 0.0115183782254133, "b:gb_bag10": 0.02619945231430549, "b:gb_bag5": -0.020300555210360074, "b:gb_l3": 0.015565962753089307, "b:gb_ohk": -0.00984019194323298, "b:gbc_bag3": -0.02404862631315889, "b:gbe_bag3": -0.013972818810897276, "b:gbn_bag5_w128": 0.0029975670367182804, "b:gbn_bag5_w512": -0.007135036326862426, "b:gbr_bag3": 0.004114999781208474, "b:gbx_bag3": 0.017925334468846237, "b:gbx_bag5": -0.023132400564110627, "b:glm_dig": 0.010066937639240698, "b:glm_dl": 0.007314593348077801, "b:glm_dl_bag": -0.0028330958579381384, "b:glm_dl_s50": -0.002175839520537544, "b:glm_dl_s50_bag": 0.0015423368181609338, "b:glm_ohk": -0.004444568057501348, "b:glm_ohk2": -0.024471503199648675, "b:glm_ohk2_bag": -0.011451053754699392, "b:glm_ohk_c01": 0.0029723168334853294, "b:glm_ohkl": -0.027827662560054853, "b:glm_ohkl_bag": -0.009722329467560088, "b:glm_sb": 0.0022911747367322715, "b:glm_te": -0.004142881232772387, "b:lgb_dig_l4": 0.0012396218662995666, "b:lgb_dig_l7": -0.008114766401794695, "b:lgb_dig_s100": 0.019211806448523126, "b:lgb_dig_s100_bag": -0.01733077300550571, "b:lgb_dig_s100_bag5": -0.020999251047181436, "b:lgb_dig_s100_l3": -0.006019646065922437, "b:lgb_dig_s100_l3_bag": 0.017816529860165877, "b:lgb_diglead_bag": -0.026091737013632463, "b:lgb_diglead_bag5": 0.01783232012979276, "b:lgb_diglead_l2": -0.005650800367632666, "b:lgb_diglead_l3": -0.024173547211873845, "b:lgb_diglead_l3_bag": 0.020075264765201058, "b:lgb_diglead_l4": 0.010798375010535696, "b:lgb_dk_l4": 0.005846821695570796, "b:lgb_raw_l31": 0.007598758207324934, "b:lgb_raw_l4": -0.0020058157286106503, "b:lgb_raw_l4_cnt": 0.0028184073805405164, "b:lgb_rawdig_l4": -0.007997870122367138, "b:lgb_te_l4": -0.027148749582880487, "b:lgb_te_l4_s100": -0.0020588123801158565, "b:lgb_te_l4_s300": 0.005253674908795605, "b:lgb_te_l4_s5": -0.004721293115702387, "b:realmlp_allcat": -0.001596072383430453, "b:realmlp_cnt": 0.005097419820302128, "b:realmlp_dig": -0.000595047447165879, "b:realmlp_dig_s100": 0.0015907310633667914, "b:realmlp_dig_s2": 0.003949654405748695, "b:realmlp_rawdig": -0.0018416480365486703, "b:realmlp_te": 0.0002848432485971535, "b:realmlp_te_cnt": 0.005319395127404591, "b:realmlp_te_s2": 0.006947146281073044, "b:realmlp_v1": 0.0018789675676402554, "b:xgb_dig_d2": 0.010106691880681223, "b:xgb_raw_d3": -0.013347910745374347, "b:xgb_te_d2": 0.006169713953203058, "x:Sold": 0.0431160865234425, "x:rmlpnew": 0.021033940472846905, "x:rmlpold": -0.0008166487863206581, "x:tabm": -0.002726926356743095, "x:davut": -0.0022825720195702066, "x:b33_lgb_g2_plaininit_l4": -0.0019244051758238036, "x:b33_lgb_g2_plaininit_l3": -0.003802670343045946, "x:b33_lgb_g2_glminit_l4": -0.0049042695224475645, "x:b33_lgb_g2_glminit_l5": -0.019708000024443378, "x:b33_lgb_g2_glminit_s50": 0.0029870320329914426, "x:b33_lgb_g2_glminit": 0.011683875796262719, "x:b33_lgb_g2_glminit_l7": -0.002993810670421759, "x:b33_lgb_g2_glminit_l4_hier": 0.01874379893462292, "x:b33_xgb_g2_glminit": -0.029201859391966088, "x:b33_lgb_dl3_g2c3": -0.005153113053624564, "x:b33_lgb_dl3": -0.004134917765977123, "x:b33_lgb_dl4_g2cc": -0.002594201402771755, "x:b33_lgb_dl4_g2": 0.009790061922257278, "x:b33_glm_plain_g2_s50": -0.004501633480818509, "x:b33_lgb_dl3_cc": -0.010128399959947819, "x:b33_lgb_dl3_hier": -0.013452367004216101, "x:b33_lgb_dl3_trie": -0.00036204135002516257, "x:b33_ebm_g2_glminit": -0.001903377832459537, "x:b33_lgb_g2_glminit_l2": 0.015005816815530044, "x:b33_lgb_dl3_s10": -3.22739981625549e-05, "x:b33_lgb_dl4_hier": 0.002150801087061717, "x:b33_xgb_dl2": -0.0031655737586606886, "x:b33_glm_plain_s50": 0.003873938864992919, "x:b33_lgb_dl4": 0.01746265302703494, "x:b33_lgb_dl4_c3": 0.004624149322645104, "x:b33_lgb_dl3_s100": 0.005273361645883805, "x:b33_lgb_dl4_s100": 0.013224246714051882, "x:b33_lgb_dl4_s10": -0.0033134374880990805, "x:b33_glm_ohk_g2_c0003_s50": 0.003268715359200505, "x:b33_glm_ohk_g2_c0003_s100": 0.023628893596598765, "x:b33_glm_ohk_g2_c0005_s50": -0.0015411666197183015, "x:b33_glm_ohk_g2_c0005_s100": 0.025932794408149603, "x:b33_glm_ohk_g2_c0002_s50": -0.015382222867955464, "x:b33_glm_ohk_g2_c0003": -0.017743711558045553, "x:b33_glm_ohk_g2_c0005_m2": -0.008940978699379458, "x:b33_glm_ohk_g2_c0005": 0.023094475749600957, "x:b33_lgb_dl2_hier": -5.196413704930624e-05, "x:b33_glm_ohk_g2x_c0005": -0.0014230764688091745, "x:b33_glm_ohk_c0005": 0.017597649906690542, "x:b33_ebm_dl": 0.0041781934373587935, "x:b33_ebm_dl_g2": 0.005947383357809754, "x:b33_glm_ohk_c0002": -0.004826764544324646, "x:b33_glm_ohk_g2_c001": -0.00902597075256553, "x:b33_glm_ohk_c001": 0.03472699667708842, "x:b33_tabm_dl": -0.00191577842600935, "x:b33_rmlp2_dl_hier": -0.009303046378469574, "x:b33_rmlp2_dl": 0.006083520372267557, "x:b33_glm_dl": 0.005700655170262357, "x:b33_rmlp_dl": 0.0006627003310012186, "x:b33_slr_c003": 0.011577669353114993, "x:b33_rmlp_dl_g2": -0.0031500213245349764, "x:b33_lgb_rawdig": 0.0028921032492949636, "x:yek_tabm": -0.016017956902088154, "x:weni_bpe_cat_d3": 0.010593690087146852, "x:weni_bpe_cat_d5": -0.0010081739867115966, "x:weni_bpe_glm": -0.009284369226247353, "x:weni_bpe_lgb_l4": -0.0064968640852515695, "x:weni_realmlp_bpe": 0.002329988102680976, "x:weni_realmlp_bpe_v2": -0.018774193686013012, "x:weni_tabm_bpe": 0.009115281532505173} """)
FILE_Q = Zr(apply(STEP_Q, L(MALR2), 'test'))
# Version 24. Malyshev Danil's 0.95540 file (step Q moved towards 39 scored files; pinned copy) and Md. Hamid Hosen's board file
# PM8r8b (0.95510; pinned copy). Step R: the part of PM8r8b - step O orthogonal to eight scored moves, 0.4 of it on the 0.95540 file.
MALR3 = pd.read_csv(find('malyshevdanil_round3_0p95540.csv', _sums['malyshevdanil_round3_0p95540.csv']))['is_fraud'].to_numpy(np.float64)
PM8 = pd.read_csv(find('hosen_PM8r8b_0p95510.csv', _sums['hosen_PM8r8b_0p95510.csv']))['is_fraud'].to_numpy(np.float64)
SC = {s: L(pd.read_csv(find(f'scored_step_{s}_{v}.csv', _sums[f'scored_step_{s}_{v}.csv']))['is_fraud'].to_numpy(np.float64))   # the exact scored
      for s, v in (('j', '0p95508'), ('k', '0p95519'), ('l', '0p95527'), ('m', '0p95531'), ('n', '0p95533'), ('o', '0p95536'))}   # files J to O
MOVES = np.column_stack([SC['k'] - SC['j'], SC['l'] - SC['k'], SC['m'] - SC['l'], SC['n'] - SC['m'], SC['o'] - SC['n'],
                         L(FILE_P) - SC['o'], L(FILE_Q) - L(FILE_P), L(MALR3) - L(FILE_Q)])
dH = L(PM8) - SC['o']
RH = dH - MOVES @ np.linalg.lstsq(MOVES, dH, rcond=None)[0]                 # what no scored move has tried yet
FILE_R = Zr(L(MALR3) + 0.4 * RH)
# Version 26. Areeb Ali's 0.95547 file = step R + 0.35 x the part of Malyshev Danil's 0.95541 fusion file (pinned copy) orthogonal
# to our scored moves O to P, P to Q and Q to R (exact scored bytes, pinned). Step S: more of that residual, more of step R's
# projected direction, and a fifth of a member step (152 members) that raises CV.
SCR = {s: L(pd.read_csv(find(f'scored_step_{s}_{v}.csv', _sums[f'scored_step_{s}_{v}.csv']))['is_fraud'].to_numpy(np.float64))
       for s, v in (('p', '0p95539'), ('q', '0p95540'), ('r', '0p95546'))}
MALR4 = L(pd.read_csv(find('malyshevdanil_round4_0p95541.csv', _sums['malyshevdanil_round4_0p95541.csv']))['is_fraud'].to_numpy(np.float64))
DQ = np.column_stack([SCR['r'] - SCR['q'], SCR['q'] - SCR['p'], SCR['p'] - SC['o']])
dF = MALR4 - SCR['q']
RF = dF - DQ @ np.linalg.lstsq(DQ, dF, rcond=None)[0]                     # the fusion residual
A547 = U(SCR['r'] + 0.35 * RF)                                             # Areeb Ali's 0.95547 file, rebuilt
STEP_S = json.loads(""" {"s:megayak_5f_lm": 0.003589875979139528, "s:masaya_gpt2": 0.03373458383421992, "s:lm7_5fold": -0.0012538340187259738, "s:gpt2_seed2026": -0.024484267849967684, "s:tabpfn_slimypunk_tp1": -0.006730873314696634, "s:sadam_tabpfn_100k_tp3": -0.016254293479898072, "s:yekenot_realmlp": 0.0197814579287145, "s:alex_glm_gen": 0.014653576779623288, "s:daoviet_gpdvae": -0.016277440568025728, "s:gbn_bag5_w512": -0.004525359276816555, "s:sameer_glm10f": 0.026094523560260315, "s:sadam_v6_gb_xgb": -0.057983845490826975, "s:sadam_v6_gb_lgb": 0.018670245701631533, "s:sadam_v4_cat": 0.02592354400663669, "s:gb_bag5": 0.11449620994478894, "s:megayak_shallow_tree": -0.07429465034412264, "wide": -0.01719264060613191, "b:ebm_all_bag": -0.11018306856104355, "b:ebm_dig": 0.06566609268077053, "b:ebm_dig_bag": -0.089963352869573, "b:ebm_dig_r7": 0.08941191689173411, "b:ebm_diglead_bag": 0.018664294388450464, "b:ebm_diglead_i25": 0.0197526342854218, "b:ebm_diglead_r11": -0.01403392022873636, "b:ebm_diglead_r13": 0.08488223355849861, "b:ebm_i25_bag": -0.09595632216337141, "b:ebm_i25_bag4": 0.08608307760114663, "b:ebm_lead4_bag": -0.05076137323609167, "b:gb_bag10": -0.05556898505257696, "b:gb_bag5": -0.06872260288750263, "b:gb_l3": 0.047934376363290516, "b:gb_ohk": 0.012442699196411026, "b:gbc_bag3": -0.034472892759227436, "b:gbe_bag3": -0.02098714742945364, "b:gbn_bag5_w128": -0.020579046690843948, "b:gbn_bag5_w512": -0.008926538556991664, "b:gbr_bag3": 0.02119313204763184, "b:gbx_bag3": 0.02392332864247426, "b:gbx_bag5": -0.014312918658049803, "b:glm_dig": -0.006591422588313537, "b:glm_dl": -0.0003715275216160931, "b:glm_dl_bag": -0.024499570749324212, "b:glm_dl_s50": 0.054300857559495194, "b:glm_dl_s50_bag": 0.004742868610331869, "b:glm_ohk": 0.0013754205958226436, "b:glm_ohk2": 0.02846722167579312, "b:glm_ohk2_bag": -0.031163680500353577, "b:glm_ohk_c01": 0.015233426920250821, "b:glm_ohkl": 0.005029108779329467, "b:glm_ohkl_bag": -0.010484547384437567, "b:glm_sb": -0.01139567249054453, "b:glm_te": -0.011317266569956887, "b:lgb_dig_l4": 0.041880973430824205, "b:lgb_dig_l7": -0.00983845279080394, "b:lgb_dig_s100": -0.020927618409196058, "b:lgb_dig_s100_bag": 0.03671862309319634, "b:lgb_dig_s100_bag5": -0.01377889234478868, "b:lgb_dig_s100_l3": 0.02486085460998966, "b:lgb_dig_s100_l3_bag": -0.004034043139642007, "b:lgb_diglead_bag": 0.016790668737855383, "b:lgb_diglead_bag5": -0.03341323840454773, "b:lgb_diglead_l2": 0.012425207979047564, "b:lgb_diglead_l3": 0.0015105251389549417, "b:lgb_diglead_l3_bag": 0.007636517487725187, "b:lgb_diglead_l4": 0.015628481667739724, "b:lgb_dk_l4": 0.0028802340848226894, "b:lgb_raw_l31": 0.008776429681335938, "b:lgb_raw_l4": 1.755186683844149e-05, "b:lgb_raw_l4_cnt": 0.016194658124622223, "b:lgb_rawdig_l4": 0.00784359841706728, "b:lgb_te_l4": -0.02832647792147779, "b:lgb_te_l4_s100": -0.004846262320625341, "b:lgb_te_l4_s300": 0.036623625179564634, "b:lgb_te_l4_s5": -0.0032444242659561175, "b:realmlp_allcat": 0.03681274932320331, "b:realmlp_cnt": 0.014751577653747152, "b:realmlp_dig": -0.013572255151609673, "b:realmlp_dig_s100": 0.021857943986363614, "b:realmlp_dig_s2": -0.000835115315316269, "b:realmlp_rawdig": -0.009335757679498641, "b:realmlp_te": -0.003945533756839068, "b:realmlp_te_cnt": 0.0010971970408482016, "b:realmlp_te_s2": -0.010671426797267403, "b:realmlp_v1": -0.014860507578309349, "b:xgb_dig_d2": -0.018758744081125926, "b:xgb_raw_d3": 0.010614956499526226, "b:xgb_te_d2": 0.0007435309255759401, "x:Sold": 0.04631911510821323, "x:rmlpnew": -0.012110829775284604, "x:rmlpold": 0.019694326593017723, "x:tabm": -0.017371714034008803, "x:davut": -0.0063613979318977495, "x:b33_lgb_g2_plaininit_l4": -0.035494705097582985, "x:b33_lgb_g2_plaininit_l3": 0.005446411830740713, "x:b33_lgb_g2_glminit_l4": 0.00993412113169888, "x:b33_lgb_g2_glminit_l5": 0.0011628399382374848, "x:b33_lgb_g2_glminit_s50": 0.031178655160566202, "x:b33_lgb_g2_glminit": 0.018198002149241804, "x:b33_lgb_g2_glminit_l7": -0.011347666205315864, "x:b33_lgb_g2_glminit_l4_hier": -0.013398238209904828, "x:b33_xgb_g2_glminit": 0.001477968087119077, "x:b33_lgb_dl3_g2c3": -0.024212597894732624, "x:b33_lgb_dl3": -0.013781261797924875, "x:b33_lgb_dl4_g2cc": -0.006658697660671929, "x:b33_lgb_dl4_g2": -0.004848541180131104, "x:b33_glm_plain_g2_s50": 0.033963712564017305, "x:b33_lgb_dl3_cc": 0.00966525732846643, "x:b33_lgb_dl3_hier": 0.0003480291471243825, "x:b33_lgb_dl3_trie": 0.00511416337729789, "x:b33_ebm_g2_glminit": -0.017520775420931303, "x:b33_lgb_g2_glminit_l2": 0.008846523251798273, "x:b33_lgb_dl3_s10": -0.005217700475866607, "x:b33_lgb_dl4_hier": 0.013270487245042288, "x:b33_xgb_dl2": -0.013016930686460636, "x:b33_glm_plain_s50": -0.010638651475851586, "x:b33_lgb_dl4": -0.002323300994106387, "x:b33_lgb_dl4_c3": -0.0026289499984704628, "x:b33_lgb_dl3_s100": -0.0004039393002104412, "x:b33_lgb_dl4_s100": -0.005955851390202177, "x:b33_lgb_dl4_s10": -0.011926700308213495, "x:b33_glm_ohk_g2_c0003_s50": -0.05156903718366323, "x:b33_glm_ohk_g2_c0003_s100": 0.056251413827438514, "x:b33_glm_ohk_g2_c0005_s50": 0.01571480685303333, "x:b33_glm_ohk_g2_c0005_s100": 0.010966902600318115, "x:b33_glm_ohk_g2_c0002_s50": -0.050564934925196564, "x:b33_glm_ohk_g2_c0003": -0.040806437171644325, "x:b33_glm_ohk_g2_c0005_m2": 0.054269226539394616, "x:b33_glm_ohk_g2_c0005": -0.020043003838981523, "x:b33_lgb_dl2_hier": -0.004567566420336548, "x:b33_glm_ohk_g2x_c0005": -0.006941742698874174, "x:b33_glm_ohk_c0005": 0.03322321825347668, "x:b33_ebm_dl": -0.001063791629186487, "x:b33_ebm_dl_g2": 0.008187383144431694, "x:b33_glm_ohk_c0002": 0.02896245994162837, "x:b33_glm_ohk_g2_c001": -0.03546957651293739, "x:b33_glm_ohk_c001": -0.009219726156783209, "x:b33_tabm_dl": -0.0029993024796972135, "x:b33_rmlp2_dl_hier": 0.0075603090920317705, "x:b33_rmlp2_dl": -0.018105564278362997, "x:b33_glm_dl": 0.01143813763108555, "x:b33_rmlp_dl": -0.0007820018753457049, "x:b33_slr_c003": 0.004444252787376204, "x:b33_rmlp_dl_g2": 0.0005797780609791837, "x:b33_lgb_rawdig": 0.004003192788758051, "x:yek_tabm": -0.025236553700129876, "x:weni_bpe_cat_d3": 0.004720077274757186, "x:weni_bpe_cat_d5": 0.019891669739690598, "x:weni_bpe_glm": -0.020798894482776276, "x:weni_bpe_lgb_l4": -0.023793423699094016, "x:weni_realmlp_bpe": -0.018063711987082693, "x:weni_realmlp_bpe_v2": -0.02151057487238725, "x:weni_tabm_bpe": 0.020532958087522552} """)
MSTEP = L(apply(STEP_S, L(A547), 'test')) - L(A547)                          # the member step, measured on that file
FILE_S = Zr(SCR['r'] + 0.7 * RF + 0.2 * RH + 0.2 * MSTEP)
# Version 27. Malyshev Danil's 0.95551 file: Areeb Ali's 0.95547 file moved by his board model (pinned copy). Step T: 0.75 more
# of that move, a fifth of the member step above and 0.15 of step R's projected direction.
MALR5 = L(pd.read_csv(find('malyshevdanil_round5_0p95551.csv', _sums['malyshevdanil_round5_0p95551.csv']))['is_fraud'].to_numpy(np.float64))
FILE_T = Zr(MALR5 + 0.75 * (MALR5 - L(A547)) + 0.2 * MSTEP + 0.15 * RH)
# Version 29. dennis6886's 0.95554 file, rebuilt: step T moved 0.12 towards the median of steps R, S and the CV-led file (each
# minus step T, clipped at the 96% quantile of its size). Step U adds four scored directions: Malyshev Danil's move, the member
# step, the move from 0.95551 to Areeb Ali's 0.95552 file (pinned copy) and step R's projected direction.
LT = L(FILE_T)
MED = np.median(np.stack([L(FILE_R) - LT, L(FILE_S) - LT, L(FILE_A) - LT]), axis=0)
CAPD = np.quantile(np.abs(MED), 0.96)
D554 = L(LT + 0.12 * np.clip(MED, -CAPD, CAPD))                              # the 0.95554 file
A552 = L(pd.read_csv(find('areebbb_0p95552.csv', _sums['areebbb_0p95552.csv']))['is_fraud'].to_numpy(np.float64))
FILE_U = Zr(D554 + 0.6 * (MALR5 - L(A547)) + 0.4 * MSTEP + 2.0 * (A552 - MALR5) + 0.1 * RH)
# Version 31. Malyshev Danil's 0.95557 file: step U moved by his board model (pinned copy). Step V: 0.75 more of that move (the
# size step T used on his 0.95551 file) and a fifth of the member step.
MALR6 = L(pd.read_csv(find('malyshevdanil_round6_0p95557.csv', _sums['malyshevdanil_round6_0p95557.csv']))['is_fraud'].to_numpy(np.float64))
FILE_V = Zr(MALR6 + 0.75 * (MALR6 - L(FILE_U)) + 0.2 * MSTEP)
# Version 33. Areeb Ali's 0.95559 file, rebuilt: step V + 0.38 x the part of the 0.95557 move orthogonal to step U's move (recent)
# + 0.45 x the part of step T's move orthogonal to the three newest moves (earlier). Step W carries the earlier residual to 0.9.
LV, LU, LTT = L(FILE_V), L(FILE_U), L(FILE_T)
MOVE6, AXIS = MALR6 - LU, LU - LTT
RECENT = MOVE6 - AXIS * (MOVE6 @ AXIS) / (AXIS @ AXIS)
MOVES = np.column_stack([MOVE6, AXIS, LV - MALR6])
EARLIER = (LTT - L(FILE_S)) - MOVES @ np.linalg.lstsq(MOVES, LTT - L(FILE_S), rcond=None)[0]
A559 = L(LV + 0.38 * RECENT + 0.45 * EARLIER)                                 # the 0.95559 file
FILE_W = Zr(A559 + 0.45 * EARLIER)
# Version 35. Step X: the earlier residual carried to 1.85 x in total, and 0.3 of step U's axis (step T to step U) taken back.
FILE_X = Zr(A559 + 1.4 * EARLIER - 0.3 * AXIS)
# Version 37. Step Y: the move from step W to step X carried further (2.65 x the earlier residual, 0.8 of step U's axis taken back).
FILE_Y = Zr(A559 + 2.2 * EARLIER - 0.8 * AXIS)
# Version 39. Step Z: the earlier residual carried to 5.65 x, 0.3 more of the recent residual.
FILE_Z = Zr(A559 + 5.2 * EARLIER - 0.8 * AXIS + 0.3 * RECENT)
# Version 41. Step AA: Malyshev Danil's 0.95567 file (v43; pinned copy) + 0.75 x his move from v42 to v43, clipped at its 99th
# percentile outside the zones, + 1.5 x the earlier residual.
MALV42 = L(pd.read_csv(find('malyshevdanil_v42_0p95565.csv', _sums['malyshevdanil_v42_0p95565.csv']))['is_fraud'].to_numpy(np.float64))
MALV43 = L(pd.read_csv(find('malyshevdanil_v43_0p95567.csv', _sums['malyshevdanil_v43_0p95567.csv']))['is_fraud'].to_numpy(np.float64))
MOVE43 = MALV43 - MALV42
CAP43 = np.quantile(np.abs(MOVE43[~ZALL]), 0.99)
FILE_AA = Zr(MALV43 + 0.75 * np.clip(MOVE43, -CAP43, CAP43) + 1.5 * EARLIER)
# Version 43. Step AB: Sriram S's orthogonal continuation of step AA (pinned copy) - 0.4 x the recent residual - 0.15 x Malyshev Danil's
# 0.95557 move (MOVE6) + 0.15 x the member step.
SRIF = L(pd.read_csv(find('sriram_orthogonal_0p95568.csv', _sums['sriram_orthogonal_0p95568.csv']))['is_fraud'].to_numpy(np.float64))
FILE_AB = Zr(SRIF - 0.4 * RECENT - 0.15 * MOVE6 + 0.15 * MSTEP)
for name, t in (('submission.csv', FILE_AB), ('submission_step_aa.csv', FILE_AA), ('submission_step_z.csv', FILE_Z), ('submission_step_y.csv', FILE_Y), ('submission_step_x.csv', FILE_X), ('submission_step_w.csv', FILE_W), ('submission_step_v.csv', FILE_V), ('submission_step_u.csv', FILE_U), ('submission_step_t.csv', FILE_T), ('submission_step_s.csv', FILE_S), ('submission_step_r.csv', FILE_R), ('submission_step_q.csv', FILE_Q), ('submission_step_p.csv', FILE_P), ('submission_step_o.csv', FILE_O), ('submission_step_n.csv', FILE_N), ('submission_step_m.csv', FILE_M), ('submission_step_l.csv', FILE_L), ('submission_step_k.csv', FILE_K), ('submission_step_j.csv', FILE_J), ('submission_step_i.csv', FILE_I), ('submission_step_h.csv', FILE_H), ('submission_step_g.csv', FILE_G), ('submission_step_f.csv', FILE_F), ('submission_step_e.csv', FILE_E), ('submission_step_d.csv', FILE_D), ('submission_step_c.csv', FILE_C), ('submission_step_b.csv', FILE_B), ('submission_cv_led.csv', FILE_A)):
    sample[['id']].assign(is_fraud=U(t)).to_csv(WORK / name, index=False)
r = lambda a, b: np.corrcoef(rankdata(a), rankdata(b))[0, 1]
print(f'step B: {len(STEP_B)} members, total weight moved {sum(STEP_B.values()):+.3f}; rank correlation with the 0.95442 file {r(FILE_B, TOP):.6f}')
print(f'step C: {len(STEP_C)} members, total weight moved {sum(STEP_C.values()):+.3f}; rank correlation with the 0.95447 file {r(FILE_C, B15):.6f}')
print(f'step D: {len(STEP_D)} members, total weight moved {sum(STEP_D.values()):+.3f}; rank correlation with the 0.95479 file {r(FILE_D, V19):.6f}')
print(f'step E: rank correlation with the 0.95484 file {r(FILE_E, V20):.6f}')
print(f'step F: {len(STEP_F)} members, total weight moved {sum(STEP_F.values()):+.3f}; rank correlation with step E {r(FILE_F, FILE_E):.6f}')
print(f'step G: {len(STEP_G)} members, total weight moved {sum(STEP_G.values()):+.3f}; rank correlation with the 0.95496 file {r(FILE_G, V22):.6f}')
print(f'step H: test rows in zone 1 {ZONE1.sum():,}, in zone 2 {ZONE2.sum():,} ({(ZONE2 & ~ZONE1).sum():,} not in zone 1); rank correlation with step G {r(FILE_H, FILE_G):.6f}')
print(f'step I: zones 3 and 4 add {(ZONE34 & ~(ZONE1 | ZONE2)).sum():,} test rows; {len(STEP_I)} members ({len(_m33)} of busyaprime); rank correlation with step H {r(FILE_I, FILE_H):.6f}')
print(f'step J: {len(STEP_J)} members ({len(WENI)} of Weni Mohamed, 1 of yekenot), total weight moved {sum(STEP_J.values()):+.3f}; rank correlation with step I {r(FILE_J, FILE_I):.6f}')
print(f'J x 3 (Malyshev Danil): beta {beta:.6f}; rank correlation with step J {r(MAL2, FILE_J):.6f}')
print(f'step K: {len(STEP_K)} members, total weight moved {sum(STEP_K.values()):+.3f}; rank correlation with the 0.95512 file {r(FILE_K, MAL2):.6f}')
print(f'K x 1.5 (Malyshev Danil): rank correlation with step K {r(MAL3, FILE_K):.6f}')
print(f'step L: {len(STEP_L)} members, total weight moved {sum(STEP_L.values()):+.3f}; rank correlation with the 0.95520 file {r(FILE_L, MAL3):.6f}')
print(f'step M: member step of {len(STEP_S1)} members; rank correlation with step L {r(FILE_M, FILE_L):.6f}')
print(f'step N: {len(STEP_N)} members, total weight moved {sum(STEP_N.values()):+.3f}; rank correlation with step M {r(FILE_N, FILE_M):.6f}')
print(f'0.95535 file: beta {float(dN @ eF / (eF @ eF)):.4f}; rank correlation with step N {r(RIS, FILE_N):.6f}')
print(f'step O: {len(STEP_O)} members, total weight moved {sum(STEP_O.values()):+.3f}; rank correlation with the 0.95535 file {r(FILE_O, RIS):.6f}')
print(f'step P: member step of {len(STEP_PA)} members; rank correlation with the 0.95537 file {r(FILE_P, MALF):.6f}, with step O {r(FILE_P, FILE_O):.6f}')
print(f'step Q: {len(STEP_Q)} members, total weight moved {sum(STEP_Q.values()):+.3f}; rank correlation with the 0.95539 file {r(FILE_Q, MALR2):.6f}')
print(pd.Series(STEP_Q).sort_values(key=abs, ascending=False).head(10).round(3).to_string())
cos = lambda a, b: float(a @ b / np.sqrt((a @ a) * (b @ b)))
print(f'step R: PM8r8b - step O has cosine {cos(dH, L(MALR3) - L(FILE_Q)):.2f} with the move to the 0.95540 file; '
      f'{np.sqrt((RH @ RH) / (dH @ dH)):.1%} of it is kept; rank correlation with the 0.95540 file {r(FILE_R, MALR3):.6f}')
print(f'step S: fusion residual keeps {np.sqrt((RF @ RF) / (dF @ dF)):.1%} of the fusion move; member step of {len(STEP_S)} members; '
      f'rank correlation with step R {r(FILE_S, FILE_R):.6f}, with the 0.95547 file {r(FILE_S, A547):.6f}')
print(f'step T: the move to the 0.95551 file has cosine {cos(MALR5 - L(A547), L(FILE_S) - L(A547)):.2f} with the move of step S; '
      f'rank correlation with the 0.95551 file {r(FILE_T, MALR5):.6f}, with step S {r(FILE_T, FILE_S):.6f}')
print(f'step U: rank correlation with the 0.95554 file {r(FILE_U, D554):.6f}, with step T {r(FILE_U, FILE_T):.6f}; '
      f'the 0.95552 move is {np.sqrt(((A552 - MALR5) @ (A552 - MALR5)) / ((MALR5 - L(A547)) @ (MALR5 - L(A547)))):.0%} of the size of the 0.95551 move')
print(f'step V: the move to the 0.95557 file has cosine {cos(MALR6 - L(FILE_U), L(FILE_U) - D554):.2f} with the move of step U; '
      f'rank correlation with the 0.95557 file {r(FILE_V, MALR6):.6f}, with step U {r(FILE_V, FILE_U):.6f}')
print(f'step W: the earlier residual keeps {np.sqrt((EARLIER @ EARLIER) / ((LTT - L(FILE_S)) @ (LTT - L(FILE_S)))):.0%} of step T\'s move; '
      f'rank correlation with the 0.95559 file {r(FILE_W, A559):.6f}, with step V {r(FILE_W, FILE_V):.6f}')
print(f'step X: rank correlation with step W {r(FILE_X, FILE_W):.6f}, with the 0.95559 file {r(FILE_X, A559):.6f}')
print(f'step Y: rank correlation with step X {r(FILE_Y, FILE_X):.6f}, with step W {r(FILE_Y, FILE_W):.6f}')
print(f'step Z: rank correlation with step Y {r(FILE_Z, FILE_Y):.6f}, with step X {r(FILE_Z, FILE_X):.6f}')
print(f'step AA: the v42-to-v43 move is clipped at {CAP43:.4f} (1% of rows); rank correlation with the 0.95567 file {r(FILE_AA, MALV43):.6f}, with step Z {r(FILE_AA, FILE_Z):.6f}')
print(f'step AB: rank correlation with Sriram S\'s orthogonal continuation {r(FILE_AB, SRIF):.6f}, with step AA {r(FILE_AB, FILE_AA):.6f}')

## The step on labelled rows

The test side above is exact. For the out-of-fold side we need the 0.95442 file on the training rows. Its two RealMLP runs have
real OOF predictions; its two base files and alexchilton's stack do not, so they are replayed: fitted on the test rows as a linear
blend of the 87 members, the same weights applied to their OOF predictions (no labels used in the fit). The numbers below are
in-sample (the step's weights saw all five folds); the nested figures in the table at the top refit the weights on four folds.

In [ ]:
O_new = find('oof_preds.csv', inside='realmlp-pytabkit', required=False); O_old = find('oof_preds.csv', inside='realmlp-pytorch', required=False)
if O_new is None or O_old is None:
    print('RealMLP OOF files not attached: this section is skipped (the submission files above do not need them).')
else:
    ids = pd.read_csv(find('train.csv', inside='playground-series-s6e11'), usecols=['id'])['id'].to_numpy()
    rd_oof = lambda p: L(pd.read_csv(p).set_index('id').loc[ids].iloc[:, -1].to_numpy(np.float64))
    XO = {'x:rmlpnew': rd_oof(O_new), 'x:rmlpold': rd_oof(O_old)}
    auc = lambda s: roc_auc_score(y, s)
    print(f"RealMLP OOF AUC: v15 {auc(XO['x:rmlpnew']):.6f}, v28 {auc(XO['x:rmlpold']):.6f}")
    K87 = [k for k in STEP_B if not k.startswith('x:')]
    A_T = np.column_stack([member(k, 'test') for k in K87] + [np.ones(len(sample))]); A_O = np.column_stack([member(k, 'oof') for k in K87] + [np.ones(len(y))])
    def replay(t):
        b = np.linalg.lstsq(A_T, t, rcond=None)[0]; return A_O @ b, 1 - ((t - A_T @ b) ** 2).mean() / t.var()
    XO['x:Sold'], r2s = replay(L(T['stack']))
    BO = {}
    for b in ('base_ours', 'base_bhuv'):
        BO[b], r2 = replay(L(T[b])); print(f'{b}: replay R2 on the test rows {r2:.4f}')
    print(f'stack: replay R2 {r2s:.4f}')
    D_O = XO['x:rmlpnew'] - XO['x:rmlpold']
    TOP_O = L(np.mean([U(BO[b] + 0.4 * D_O) for b in BO], axis=0))
    def change(s, base):
        up = sum(roc_auc_score(y[FOLD == f], s[FOLD == f]) > roc_auc_score(y[FOLD == f], base[FOLD == f]) for f in range(5))
        return 1e5 * (auc(s) - auc(base)), up
    for name, w in (('step B (submission_step_b.csv)', STEP_B), ('step A (submission_cv_led.csv)', STEP_A)):
        ch = change(apply(w, TOP_O, 'oof'), TOP_O)
        print(f'{name:32s} OOF change from the 0.95442 file (in-sample, replay) {ch[0]:+6.1f}  folds up {ch[1]}/5')
    B_O = apply(STEP_B, TOP_O, 'oof'); B15_O = TOP_O + 1.5 * (B_O - TOP_O)
    ch = change(B15_O, TOP_O); print(f"{'1.5 x step B (asteriospanoras)':32s} OOF change from the 0.95442 file (in-sample, replay) {ch[0]:+6.1f}  folds up {ch[1]}/5")
    C_O = apply(STEP_C, B15_O, 'oof')
    ch = change(C_O, B15_O); print(f"{'step C (submission_step_c.csv)':32s} OOF change from the 0.95447 file (in-sample, replay) {ch[0]:+6.1f}  folds up {ch[1]}/5")
    XO['x:tabm'] = L(np.load(find('tabm_oof.npy', 'aa0f917f7a4727377cbd349970cc45240137d2a5e589c1ab00a33972789eafcd')).astype(np.float64))
    print(f"TabM OOF AUC {auc(XO['x:tabm']):.6f}")
    TD_O = XO['x:tabm'] - XO['x:rmlpold']
    ST_O = L(np.mean([U(BO[b] + 0.4 * D_O + 0.15 * TD_O) for b in BO], axis=0))
    V19_O = L(B15_O + 1.5 * (C_O - B15_O)) + (ST_O - TOP_O)
    ch = change(V19_O, TOP_O); print(f"{'v19 (asteriospanoras)':32s} OOF change from the 0.95442 file (in-sample, replay) {ch[0]:+6.1f}  folds up {ch[1]}/5")
    D_Ov = apply(STEP_D, V19_O, 'oof')
    ch = change(D_Ov, V19_O); print(f"{'step D (submission_step_d.csv)':32s} OOF change from the 0.95479 file (in-sample, replay) {ch[0]:+6.1f}  folds up {ch[1]}/5")
    V20_O = L(V19_O + 2.0 * (D_Ov - V19_O)) + 0.3 * (ST_O - TOP_O)
    Q2_Ov = apply(STEP_Q2, V19_O, 'oof')
    E_O = V20_O + (Q2_Ov - D_Ov) + (apply(STEP_E1, V20_O, 'oof') - V20_O) + (apply(STEP_E2, V20_O, 'oof') - V20_O)
    ch = change(V20_O, V19_O); print(f"{'v20 (asteriospanoras)':32s} OOF change from the 0.95479 file (in-sample, replay) {ch[0]:+6.1f}  folds up {ch[1]}/5")
    ch = change(E_O, V20_O); print(f"{'step E (submission_step_e.csv)':32s} OOF change from the 0.95484 file (in-sample, replay) {ch[0]:+6.1f}  folds up {ch[1]}/5")
    F_O = apply(STEP_F, E_O, 'oof')
    ch = change(F_O, E_O); print(f"{'step F (submission_step_f.csv)':32s} OOF change from step E (in-sample, replay) {ch[0]:+6.1f}  folds up {ch[1]}/5")
    XO['x:davut'] = L(pd.read_csv(find('davutkursun_xgb_glm_bag_oof.csv', '82201b17609df466e06e31b56ec594914bfd41d92017fdc74e5587f3203dd800')).set_index('id').loc[ids, 'pred'].to_numpy(np.float64))
    V22_O = E_O + 6.0 * (F_O - E_O)
    ch = change(V22_O, E_O); print(f"{'6 x step F (asteriospanoras v22)':32s} OOF change from step E (in-sample, replay) {ch[0]:+6.1f}  folds up {ch[1]}/5")
    G_O = apply(STEP_G, V22_O, 'oof')
    ch = change(G_O, V22_O); print(f"{'step G (submission_step_g.csv)':32s} OOF change from the 0.95496 file (in-sample, replay) {ch[0]:+6.1f}  folds up {ch[1]}/5")
    # The zero-fraud zones on the labelled rows: no fraud inside either, and what sending them to the bottom does to step G
    trn = pd.read_csv(find('train.csv', inside='playground-series-s6e11'), usecols=['velocity_score', 'merchant_risk_score'])
    Z1 = trn['merchant_risk_score'].to_numpy() < 13.1
    Z2 = (trn['velocity_score'].to_numpy() <= 4.1) & (trn['merchant_risk_score'].to_numpy() <= 20.0)
    print(f'zone 1: {Z1.sum():,} training rows, {int(y[Z1].sum())} frauds | zone 2: {Z2.sum():,} rows ({(Z2 & ~Z1).sum():,} not in zone 1), {int(y[Z2].sum())} frauds')
    trf = pd.read_csv(find('train.csv', inside='playground-series-s6e11'))
    CL = ((trf['cvv_retry_count'] == 0) & (trf['prior_disputes'] == 0) & ~trf['is_new_merchant'] & ~trf['is_foreign_transaction'] & ~trf['used_vpn']
          & ~trf['ip_country_mismatch'] & ~trf['billing_shipping_mismatch'] & ~trf['is_ai_generated_scam_attempt']).to_numpy()
    Z34 = CL & (((trn['velocity_score'].to_numpy() <= 4.1) & (trn['merchant_risk_score'].to_numpy() <= 33.8))
                | ((trn['velocity_score'].to_numpy() <= 6.0) & (trn['merchant_risk_score'].to_numpy() <= 25.0)))
    print(f'zones 3 and 4: {Z34.sum():,} training rows ({(Z34 & ~(Z1 | Z2)).sum():,} not in zones 1-2), {int(y[Z34].sum())} frauds')
    for name, m in (('zone 1 to the bottom', Z1), ('zones 1 and 2 to the bottom', Z1 | Z2), ('zones 1 to 4 to the bottom', Z1 | Z2 | Z34)):
        ch = change(np.where(m, G_O - 100, G_O), G_O); print(f'{name:32s} OOF change on step G {ch[0]:+6.2f}  folds up {ch[1]}/5')
    # The new members of step J on the labelled rows (pinned copies, same five folds)
    for nm in WENI:
        XO['x:weni_' + nm] = L(np.load(find(f'oof_weni_{nm}.npy', _sums[f'oof_weni_{nm}.npy'], inside='board-model-pinned-inputs')).astype(np.float64))
    XO['x:yek_tabm'] = rd_oof(find('yekenot_tabm_pytabkit_oof_preds.csv', _sums['yekenot_tabm_pytabkit_oof_preds.csv']))
    for k in [k for k in XO if k.startswith(('x:weni_', 'x:yek_'))]:
        print(f'{k[2:]:22s} OOF AUC {auc(XO[k]):.5f}  folds ' + ' '.join(f'{roc_auc_score(y[FOLD == f], XO[k][FOLD == f]):.4f}' for f in range(5)))

## Limits

* **The base files are public-board files.** The 0.95442 file and its two bases were built by steps chosen on the public board;
  their authors' own out-of-fold readings put several of those steps below zero. This notebook's steps are read on both the forecast and
  nested CV (each version's table); they do not undo the earlier ones. For the private rows, a CV-backed file such as the
  `submission_cv.csv` of our LM Ladder notebook (OOF 0.95266) is the safer pick.
* **The board model is not shipped.** It is fitted on 151 files whose public score is stated (ours and other authors'); its
  forecasts and the nested CV of every candidate are reported as we measured them. Near the top it forecasts a left-out file to
  about 0.7 points.
* **The replay is approximate.** Three inputs have no OOF predictions and are replayed from the 87 members (R2 printed above).

## Credits

* **Vladimir Demidov (yekenot)**: [RealMLP PyTabKit](https://www.kaggle.com/code/yekenot/ps-s6-e11-realmlp-pytabkit) and
  [RealMLP PyTorch](https://www.kaggle.com/code/yekenot/ps-s6-e11-realmlp-pytorch): the two runs, test and OOF.
* **sriram00777** ([Top 20 CPU Blend](https://www.kaggle.com/code/sriram00777/s6e11-top-20-cpu-blend-eda-lb-0-95440)): the neural-correction recipe.
* **asteriospanoras** ([Top Public, Fully Explained](https://www.kaggle.com/code/asteriospanoras/s6e11-lb-0-95447-top-public-fully-explained)):
  the 0.95442 file, 1.5 x step B (0.95447, the start of step C) and the pinned copies in [s6e11-public-board-files](https://www.kaggle.com/datasets/asteriospanoras/s6e11-public-board-files).
* **busyaprime** ([124 Voices + the Top Rebuilt](https://www.kaggle.com/code/busyaprime/s6e11-124-voices-the-top-rebuilt-lb-0-95465)) and the
  [S6E11 33 own members](https://www.kaggle.com/datasets/busyaprime/s6e11-33-own-members) dataset (CC0): the TabM model, its test and OOF
  predictions, and the six-point TabM curve on the board.
* **asteriospanoras** v18, v19 and v20 (1.5 x step C and the TabM step on top: 0.95479, the start of step D; 2 x step D with more
  TabM: 0.95484, the start of step E; 2 x step E: 0.95486, and step E as it is; step F at 1 to 10 times, 0.95496 at 6 times,
  the start of step G).
* **Weni Mohamed** ([39,149 Zero-Fraud Rows (Z1-Z4)](https://www.kaggle.com/code/wenimohamed/s6e11-39-149-zero-fraud-rows-z1-z4-risk-eda)): zones 3 and 4.
* **busyaprime**: the 52 public models of [S6E11 33 own members](https://www.kaggle.com/datasets/busyaprime/s6e11-33-own-members) (CC0), members of step I.
* **Md. Hamid Hosen** ([E1 orthogonal step](https://www.kaggle.com/code/hosen42/s6e11-e1-orthogonal-step)): the board file PM8r8b (0.95510; pinned copy),
  the idea of keeping only its part orthogonal to scored steps, and two scored sizes on step O (0.95544), the start and the size of step R.
* **dennis6886** ([Fraud signal rank fusion four refinements](https://www.kaggle.com/code/dennis6886/s6e11-fraud-signal-rank-fusion-four-refinements)):
  step T moved towards the median residual of steps R, S and the CV-led file (0.95554), the start of step U.
* **Malyshev Danil**: step U moved by his board model with 83 scored rankings and 153 members (0.95557; pinned copy), the start of step V;
  Areeb Ali's 0.95547 file moved by his board model (0.95551; pinned copy), the start of step T; the fusion file with 70 scored rankings and 152 members (0.95541; pinned copy), whose residual step S uses; step Q moved towards 39 scored files (0.95540, the start of step R; pinned copy); step P moved towards 37 scored files (0.95539, the start of step Q; pinned copy); the version-pinned forecast from 36 scored files (0.95537, the start of step P; pinned copy) and seven
  earlier scored versions. **BhuvanNR** ([40 Scored Files, One Forecast](https://www.kaggle.com/code/bhuvannr2006/s6e11-40-scored-files-one-forecast-lb-0-95534)): an open board model.
* **Sriram S** ([Orthogonal Residuals & EDA](https://www.kaggle.com/code/sriram00777/s6e11-orthogonal-residuals-eda-lb-0-95568)): step AA's move carried once
  more, with its overlap with an earlier update projected out (0.95568; pinned copy), the start of step AB, and the five other continuations with their
  scores. **Areeb Ali** ([LB 0.95568](https://www.kaggle.com/code/areebbb/s6e11-which-signals-rank-fraud-lb-0-95568)): 70% of step AA's move under a
  95th-percentile cap (0.95568), part of the board model.
* **Malyshev Danil** ([Fraud Signal Lab](https://www.kaggle.com/code/malyshevdanil/fraud-signal-lab-gpt-2-and-glm-fusion)): step Z moved by his board model
  (v42, 0.95565) and carried once more (v43, 0.95567), both pinned copies, the start of step AA. **Areeb Ali**
  ([Which Signals Rank Fraud? | LB 0.95567](https://www.kaggle.com/code/areebbb/s6e11-which-signals-rank-fraud-lb-0-95567)): two further sizes of that
  move, scored and published, including the 99th-percentile clip. **dennis6886**
  ([Fraud signal rank fusion four refinements](https://www.kaggle.com/code/dennis6886/s6e11-fraud-signal-rank-fusion-four-refinements)): step Z moved
  back towards step Y with the same clip (0.95562). **DavutKursun** ([What a Board Model Learns](https://www.kaggle.com/code/davutkursun/s6e11-what-a-board-model-learns)):
  three scored files, part of the board model.
* **Mirza Ishran** ([Blending the Public Leaders](https://www.kaggle.com/code/mzishran/credit-card-fraud-blending-the-public-leaders)) and **Rishabh Bafna**
  ([Gram-Schmidt Optimization](https://www.kaggle.com/code/rishabhbafnaiiitd/credit-card-fraud-gram-schmidt-optimization)): two blends of step W and Areeb Ali's
  0.95559 file, both scored 0.95559 and published; two of the 209 files of the board model behind step X.
* **Areeb Ali** ([Which Signals Rank Fraud? | LB 0.95559](https://www.kaggle.com/code/areebbb/s6e11-which-signals-rank-fraud-lb-0-95559)): step V + the recent and
  earlier residuals (0.95559, rebuilt here) and his published probes, the start of step W;
  ([LB 0.95552](https://www.kaggle.com/code/areebbb/s6e11-which-signals-rank-fraud-lb-0-95552)) the 0.95551 file + an orthogonal
  residual (0.95552; pinned copy, also in Asterios Panoras' notebook), a direction of step U; step R + 0.35 x the residual of
  Malyshev Danil's 0.95541 fusion file against our moves O to R (0.95547), the start of step S; earlier, the residual of step N against
  Malyshev Danil's file and N + 0.5 r (0.95534). **Rishabh Bafna** ([Gram-Schmidt Optimization](https://www.kaggle.com/code/rishabhbafnaiiitd/credit-card-fraud-gram-schmidt-optimization)):
  N + 1.5 r (0.95535), the start of step O. **sriram00777** ([Rank Residuals & EDA](https://www.kaggle.com/code/sriram00777/s6e11-rank-residuals-eda-lb-0-95534)): ten scored files
  around them.
* **asteriospanoras**: step L at 1.25, 1.5 and 2 times (0.95527, 0.95528, 0.95528), published with their scores.
* **Mayank Raj** ([Fraud Lab: Learn, Build & Review](https://www.kaggle.com/code/mayankraaz/s6e11-fraud-lab-learn-build-review)): the 98 / 2 mix of step L with an
  independently trained model, scored 0.95526 and published with its score.
* **Malyshev Danil** ([Fraud Signal Lab](https://www.kaggle.com/code/malyshevdanil/fraud-signal-lab-gpt-2-and-glm-fusion)): step L moved away from that mix (0.95527); K x 1.5 (0.95520), the start of step L; J x 3 with the step-I part
  removed (0.95512), the start of step K. **sriram00777** ([BPE + TabM Ensemble & EDA](https://www.kaggle.com/code/sriram00777/s6e11-bpe-tabm-ensemble-eda-lb-0-95508)):
  ten scored files around step J, all published with their scores.
* **Weni Mohamed**: the seven BPE-digit models of [S6E11 Diverse OOF Pack](https://www.kaggle.com/datasets/wenimohamed/s6e11-diverse-bpe-realmlp-tabm-oof-pack) (CC0),
  and **Vladimir Demidov (yekenot)**: [TabM · PyTabKit](https://www.kaggle.com/code/yekenot/ps-s6-e11-tabm-pytabkit), members of step J
  (pinned copies in [s6e11-board-model-pinned-inputs](https://www.kaggle.com/datasets/goodpjw2008/s6e11-board-model-pinned-inputs)).
* **daoviet** ([On the journey to your first top 100](https://www.kaggle.com/code/daoviet/s6e11-on-the-journey-to-your-first-top-100)):
  the zero-fraud zone `merchant_risk_score < 13.1` (zone 1) and the idea of looking for such zones.
* **Malyshev Danil** ([Fraud Signal Lab](https://www.kaggle.com/code/malyshevdanil/fraud-signal-lab-gpt-2-and-glm-fusion)): the 0.95503 file, the
  start of step H. **Rishabh Bafna** ([Gram-Schmidt Optimization](https://www.kaggle.com/code/rishabhbafnaiiitd/credit-card-fraud-gram-schmidt-optimization)): the 0.95501 split.
* **DavutKursun** ([One Model, No Public Inputs](https://www.kaggle.com/code/davutkursun/s6e11-one-model-no-public-inputs-lb-0-9514)):
  the XGBoost-on-GLM bag in step G (pinned copy in [s6e11-board-model-pinned-inputs](https://www.kaggle.com/datasets/goodpjw2008/s6e11-board-model-pinned-inputs)).
* **sriram00777** ([TabM Ensemble + EDA](https://www.kaggle.com/code/sriram00777/s6e11-tabm-ensemble-eda-lb-0-95482)): five open comparisons of the files at the top.
* **Aman Atar** ([Sovereign 0.95447+ Frontier](https://www.kaggle.com/code/amanatar/s6e11-sovereign-0-95447-frontier-dual-shield)): a full write-up of the 0.95446 and 0.95447 files.
* **busyaprime** ([124 Voices + the Top Rebuilt](https://www.kaggle.com/code/busyaprime/s6e11-124-voices-the-top-rebuilt-lb-0-95442)): the step-size curve.
* **BhuvanNR** (the 0.95427 base), **alexchilton** (the shallow-tree stack), **Aman Atar** (the bundle of 16 public streams) and the
  authors of the streams in it: **megayak**, **masayakawamata**, **slimypunk03**, **sadamtorres**, **daoviet**, **sameershaikh27**;
  **Areeb Ali** and **Malyshev Danil** for the steps below the top.
* Ours: the 0.95432 base, the 70-model [OOF bank](https://www.kaggle.com/datasets/goodpjw2008/s6e11-oof-bank) and the wide LM stack
  ([LM Ladder](https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-ladder-meets-stack-lb-0-95432)).

If this was useful, please upvote the notebooks above as well.